# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 5/12** (CPU, internet off). **1778 series**, ~10.6 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part5.zip`
- `manifest_rgb25d_320_part5.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 5
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585","series":"1.2.826.0.1.3680043.8.498.83327574802189551711953505990029424266","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764","series":"1.2.826.0.1.3680043.8.498.10407754553908392233077656347467111543","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339","series":"1.2.826.0.1.3680043.8.498.13316012614172613981502777804567639106","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513","series":"1.2.826.0.1.3680043.8.498.17825809991194664057092474839495289855","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938","series":"1.2.826.0.1.3680043.8.498.93204062869632903043115397277688722632","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981","series":"1.2.826.0.1.3680043.8.498.85682196050262263484642671710534742181","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801","series":"1.2.826.0.1.3680043.8.498.12351434285244105587327544381760836887","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269","series":"1.2.826.0.1.3680043.8.498.12167816600846465625393732895688146085","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097","series":"1.2.826.0.1.3680043.8.498.59555639823582486217889487781737126277","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941","series":"1.2.826.0.1.3680043.8.498.30612781365806999657709742696576857451","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449","series":"1.2.826.0.1.3680043.8.498.89861501432639837110110706871986340061","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678","series":"1.2.826.0.1.3680043.8.498.23891847611040842624347800325099754983","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10096216489317602387766193675918767557","series":"1.2.826.0.1.3680043.8.498.43172859976153629688903126690946894177","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659","series":"1.2.826.0.1.3680043.8.498.48015055230453293251293557651072112169","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100","series":"1.2.826.0.1.3680043.8.498.10117727651562277245418454493160349995","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343","series":"1.2.826.0.1.3680043.8.498.13070981013263561967641428142796521676","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833","series":"1.2.826.0.1.3680043.8.498.12707661854492179457840246929406874820","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10132169498032330070492330913783717864","series":"1.2.826.0.1.3680043.8.498.12158138298476613647307379500495222490","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104","series":"1.2.826.0.1.3680043.8.498.84336796506378368646023506678641354566","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314","series":"1.2.826.0.1.3680043.8.498.11409940927618962354373278521471165954","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725","series":"1.2.826.0.1.3680043.8.498.39614516967167113754028577752363330221","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019","series":"1.2.826.0.1.3680043.8.498.14860289040260638280423011827899962328","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320","series":"1.2.826.0.1.3680043.8.498.12944119540649409680042692807606147682","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10158178668962379857908493434842864484","series":"1.2.826.0.1.3680043.8.498.95898842538702235230424113830041599584","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10162168547770229150261479353227398478","series":"1.2.826.0.1.3680043.8.498.22411188800278791134115325107812945132","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269","series":"1.2.826.0.1.3680043.8.498.80784574440782299658628251628549446552","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10181291283712008544475048457589664787","series":"1.2.826.0.1.3680043.8.498.13297596011774342485955610182900815530","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10183939841890895037058579455663880292","series":"1.2.826.0.1.3680043.8.498.11051138355881136273281600148427208911","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10197515493414564227930855210542221039","series":"1.2.826.0.1.3680043.8.498.74446800626929046065841952268681097277","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773","series":"1.2.826.0.1.3680043.8.498.32008633693425842340822037558562279727","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10205592316907266136938526795826751183","series":"1.2.826.0.1.3680043.8.498.11970679083213808961422562493389843440","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10211034309703220222647821582770542129","series":"1.2.826.0.1.3680043.8.498.31764336028067398645713721813842595776","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10218285921271312931891912812808052375","series":"1.2.826.0.1.3680043.8.498.16630902380428122462742659928321447586","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115","series":"1.2.826.0.1.3680043.8.498.52963952898255685734972499283574700146","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10240237086492917349933491458808329326","series":"1.2.826.0.1.3680043.8.498.53575048475669131286770162715177264113","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10243300429607935614655219482457289307","series":"1.2.826.0.1.3680043.8.498.51843921882403184725351047574863626741","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10252163031705285141538200795896549203","series":"1.2.826.0.1.3680043.8.498.64960153577200473556905487910908485859","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766","series":"1.2.826.0.1.3680043.8.498.73229709248183617446995179304669316109","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10262024906324456863812446480254600405","series":"1.2.826.0.1.3680043.8.498.82287621350865318295670877623493088935","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10271567891603078238024189515805908561","series":"1.2.826.0.1.3680043.8.498.92379588327281962276627749930221541494","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029","series":"1.2.826.0.1.3680043.8.498.77766976116670551979942629232553368416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10284263494252700630857173843779724077","series":"1.2.826.0.1.3680043.8.498.40245708400977692920839832761464137379","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10299396087685174806129850512856451005","series":"1.2.826.0.1.3680043.8.498.64324908121907952100122966087354877131","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10301089872627496769249366231947887097","series":"1.2.826.0.1.3680043.8.498.11752981161784818311442613704511388570","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10309477218703786877612946261946268132","series":"1.2.826.0.1.3680043.8.498.57883802715275003472393086208593811624","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738","series":"1.2.826.0.1.3680043.8.498.12371741272528045075821788654879530170","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321461243762453573005633063753741725","series":"1.2.826.0.1.3680043.8.498.60764387818489647929935798986847764676","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10321702681363532511953946967334431952","series":"1.2.826.0.1.3680043.8.498.84154643381631545610326042055883843249","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10326352669696598934883805147731461341","series":"1.2.826.0.1.3680043.8.498.46879239651882912547134951736220008929","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10330805760778577501878048995484987687","series":"1.2.826.0.1.3680043.8.498.11868301510827889028863462311529831208","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10339654093023509742663983374586998753","series":"1.2.826.0.1.3680043.8.498.46570269095648993380087102698556134622","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10349368332722015923160611226877741306","series":"1.2.826.0.1.3680043.8.498.18844536790485528654109203640739498454","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179","series":"1.2.826.0.1.3680043.8.498.11354203021982743106720525294138594122","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10354576637002647863048798607011636355","series":"1.2.826.0.1.3680043.8.498.77834783911923335736291528227805236444","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10371162371589200772388569623043753617","series":"1.2.826.0.1.3680043.8.498.23096471439003297465310065770422966179","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10378535286770237477615181844634903700","series":"1.2.826.0.1.3680043.8.498.77232791818973939517094032028501900462","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066","series":"1.2.826.0.1.3680043.8.498.12682189701143386265725229631656951755","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10387876708310019345913209598612198432","series":"1.2.826.0.1.3680043.8.498.12947332010123481597766123950777769957","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10401813253995823397883215400000450724","series":"1.2.826.0.1.3680043.8.498.31293113001082634943546556250857495731","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10405028141768633100332386157519916334","series":"1.2.826.0.1.3680043.8.498.12413108080252032643792836217673075284","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434","series":"1.2.826.0.1.3680043.8.498.61055644564394168698042963913913650719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10418888172081069709796646795446885715","series":"1.2.826.0.1.3680043.8.498.60667299803054682137883632847190963197","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10422130685453254363700365908336246863","series":"1.2.826.0.1.3680043.8.498.87694367440719007054069561886860809894","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10423586531262405323219596291234698626","series":"1.2.826.0.1.3680043.8.498.10369515136355649398279949457442463385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10429032846672074760218668873927822056","series":"1.2.826.0.1.3680043.8.498.13474463091331523398806068597845119285","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10440858363080910322920496514558347561","series":"1.2.826.0.1.3680043.8.498.73187744829810816894608311328922262132","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409","series":"1.2.826.0.1.3680043.8.498.58225810467834023526008998280623317934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10454875772176439944597553400963295397","series":"1.2.826.0.1.3680043.8.498.88863725628634972901064023858499017410","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818","series":"1.2.826.0.1.3680043.8.498.15385913854350179995371121262900351489","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695","series":"1.2.826.0.1.3680043.8.498.78313975877692478562098934801701090080","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10491012711700179118010977263805828326","series":"1.2.826.0.1.3680043.8.498.10875905640054813780377283856484616469","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590","series":"1.2.826.0.1.3680043.8.498.11397620847069025463412980693484611379","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117","series":"1.2.826.0.1.3680043.8.498.15380940264024332616592928680039003642","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10505282086293876725410939800054491454","series":"1.2.826.0.1.3680043.8.498.93669416873236790449498964572598847249","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10519480654942067141173172019311858213","series":"1.2.826.0.1.3680043.8.498.11215735489692502733159068204464736610","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946","series":"1.2.826.0.1.3680043.8.498.12804377361642432799715051095582384327","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10530226787566708645477257565528473018","series":"1.2.826.0.1.3680043.8.498.31101444675680882652568411858682904498","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10534260101860670685224894870019533364","series":"1.2.826.0.1.3680043.8.498.97049620832590133387398892006127242874","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10537503354035115750949427337500909327","series":"1.2.826.0.1.3680043.8.498.12870240657077141408539129272516401295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10540882690103231159064894952851205573","series":"1.2.826.0.1.3680043.8.498.71756190113907904121735845334132837824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541215772904358938061268226270989801","series":"1.2.826.0.1.3680043.8.498.38781724644439616132736493001180098309","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10542526906255526405613760733936070251","series":"1.2.826.0.1.3680043.8.498.11707867999435371281312538438006178305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736","series":"1.2.826.0.1.3680043.8.498.10465212306729354143265671053471719815","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10574706940531650593705137704922433109","series":"1.2.826.0.1.3680043.8.498.35085809899550840803765051471609153495","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169","series":"1.2.826.0.1.3680043.8.498.64777809540902121696715362743978786269","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816","series":"1.2.826.0.1.3680043.8.498.12303870338077338549548925999965801249","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10591507262178959553431147886647585916","series":"1.2.826.0.1.3680043.8.498.93102374935569403642499850338196963769","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10600654987896381983520458496207741798","series":"1.2.826.0.1.3680043.8.498.75015587106251309157840274081259326303","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10601711635138681269270505152909539290","series":"1.2.826.0.1.3680043.8.498.51529197425819413107403606880858510384","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10603952065748622499432940419853732241","series":"1.2.826.0.1.3680043.8.498.11129361903313479398925104395247769084","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10615816709018712955642546871639043403","series":"1.2.826.0.1.3680043.8.498.10537502922202855015405248746117100342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10624417936475721846835761375334601839","series":"1.2.826.0.1.3680043.8.498.95358607778751626481372143315981817117","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10626411974489938664325062396760010912","series":"1.2.826.0.1.3680043.8.498.10312897226052387933307451576668056737","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506","series":"1.2.826.0.1.3680043.8.498.41663442834395023245192801095947057352","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066","series":"1.2.826.0.1.3680043.8.498.33713300229424206979331394596728155281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10650751856837517181090589171468851374","series":"1.2.826.0.1.3680043.8.498.11020709800412337373031465812752146821","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10653432159060774337155662807644609967","series":"1.2.826.0.1.3680043.8.498.65727105359849653404322871285266098579","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831","series":"1.2.826.0.1.3680043.8.498.11857156675146631276334353675579164655","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10659781998801845781246073928620795082","series":"1.2.826.0.1.3680043.8.498.95609133856347371182169225001496260081","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10674673641872871321890995010616205087","series":"1.2.826.0.1.3680043.8.498.11974899098484068292013146913184403085","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10676253869668012791900504544106495589","series":"1.2.826.0.1.3680043.8.498.44952729788559241725281888235590526923","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10686407053180376367066321489809395592","series":"1.2.826.0.1.3680043.8.498.79818135733771720236658026557538870341","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10702284568670902901846767166679293654","series":"1.2.826.0.1.3680043.8.498.10464961200378212712702097593688073262","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10713285016170190922777298047146228811","series":"1.2.826.0.1.3680043.8.498.88679911582470671432235314679447164082","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10724604223190030113012801427752979879","series":"1.2.826.0.1.3680043.8.498.98353629474736025133984065113480552032","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10729725525284633011179038668107963557","series":"1.2.826.0.1.3680043.8.498.17473565554134777852133323992619963115","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10738640862162795323010965765443420384","series":"1.2.826.0.1.3680043.8.498.13116216758739585185061627177802257490","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10740857795874468664161171813136103519","series":"1.2.826.0.1.3680043.8.498.12649541213222516660396140280224348987","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445","series":"1.2.826.0.1.3680043.8.498.10252546933994968206229718620052753740","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10756245293186761042431460535210737327","series":"1.2.826.0.1.3680043.8.498.67044709967015946890770657247512624561","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826","series":"1.2.826.0.1.3680043.8.498.11634002969367898745876429892053351796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10765617676329232046176300833850901740","series":"1.2.826.0.1.3680043.8.498.37575170619088750403635850264152926482","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10769126730281890056368854871900324448","series":"1.2.826.0.1.3680043.8.498.79311970047889494683837678066541662998","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10777331080659372520952742261140923555","series":"1.2.826.0.1.3680043.8.498.10937590411688291663808447737631937752","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10783501127779906316117667790784040770","series":"1.2.826.0.1.3680043.8.498.84744523579847593352579274247888005350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10803283287658556463524649793955055077","series":"1.2.826.0.1.3680043.8.498.36092178526045200376082890956211511831","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10812894672739619566834776054635588406","series":"1.2.826.0.1.3680043.8.498.90519725310611999715844918954073082345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263","series":"1.2.826.0.1.3680043.8.498.10903070197832289254001715173383697210","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10835889400414225749339608096119636738","series":"1.2.826.0.1.3680043.8.498.11700408793399036218435309108045701654","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839","series":"1.2.826.0.1.3680043.8.498.20777921698448069425369525866577598114","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10848350333764008140459854470293737650","series":"1.2.826.0.1.3680043.8.498.32826741751893361181800578484056467876","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10852517092524464338210061305792376636","series":"1.2.826.0.1.3680043.8.498.63764393282410384517857877679815540540","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10855290714402117092990651362697310270","series":"1.2.826.0.1.3680043.8.498.78768994275563700784706646526234195085","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10857771747813858957718058322384884551","series":"1.2.826.0.1.3680043.8.498.11834914770726200996931593893320787119","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10867622488467375444427908425709367385","series":"1.2.826.0.1.3680043.8.498.67787477230257805958675985329793735963","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10884987036948550523442514147204321748","series":"1.2.826.0.1.3680043.8.498.26816165583443193612810331968538146995","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10895521387894821238259709507311341758","series":"1.2.826.0.1.3680043.8.498.27566900786355199912973018935325917632","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10902166579916526176697193295886992412","series":"1.2.826.0.1.3680043.8.498.49977660763019599396861418472683409743","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10917606677895195837968927716728385881","series":"1.2.826.0.1.3680043.8.498.85248622955428909989665892735316664622","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10920269679387171659375091645187772303","series":"1.2.826.0.1.3680043.8.498.85327307977112776018181518623340296077","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10924995713871537837233985073507509566","series":"1.2.826.0.1.3680043.8.498.65578402366683336704642647726028684821","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398","series":"1.2.826.0.1.3680043.8.498.94660930394364666351564827059520482605","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10933003998197427979325590791540457658","series":"1.2.826.0.1.3680043.8.498.20369117810325454002865995416195274789","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10937585701850306334458420860276069606","series":"1.2.826.0.1.3680043.8.498.56310738969246156351683277206080443340","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409","series":"1.2.826.0.1.3680043.8.498.15805086434277019962649095481574376484","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10959153852077371752490787490374555514","series":"1.2.826.0.1.3680043.8.498.12529584409644198518198483724737180992","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10964287649604862436796341368993301245","series":"1.2.826.0.1.3680043.8.498.11831194092603685651864824383584453938","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10979105661314213697886164132444977106","series":"1.2.826.0.1.3680043.8.498.30182052441361004487801887456296880814","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962","series":"1.2.826.0.1.3680043.8.498.11919315236035687516961955980068208408","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994848378662155711176898659205016445","series":"1.2.826.0.1.3680043.8.498.48538276635909910934194635433920835908","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10994899269366718796438389394954844979","series":"1.2.826.0.1.3680043.8.498.11050874913407926375137685270089656741","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10998405409474759941744305665540376339","series":"1.2.826.0.1.3680043.8.498.10972967353708330418713283752625474260","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11007527194601989039620198927241696256","series":"1.2.826.0.1.3680043.8.498.11595466518537225885820964711498095800","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11014091506864318694851337553197477906","series":"1.2.826.0.1.3680043.8.498.10150849228435818500982508942849425937","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11041694833778404465582610810664391088","series":"1.2.826.0.1.3680043.8.498.25373876949557981717348374277238214146","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043926695471016727738735328317975282","series":"1.2.826.0.1.3680043.8.498.41965136620924285932792578288801966244","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11049882927267755769995376553330493298","series":"1.2.826.0.1.3680043.8.498.54430662917765706706366356262726693442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11052720536194493057619720620119095739","series":"1.2.826.0.1.3680043.8.498.73257179396517971234171195727450239000","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547","series":"1.2.826.0.1.3680043.8.498.53596599813707844574925843216643752795","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11077406555624926104628200252710914930","series":"1.2.826.0.1.3680043.8.498.10078685968426648322211784479447374488","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11080651962904815250736183346281427445","series":"1.2.826.0.1.3680043.8.498.11638776577296712421589998916392017897","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11091958567356981151034727948765010881","series":"1.2.826.0.1.3680043.8.498.78150561106660961652468220709313005475","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11103894265181149878864749992746236768","series":"1.2.826.0.1.3680043.8.498.79749987362812167184690551844633985003","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11111022632180183697497694737350901427","series":"1.2.826.0.1.3680043.8.498.12806007387283483960089642565569078735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11122485005317809892694992204053124213","series":"1.2.826.0.1.3680043.8.498.26228187579960078823424170217249259424","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11125718814639515292482773418372267821","series":"1.2.826.0.1.3680043.8.498.23709746864601254160288803133166677626","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290","series":"1.2.826.0.1.3680043.8.498.97429411189660095803742436552642883670","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419","series":"1.2.826.0.1.3680043.8.498.17208998495562885336807599923355830781","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425","series":"1.2.826.0.1.3680043.8.498.34716280168819137214179207362609752642","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11149530279799223497509928655010867960","series":"1.2.826.0.1.3680043.8.498.10749223286829806719253927186524623097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150751959089942633754454715290410758","series":"1.2.826.0.1.3680043.8.498.12418547158713740570455624056070933805","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11154238555579687338157479416363307605","series":"1.2.826.0.1.3680043.8.498.25911701945215924104687030389512496901","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11173951743276817877673759975853741079","series":"1.2.826.0.1.3680043.8.498.57613371491385693316275767697163793067","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11183915112115196028799391678356464392","series":"1.2.826.0.1.3680043.8.498.81992436255971386174144831524961429097","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11190292536746235063218740991532759863","series":"1.2.826.0.1.3680043.8.498.22139886809385623191975162211071150721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11201589885283493748455257854104771618","series":"1.2.826.0.1.3680043.8.498.12992270887703188127076167525473897130","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11207013225631251718312189933025529824","series":"1.2.826.0.1.3680043.8.498.13282866612441664230008745457803047068","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11208663314213048426907645330143931419","series":"1.2.826.0.1.3680043.8.498.75994728789601964816909657126630474498","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11212667888429022863778740306223585731","series":"1.2.826.0.1.3680043.8.498.73069648459520276783474359846653342459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11218782547268547937641080283066300678","series":"1.2.826.0.1.3680043.8.498.91486798822353301442606827207470987285","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11221318524515817857172161854667955115","series":"1.2.826.0.1.3680043.8.498.81940377421637831450024997832980055932","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11229566047228169850473554486278603754","series":"1.2.826.0.1.3680043.8.498.12434317978423172392089367844577460720","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11232674915186872899189440191805868376","series":"1.2.826.0.1.3680043.8.498.10273110376665282324584020207506141268","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11249981834802444119817546155435759007","series":"1.2.826.0.1.3680043.8.498.33768242062565931148491037346265910420","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11257869433929994221899994675448655047","series":"1.2.826.0.1.3680043.8.498.52300701837844830296016277279376933485","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11264140845869814478794346490273061499","series":"1.2.826.0.1.3680043.8.498.11253421744068482526064379400472789979","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885","series":"1.2.826.0.1.3680043.8.498.30283551753959006425936734452970690319","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551","series":"1.2.826.0.1.3680043.8.498.71531833462787446311717090953342441749","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11277278232377185347750793496260885546","series":"1.2.826.0.1.3680043.8.498.12995791444190256772994011114647967589","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593","series":"1.2.826.0.1.3680043.8.498.30415432733669167886836818492538562768","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11291875173369366854889507729635968509","series":"1.2.826.0.1.3680043.8.498.70930107083712374577611740637737884252","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644","series":"1.2.826.0.1.3680043.8.498.79061417113084232928104592725683475721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11302750434639976191333636496428935632","series":"1.2.826.0.1.3680043.8.498.11361532090754064648378591984442589246","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11312798102268182343541500463824266929","series":"1.2.826.0.1.3680043.8.498.11062993529602593933581508546824250056","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11315871586936178043350206344448851055","series":"1.2.826.0.1.3680043.8.498.11389419183139401860015702426958310365","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11334486185870324597335039163116965763","series":"1.2.826.0.1.3680043.8.498.13389853638754174416509934256165327218","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11337616353935430160375524217870968575","series":"1.2.826.0.1.3680043.8.498.97407224638418001514936053961805787673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11345369648571407541082013397591193473","series":"1.2.826.0.1.3680043.8.498.83884781955649652940148727554601304136","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11353976221682532422630231842008146722","series":"1.2.826.0.1.3680043.8.498.20614753124649234563835460546525944748","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11369844885381156110080055326139875897","series":"1.2.826.0.1.3680043.8.498.12287374883581616734142307386828988155","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11383056981606331444758741778014516431","series":"1.2.826.0.1.3680043.8.498.12650762027901425253127336877714893404","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11386074316342852649653530230563922899","series":"1.2.826.0.1.3680043.8.498.33480105726221184165952819814100070558","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11395349736725035422736613041202251830","series":"1.2.826.0.1.3680043.8.498.51105164939419460607618668767954137775","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11406304618685790128709325242250648417","series":"1.2.826.0.1.3680043.8.498.30887025016130975743726755727908776123","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11413427362266862411525989114324631512","series":"1.2.826.0.1.3680043.8.498.66220482246531160063008538719287136315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11425019165071014217290822852865654127","series":"1.2.826.0.1.3680043.8.498.77918823455977078844797342740627434419","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597","series":"1.2.826.0.1.3680043.8.498.26883315735779577547806534628592599360","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11440217845998767853986099058965644463","series":"1.2.826.0.1.3680043.8.498.10895816793754457526702114263682776425","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11446264194433135282793848490754001623","series":"1.2.826.0.1.3680043.8.498.66310861504330886319496065207894750523","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11451448731901996559834481853733404099","series":"1.2.826.0.1.3680043.8.498.63923270540137270609238793089946840737","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11455388290952699607973723923118848769","series":"1.2.826.0.1.3680043.8.498.40869364922747559066234322429194823042","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197","series":"1.2.826.0.1.3680043.8.498.76258345246046929099161623826799647315","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11468812519969607938527116557104410121","series":"1.2.826.0.1.3680043.8.498.12346026796944003151808964122901912888","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558","series":"1.2.826.0.1.3680043.8.498.10318075660155203787041198805486894343","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11482325994550095339373157256174349858","series":"1.2.826.0.1.3680043.8.498.66061737114849789544661443739134890062","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11489881286208501558797333161632486099","series":"1.2.826.0.1.3680043.8.498.56019961138123934300068753549912844044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11497761063752461132736045914751707046","series":"1.2.826.0.1.3680043.8.498.11828227454221266337124062658451875174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498845504486569595587386245872511981","series":"1.2.826.0.1.3680043.8.498.10490897969079530364437941524219965949","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11504079342934513987748646905909982588","series":"1.2.826.0.1.3680043.8.498.83661754057304113572066158751101730432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11508340006652282767645009271305921426","series":"1.2.826.0.1.3680043.8.498.20263243137172539645959523747626935592","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11516498203727058146656616413699979433","series":"1.2.826.0.1.3680043.8.498.55260027541471623811376892467496123665","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11521971289581057766555206729274639736","series":"1.2.826.0.1.3680043.8.498.12173751120094584280747934900707629975","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11530446425918247469882351709851883983","series":"1.2.826.0.1.3680043.8.498.75297551628981885359120000851662968184","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11542298537902846156898976374522470463","series":"1.2.826.0.1.3680043.8.498.12213358282906869208407668260512853996","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11545541207342029419461179316053148974","series":"1.2.826.0.1.3680043.8.498.13245654908835864829797563038401066486","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935","series":"1.2.826.0.1.3680043.8.498.53355900447142549760284049936407928123","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11551770225828320378339882849245348148","series":"1.2.826.0.1.3680043.8.498.93381513372233333491197687910932226270","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11561519302314146020052258325459371962","series":"1.2.826.0.1.3680043.8.498.25379697062877044081295161848958893386","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11571057273590444914079281238002921868","series":"1.2.826.0.1.3680043.8.498.30219043134234956704555002341935608127","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967","series":"1.2.826.0.1.3680043.8.498.46557076238541415942574934230838300286","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11579571696748051185342754881791715610","series":"1.2.826.0.1.3680043.8.498.11717710242065313156437228031000666046","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11586616391541918134110642641868079265","series":"1.2.826.0.1.3680043.8.498.16238345368341750914865489782399196737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11592056376890319271545710970155310824","series":"1.2.826.0.1.3680043.8.498.11898951572928062122766572951214119730","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11595338216297251382188195835520492789","series":"1.2.826.0.1.3680043.8.498.22550223174935611453424008780293501617","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11598606533748611505510113033734692457","series":"1.2.826.0.1.3680043.8.498.14535603304351434324560921134079187358","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11604138460812393624691789275263090511","series":"1.2.826.0.1.3680043.8.498.49254522289682662964205199881467257392","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11610176134207131408691324676304640501","series":"1.2.826.0.1.3680043.8.498.84957100687119561770226067080739358202","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11613637647831801354398273334552399874","series":"1.2.826.0.1.3680043.8.498.11453584867368351773197729939361077885","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11629659616217352895144469980856259097","series":"1.2.826.0.1.3680043.8.498.12517197608312229351596494568915776378","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11641866550826314411250696478180861612","series":"1.2.826.0.1.3680043.8.498.11628388114661864410736883151877130262","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11642678020345468499556230871411308014","series":"1.2.826.0.1.3680043.8.498.57180692012960681862417646036764525166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11651755040149549966484720727140190853","series":"1.2.826.0.1.3680043.8.498.11204185280943885723862314799772090895","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11655692245049710970881619360721342623","series":"1.2.826.0.1.3680043.8.498.38728958902616062657662931160456168193","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11661333712307504012375232163708358493","series":"1.2.826.0.1.3680043.8.498.17564630022789307936870766474258900278","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11679044101697033975478370934058085847","series":"1.2.826.0.1.3680043.8.498.12610670222707848423189303757923211925","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11682243736539231948790371952356165171","series":"1.2.826.0.1.3680043.8.498.11077147645787997851240110358799424970","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11689820638581431730501617718741779835","series":"1.2.826.0.1.3680043.8.498.30156440426404040405021093797787083907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860","series":"1.2.826.0.1.3680043.8.498.24869180595160857654818968574082954311","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11709909941833245598246010905322129256","series":"1.2.826.0.1.3680043.8.498.13134253349637761227751389702493531643","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11719502398697390809777045891822044637","series":"1.2.826.0.1.3680043.8.498.10538522924396393001125524279628247572","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928","series":"1.2.826.0.1.3680043.8.498.11781037042568455374218112304207523884","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11723945299462627737346907048285143063","series":"1.2.826.0.1.3680043.8.498.45839012883827155133741591710692736078","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11726452645519338341002251323463736083","series":"1.2.826.0.1.3680043.8.498.31500892862123086648272319156523315490","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11752091432368750506312921028559788225","series":"1.2.826.0.1.3680043.8.498.49949429112825411506066231569436403668","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11762851510866726051342964961256680025","series":"1.2.826.0.1.3680043.8.498.10681792757260951390673010289498737862","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11775352515236862473778507708524782015","series":"1.2.826.0.1.3680043.8.498.13217930799043831527711363533047493597","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11779387602988880691008088661075117406","series":"1.2.826.0.1.3680043.8.498.29948458940367080391860243162460291540","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11786296135006283515679011075884245663","series":"1.2.826.0.1.3680043.8.498.18703713369124905961153118429426669858","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11791028039098776993164396657149877483","series":"1.2.826.0.1.3680043.8.498.13039442310675095913563712581522343891","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11796547950032543541071896156852778697","series":"1.2.826.0.1.3680043.8.498.12754226080259013234821086338218015172","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855","series":"1.2.826.0.1.3680043.8.498.10674642636063197010456542407948227897","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758","series":"1.2.826.0.1.3680043.8.498.20482180143728693770410526665771513819","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11812607848520072826145426247813670653","series":"1.2.826.0.1.3680043.8.498.85407847697620587030669139334864710338","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11823120042104793954689778719229868108","series":"1.2.826.0.1.3680043.8.498.77140656413465007300370999791607948138","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11848828525319749834876185415845381479","series":"1.2.826.0.1.3680043.8.498.81975283709907310568981233914194908936","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604","series":"1.2.826.0.1.3680043.8.498.75774353008974228916065002158103177868","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11867198902144602395671904591496233129","series":"1.2.826.0.1.3680043.8.498.79613518812240947315442666058396719197","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11868389442143514705330687746274512632","series":"1.2.826.0.1.3680043.8.498.10966454150810748156541805003090430580","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11880385031047315524154761508473409762","series":"1.2.826.0.1.3680043.8.498.99160632836031308462290461652454117368","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229","series":"1.2.826.0.1.3680043.8.498.10473191078479931910722256197584763155","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11900355716044995266104285198121445744","series":"1.2.826.0.1.3680043.8.498.65949343165754276499737681557922635690","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11902464730418099436837163594625541861","series":"1.2.826.0.1.3680043.8.498.59008569941238877675120681140178893684","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11906266535122698149967624908585010734","series":"1.2.826.0.1.3680043.8.498.11521370025830027586410873694388138735","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707","series":"1.2.826.0.1.3680043.8.498.41056511439077581640944389642998790973","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11915933561045570822185908784072057715","series":"1.2.826.0.1.3680043.8.498.49073042105078695215276461004712153487","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11920026019322352546372153111621676066","series":"1.2.826.0.1.3680043.8.498.19627956643818311458500735231432133339","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11924999889245769866426422681253524920","series":"1.2.826.0.1.3680043.8.498.24241741434894675073965091412145418330","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11927352926669104658144933891780551483","series":"1.2.826.0.1.3680043.8.498.33560950266970405768561849665380041719","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11935893582662912138522962760478513770","series":"1.2.826.0.1.3680043.8.498.74122219509524546321876267340258665639","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11944762815616453054528091721553484857","series":"1.2.826.0.1.3680043.8.498.48967576305671178839789355196296917426","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11956771495610267604936834092317960600","series":"1.2.826.0.1.3680043.8.498.72558731305494109241974340458255964509","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11964826300307458382604479841253457304","series":"1.2.826.0.1.3680043.8.498.13325327387831831545477829801989431836","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11977465062890611952998635094033472451","series":"1.2.826.0.1.3680043.8.498.12129679189683182783588630115465658804","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11980187124986749407584543247586384443","series":"1.2.826.0.1.3680043.8.498.20283759723743985426473080224981869586","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11988551795896617815207749665102049499","series":"1.2.826.0.1.3680043.8.498.19200054887355134881301271267349822495","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11992172807590757302730513420827032389","series":"1.2.826.0.1.3680043.8.498.38782812191460705487083888583990409868","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878","series":"1.2.826.0.1.3680043.8.498.53694460185787479263402023661402824793","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12009816058271421855131593296751372723","series":"1.2.826.0.1.3680043.8.498.18467163293759786455164202057101201055","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12014959202771141042815134335571931814","series":"1.2.826.0.1.3680043.8.498.89731509730013970115786688955399256740","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12031315219947933845048665397149229969","series":"1.2.826.0.1.3680043.8.498.41778583210906204872994103436434468916","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12036404333388388978317335770611404933","series":"1.2.826.0.1.3680043.8.498.34201137085587514875482850835896687903","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12045300645580015230470991493790324857","series":"1.2.826.0.1.3680043.8.498.12933054542706021706301613835244261197","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12051169109587120248372991913002670813","series":"1.2.826.0.1.3680043.8.498.13207057529644015118965712412708175559","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953","series":"1.2.826.0.1.3680043.8.498.51583327743594349624349783127949429613","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12068674252637046261029360607215822017","series":"1.2.826.0.1.3680043.8.498.53842863208242402680471710397069468905","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12073401429398042797505820525732905336","series":"1.2.826.0.1.3680043.8.498.58321122336212662809258856781916120744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12098718763390783455090805975786780205","series":"1.2.826.0.1.3680043.8.498.90316499787945841268703161068716740257","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12106775587428595108771304868101362618","series":"1.2.826.0.1.3680043.8.498.12124219007849475701598543530023432315","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12110510829608190094905588610131531103","series":"1.2.826.0.1.3680043.8.498.82489281899440205334065589455264631718","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12130868132537772155475981935868588761","series":"1.2.826.0.1.3680043.8.498.37556080528108124864681705742678770834","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990","series":"1.2.826.0.1.3680043.8.498.94448077130340841701274896751522434246","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12148236440353424874546037143134438088","series":"1.2.826.0.1.3680043.8.498.12792512126227846784502519288055230374","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12153755823296803843819361055386265595","series":"1.2.826.0.1.3680043.8.498.92229878059804366882579979607541810234","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12160065317167766993207036411434801297","series":"1.2.826.0.1.3680043.8.498.46294431808464921956878816136471061601","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12166081590548185636776183599317715045","series":"1.2.826.0.1.3680043.8.498.12903137383558124548346418807599252300","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12170753305862973517190756418865414984","series":"1.2.826.0.1.3680043.8.498.69601936141665045530455543548938793653","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217","series":"1.2.826.0.1.3680043.8.498.27771050006302516356918319380221661681","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12176469158911896593955662590695392044","series":"1.2.826.0.1.3680043.8.498.46705286026696618223347560715774368048","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12181591358572249491885248089245858515","series":"1.2.826.0.1.3680043.8.498.70022295668067974367224959502226966008","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12186494601308291174548814564387764515","series":"1.2.826.0.1.3680043.8.498.88789014444501414128844727698405715447","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12195007606427082866449558290311905939","series":"1.2.826.0.1.3680043.8.498.56342913118016756610172775073191374658","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12200672323103829956274910258854933266","series":"1.2.826.0.1.3680043.8.498.18447928318127866883997491822205465150","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12207279366369605175255203969558548753","series":"1.2.826.0.1.3680043.8.498.37001145293727432888522513396773272595","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12215902620611618183321984433114460986","series":"1.2.826.0.1.3680043.8.498.20177697011920872965734146207374555895","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12225975251304101975065271521710735735","series":"1.2.826.0.1.3680043.8.498.59356366004177342518447217047078972546","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12232127448545053821926845887026738988","series":"1.2.826.0.1.3680043.8.498.10351585475805204396049393720510885444","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12238865510201989224648662550299925120","series":"1.2.826.0.1.3680043.8.498.42782954653948084836336282462770128697","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12243764506921404196245986681613075136","series":"1.2.826.0.1.3680043.8.498.73157749419812528762190923180818747936","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12246086238999179953298261549112671014","series":"1.2.826.0.1.3680043.8.498.13184702167487077714054466743104573701","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12256365473385257798561212457071712327","series":"1.2.826.0.1.3680043.8.498.88573797465300343008937348974757699551","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12257484608743859456600273083454447341","series":"1.2.826.0.1.3680043.8.498.11062380002379767131614317198539036810","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12261175520056712414700866621816882336","series":"1.2.826.0.1.3680043.8.498.10883578695996141450217846712500791838","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12266133889557145601371064833389800065","series":"1.2.826.0.1.3680043.8.498.91671871895947518941586754870355034251","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12267944258766605501055560486655358039","series":"1.2.826.0.1.3680043.8.498.10242120377092561768083778499470457344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12273490180961458167930039242826008249","series":"1.2.826.0.1.3680043.8.498.35882728926096629485522377792453728443","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12275265458583331749472987758046974406","series":"1.2.826.0.1.3680043.8.498.11021642679865125650110439243981629483","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12283662652762881783845275662580546217","series":"1.2.826.0.1.3680043.8.498.18674930927019091034493787720733010860","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12285469859900617016859868410323043734","series":"1.2.826.0.1.3680043.8.498.52920089578993921945877431441845610913","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729","series":"1.2.826.0.1.3680043.8.498.21515306653523763613064783220424514345","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12296624595750602443033689366743784387","series":"1.2.826.0.1.3680043.8.498.73052035927321961965033901323848780999","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12301945990161670321760708884248055936","series":"1.2.826.0.1.3680043.8.498.10162155245116653701801632880939068281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12305733049746414359748178404911549573","series":"1.2.826.0.1.3680043.8.498.10245562101387703629408808563319464944","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12314729913655626716039376606973474150","series":"1.2.826.0.1.3680043.8.498.69006855054819705568511467294061632691","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12326930847104022196329390608780066947","series":"1.2.826.0.1.3680043.8.498.31824203430021642015055893438101599837","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12332727488899501354972751168284397151","series":"1.2.826.0.1.3680043.8.498.55631060044364321684532987741749603509","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12335573784261147388984372283481919933","series":"1.2.826.0.1.3680043.8.498.28483336484136695326549223982384249050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578","series":"1.2.826.0.1.3680043.8.498.42503680102683085645138920061605067372","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12348960754791091583201828167607825033","series":"1.2.826.0.1.3680043.8.498.12678272996752264879917250783475358662","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12354685390005531691919754048141542648","series":"1.2.826.0.1.3680043.8.498.62537253039837413478561229714784634143","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12361144153852256276011948786791830435","series":"1.2.826.0.1.3680043.8.498.84170575493055359111860403635407688645","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12366238437963147962850341556508931973","series":"1.2.826.0.1.3680043.8.498.11799563748137755252552378062246116537","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12377852033407536077897684568488365785","series":"1.2.826.0.1.3680043.8.498.63568589035263888267641810542719235601","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12381325071746339154385415220972069782","series":"1.2.826.0.1.3680043.8.498.12612525517429580775667239264184125019","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12386255059403233724886571725482313876","series":"1.2.826.0.1.3680043.8.498.26726549132627442358774389067857645896","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910","series":"1.2.826.0.1.3680043.8.498.12012846866132739640873958327520867072","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12390186846317493876612228407734068726","series":"1.2.826.0.1.3680043.8.498.13041006473096998362541393711271551419","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408186961945791320086301369036829266","series":"1.2.826.0.1.3680043.8.498.34379431925794476399681429533227648188","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408933790879905193765468926022393696","series":"1.2.826.0.1.3680043.8.498.25943734652983877999713267376156006481","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12411396124304166984011280826756258628","series":"1.2.826.0.1.3680043.8.498.10172866265042618201848399951568138585","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12426381920136706157912507136730458010","series":"1.2.826.0.1.3680043.8.498.74076122648921937894848517916369154647","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12435506274406010562396858512839065232","series":"1.2.826.0.1.3680043.8.498.93663436249684607261391822096012982180","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12438775479300781065075835270622538230","series":"1.2.826.0.1.3680043.8.498.19954579048801120223764847580245379469","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12450858400030959723767712312663176092","series":"1.2.826.0.1.3680043.8.498.11526961444506381543155555046798334450","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12471165461877621368840037945454256612","series":"1.2.826.0.1.3680043.8.498.73889806331741942409995417572834683898","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12473755232610784318190599222252893949","series":"1.2.826.0.1.3680043.8.498.12860622582961800565646371415873334732","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12484700311993345120015691857089576898","series":"1.2.826.0.1.3680043.8.498.10822861933781504112846118938982684628","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699","series":"1.2.826.0.1.3680043.8.498.89902604523451921086310052049782799144","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12491395125580387847235487841392723348","series":"1.2.826.0.1.3680043.8.498.77657134406610723985453108562447745069","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491796764641306046176352066215240074","series":"1.2.826.0.1.3680043.8.498.33118459143370246234610331614793196300","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12497144849433082154223897356471317072","series":"1.2.826.0.1.3680043.8.498.87266952665219795800797785421356279112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819","series":"1.2.826.0.1.3680043.8.498.41911386079999648645730179349529867886","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12509825237087140646629745447753158673","series":"1.2.826.0.1.3680043.8.498.88922810938770847212169376923207602009","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12514357232100339485873453417784586592","series":"1.2.826.0.1.3680043.8.498.68657038825317037143609816574019691843","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12520203592477518476890010185037272575","series":"1.2.826.0.1.3680043.8.498.29189722062475450062327030875467968364","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12523676732876902778577970655429060858","series":"1.2.826.0.1.3680043.8.498.58211784878373746752777325606501620572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12526876471494728147635430816990398219","series":"1.2.826.0.1.3680043.8.498.31854964090195592349303964596840693958","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942","series":"1.2.826.0.1.3680043.8.498.77738882407207531314197472772164625107","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12547308734950397389859712060653254111","series":"1.2.826.0.1.3680043.8.498.47337290545922212670144401860662493428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12553010776519233077752150342664160826","series":"1.2.826.0.1.3680043.8.498.32155610372324202807052686056367898238","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12554824497579383319980187351290254727","series":"1.2.826.0.1.3680043.8.498.86406904388991553240777089011229728023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12568337973304376222940287737691898395","series":"1.2.826.0.1.3680043.8.498.24014121320642853623111841740566509924","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12570576337098834311171883034999327454","series":"1.2.826.0.1.3680043.8.498.95820781154415891769997810889894942600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12576055921678075813718486704978381993","series":"1.2.826.0.1.3680043.8.498.88238994076514642388988388522756590852","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431","series":"1.2.826.0.1.3680043.8.498.32888321929909832918725866186239671012","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918","series":"1.2.826.0.1.3680043.8.498.52144614390949620175088532265017443260","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973","series":"1.2.826.0.1.3680043.8.498.37032826821035854589423026056104771845","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12607659092357834912029375690348886159","series":"1.2.826.0.1.3680043.8.498.75297579259988973493932554142852438862","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12616376042564380071463081913949255574","series":"1.2.826.0.1.3680043.8.498.82680017529992827460752050673462747027","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12620718517255799151283445360332799702","series":"1.2.826.0.1.3680043.8.498.10007282253941170661178728530472447162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12629372332421954368597330267888483345","series":"1.2.826.0.1.3680043.8.498.15530256794376691030562863615998412608","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12632129640405661298623288977632682289","series":"1.2.826.0.1.3680043.8.498.11581716569423236536531063588197777411","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12638205949145637289277744267473965719","series":"1.2.826.0.1.3680043.8.498.13275948419294893192083304382691888381","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12642641661988102031826128876502303807","series":"1.2.826.0.1.3680043.8.498.60268876120540555021221043617575546198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12653936535855858693516515692319733265","series":"1.2.826.0.1.3680043.8.498.43244374337995939656775695711089539448","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12658572494577556884626613972035441625","series":"1.2.826.0.1.3680043.8.498.10833991747666315114829377886113887764","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12673561338227595483297605409292531492","series":"1.2.826.0.1.3680043.8.498.37237683873352233119101198808311995248","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12674930973831274527829261576087154644","series":"1.2.826.0.1.3680043.8.498.53674084167489037735940693436304262443","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12689874339887047433232628649886917274","series":"1.2.826.0.1.3680043.8.498.99271579979705848183952489651913093657","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12690803821768205116702487219908820630","series":"1.2.826.0.1.3680043.8.498.22753445121041999578522834735708064403","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12699751030439008569144814819343161608","series":"1.2.826.0.1.3680043.8.498.93622817279100835202963039864105437751","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12701097867143790763644826659974337046","series":"1.2.826.0.1.3680043.8.498.76754197855818892385275519172615707311","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715067947657150279232238280712066972","series":"1.2.826.0.1.3680043.8.498.47903276551151541610831133601133694948","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715402081094788880283826571710131940","series":"1.2.826.0.1.3680043.8.498.10908632121116155814181161097885609598","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12718966639501121738042125605376714231","series":"1.2.826.0.1.3680043.8.498.58341292142232844791855133891362456476","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12724250000565012779853884787450109956","series":"1.2.826.0.1.3680043.8.498.11884029997212277863610718935370069821","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12727488524631885827798129418558343386","series":"1.2.826.0.1.3680043.8.498.10098199684290396964958046291222691747","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12748840878706905309050091576449488666","series":"1.2.826.0.1.3680043.8.498.11986208703289371259819059330414278332","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12759704450656741777120047828560448622","series":"1.2.826.0.1.3680043.8.498.10368646287856025912294032513214210218","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12762900657007480138202359874575403361","series":"1.2.826.0.1.3680043.8.498.70311104296638405289064509993017623649","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12769887673556938425710414718385118283","series":"1.2.826.0.1.3680043.8.498.12967197914567159634543994418926895429","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12775050278338123442507219047972133270","series":"1.2.826.0.1.3680043.8.498.42580531923665779341931355586643006128","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12778920060586364664953350981285170949","series":"1.2.826.0.1.3680043.8.498.11271902850800909227481197216695316646","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12785620442256422697216192462703273802","series":"1.2.826.0.1.3680043.8.498.78238859714602267067933232532555582321","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12792431109288033212787919118858331455","series":"1.2.826.0.1.3680043.8.498.87721904063003239485530551803539481749","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12799849540231315910448269889175806934","series":"1.2.826.0.1.3680043.8.498.98141110304854730295772967159852752024","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12801643268939133682891906705186571010","series":"1.2.826.0.1.3680043.8.498.28407586108779789857148137033041798193","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12810074901642984174782853829149619043","series":"1.2.826.0.1.3680043.8.498.12028015624082793300229800710890100004","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12815132582382825980088023336277934996","series":"1.2.826.0.1.3680043.8.498.21244034621683285947090805668526677117","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12819437506700585448407291050085063174","series":"1.2.826.0.1.3680043.8.498.52288664478645389042682581376064092796","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12826828527176716109949704557150950675","series":"1.2.826.0.1.3680043.8.498.70622328281381000088206481025488391528","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12831769902826114242555323443809541052","series":"1.2.826.0.1.3680043.8.498.48621946716071938384247453877155194432","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12839498544183083639548298117995964432","series":"1.2.826.0.1.3680043.8.498.12986950124060022872099122618407006988","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12864336642218457767597053831458716404","series":"1.2.826.0.1.3680043.8.498.13337975248392262079830617706473600663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12896740277483734090633132494285615502","series":"1.2.826.0.1.3680043.8.498.40700610248737343081188984717465491366","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12907457183137517669505067507754394039","series":"1.2.826.0.1.3680043.8.498.44099058493749184911666285628712307110","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12914677763148740041755977075799271057","series":"1.2.826.0.1.3680043.8.498.46750872662364497967657682961931800704","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12919303621342250989179219420795277559","series":"1.2.826.0.1.3680043.8.498.11369560928409037122147737117847597476","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12936296973456045538701035923586653502","series":"1.2.826.0.1.3680043.8.498.19125380753075891196759915303526067956","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12946163460972553911280889205685745877","series":"1.2.826.0.1.3680043.8.498.37379531980110846886072714378673567987","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12949322114105642048256996295428966924","series":"1.2.826.0.1.3680043.8.498.65794416608323322541609288811153246311","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12957252711278416684461267069498115130","series":"1.2.826.0.1.3680043.8.498.10247301311681188734606666222546041603","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12958999546496283565938189274771483491","series":"1.2.826.0.1.3680043.8.498.12506105094362119234838979672684630824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12968535455086298222684461157825070395","series":"1.2.826.0.1.3680043.8.498.35674353065464059879376427518584455354","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803","series":"1.2.826.0.1.3680043.8.498.95737920059944686538141068233888690539","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12981413356524881858859912294078498486","series":"1.2.826.0.1.3680043.8.498.10414044527359773490155137354355415523","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12985683732818911488433522743768046254","series":"1.2.826.0.1.3680043.8.498.12214659316688634807983610859729101090","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12986517636774565418070546261705439006","series":"1.2.826.0.1.3680043.8.498.12048244575051588554459454894402900151","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13003520962744192762179786723724282013","series":"1.2.826.0.1.3680043.8.498.68469219914120876147954061629664256127","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13009271274636013067662786774946029285","series":"1.2.826.0.1.3680043.8.498.63603923571849050581553056440305019414","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13017199074506191657941905498813292773","series":"1.2.826.0.1.3680043.8.498.57533223893702357661294131771348894935","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13024250150960117282590397343184827466","series":"1.2.826.0.1.3680043.8.498.30554393399729010709088188416948889460","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870","series":"1.2.826.0.1.3680043.8.498.11144603881346141657717536917524229354","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13027024577729659065187419038453181852","series":"1.2.826.0.1.3680043.8.498.62882889127229871686192780838291152599","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363","series":"1.2.826.0.1.3680043.8.498.61573382429891583248566517726172307372","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13035632998746261521316395720102873141","series":"1.2.826.0.1.3680043.8.498.70023403105377432174962470255148010943","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13062438493893996542058869514390908492","series":"1.2.826.0.1.3680043.8.498.12170598572972347855818252328813421560","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13065120731888368919957856813755701101","series":"1.2.826.0.1.3680043.8.498.20009363618206079505967417106678315673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13072908567275370866813748886351904593","series":"1.2.826.0.1.3680043.8.498.22065518098540042153391083553310413744","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13078009378379325641560321462188164769","series":"1.2.826.0.1.3680043.8.498.43894969330099066291250222070568843509","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13078834627392793401892111941627273421","series":"1.2.826.0.1.3680043.8.498.13419898958792828572608370029750057298","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13086260989634665810181105278930088749","series":"1.2.826.0.1.3680043.8.498.20409627563850415106397655462862952688","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13092094949394905924922534008771583745","series":"1.2.826.0.1.3680043.8.498.11482656292757033789904906823427491296","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13093751810869039645959119469231437894","series":"1.2.826.0.1.3680043.8.498.95951589738756470057247404834108574969","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13099702881219862576749493772702888064","series":"1.2.826.0.1.3680043.8.498.15240229991771372897769910665399898811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590","series":"1.2.826.0.1.3680043.8.498.27535139479026182224816448702604623225","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13114693814781737495494700902699619138","series":"1.2.826.0.1.3680043.8.498.12912664153482988258323003806478904065","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13118325043138522704531128725514354285","series":"1.2.826.0.1.3680043.8.498.10544213872578968001681618528970630106","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13126070481622692281046898445918832841","series":"1.2.826.0.1.3680043.8.498.78533765009703181277921374886673131331","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13128352910661672966143173193241171100","series":"1.2.826.0.1.3680043.8.498.31240499032036142443908545494025879078","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13129258242739880102472095242703169773","series":"1.2.826.0.1.3680043.8.498.12854863786638217202897329848859230712","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660","series":"1.2.826.0.1.3680043.8.498.95082009562190063890137445124750743279","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13136367866260644484250529322937554927","series":"1.2.826.0.1.3680043.8.498.10772557159629787087344600612994589129","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13137510961466386836387239510287919268","series":"1.2.826.0.1.3680043.8.498.11530584418390449632081707850531322765","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13139684164906828131777150061532685584","series":"1.2.826.0.1.3680043.8.498.10411616519664381572519590389315562281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13147249459968468500804358644901642270","series":"1.2.826.0.1.3680043.8.498.43980687171532504774720685920111009403","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13150780733560495803861509587935093776","series":"1.2.826.0.1.3680043.8.498.75996334913980370088534013586239214447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13152690996267297597178587721720788387","series":"1.2.826.0.1.3680043.8.498.70525060792695307293962415809468581587","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13157776420553094741176082047621957775","series":"1.2.826.0.1.3680043.8.498.74739965214954546010394491439709965098","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13158644154518335426512578922874968296","series":"1.2.826.0.1.3680043.8.498.27911153680377862749738355539772051491","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13163727085668231782866823012535268245","series":"1.2.826.0.1.3680043.8.498.12607537613356709830372044641169446143","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13167727300538353052482338786183678684","series":"1.2.826.0.1.3680043.8.498.12249557648092799238989408928993028124","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13173781757294564816839795307893228067","series":"1.2.826.0.1.3680043.8.498.12434935574145978968908583269826929210","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13176146790594027813190776550419533937","series":"1.2.826.0.1.3680043.8.498.15762526958004536018423594455958591370","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13197409499722328227876152940338160369","series":"1.2.826.0.1.3680043.8.498.12948478195387372753671886049475465827","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13203246712933982189680087013566455075","series":"1.2.826.0.1.3680043.8.498.86112517179774720174442330902939890560","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13210084125872700841685814144002495145","series":"1.2.826.0.1.3680043.8.498.11679138615416605394300681408319528584","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13212280182729018182073794487313115605","series":"1.2.826.0.1.3680043.8.498.72845501398471386536221221540643977538","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13224933005220329091792052916905776921","series":"1.2.826.0.1.3680043.8.498.93319098130843696153014887822815453600","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13229563888378977350299319159748823477","series":"1.2.826.0.1.3680043.8.498.27869674715378862224829808342924745879","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13234278816788324761159326624155958655","series":"1.2.826.0.1.3680043.8.498.12999885135467675044392619983853852057","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13245409855393847268063261203253424169","series":"1.2.826.0.1.3680043.8.498.11128716295312250059188179408483704349","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13261595972961358158990410740461308054","series":"1.2.826.0.1.3680043.8.498.15449252141627639375107549460899320980","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13262473718294425031790270527413368245","series":"1.2.826.0.1.3680043.8.498.13307704177961220165775762642033054737","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13267708980958529833316446819156908381","series":"1.2.826.0.1.3680043.8.498.12567641413989519845738256162423557707","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13269841239774089690227054060975903636","series":"1.2.826.0.1.3680043.8.498.83950279556675315393700306013850947534","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13283411144589724334502427926341646116","series":"1.2.826.0.1.3680043.8.498.12370137687483671935508955129717304455","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13286048014183811835847662577023535662","series":"1.2.826.0.1.3680043.8.498.66400697942022808831101126789783120447","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982","series":"1.2.826.0.1.3680043.8.498.43792449251722232927072959376323974045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13295633458001436592005068637719195786","series":"1.2.826.0.1.3680043.8.498.68191670574750869061861475357914526610","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13309299049346932814529962458932580753","series":"1.2.826.0.1.3680043.8.498.38217738737327077731992406456795800810","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13319886927127591971322856062062654146","series":"1.2.826.0.1.3680043.8.498.12826005963645536975302238408240478715","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13323187196249612936800308722675884644","series":"1.2.826.0.1.3680043.8.498.66862429273085349590085143151280455774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13332521365069604440230185026865466121","series":"1.2.826.0.1.3680043.8.498.86384604708792166666291125601697947334","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13335786652653204791740906118815827207","series":"1.2.826.0.1.3680043.8.498.18570878536523898215744519073510451173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13353409340862821090624127486679097099","series":"1.2.826.0.1.3680043.8.498.46877612393197212793968702043928232274","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13357923062946813820390731946535198170","series":"1.2.826.0.1.3680043.8.498.12543820180205307415494391796554677425","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13363562142644508091380459785810556567","series":"1.2.826.0.1.3680043.8.498.83407990508130202259819544763061269191","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13367740087756379568826046059252678700","series":"1.2.826.0.1.3680043.8.498.13024619671941973885621766021623968203","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13369443704765101449274062863864122737","series":"1.2.826.0.1.3680043.8.498.55433222188467286107818097612740721157","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13377426362348493372112241208607685717","series":"1.2.826.0.1.3680043.8.498.80531221482059316948077072839139840934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13390551515761269433524925969561757962","series":"1.2.826.0.1.3680043.8.498.26249791782192171881033808838775284271","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13393268272072264469105890769640363680","series":"1.2.826.0.1.3680043.8.498.11490964504074457321683350279046368133","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385","series":"1.2.826.0.1.3680043.8.498.10115177203059217841409681480824139468","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13440909104569690641314836192184434461","series":"1.2.826.0.1.3680043.8.498.78215186478283206718932639891393172840","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13460078631594032326006554321483799928","series":"1.2.826.0.1.3680043.8.498.44001678589625757006984355925387560955","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13461073016327661159616957998589287085","series":"1.2.826.0.1.3680043.8.498.89330924282837497351038943171171149912","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13505671694940677161128783863912245059","series":"1.2.826.0.1.3680043.8.498.12074473792091085913487088138840229315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13616982533395291590655841043722813973","series":"1.2.826.0.1.3680043.8.498.46109348177256336877322115539076538893","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13695989890112408005495264068064110179","series":"1.2.826.0.1.3680043.8.498.46994041934285903958945317843925646630","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13762716088334302039143651507177887969","series":"1.2.826.0.1.3680043.8.498.54059918329460660895236762513494833829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13879163301346984697246311226647565712","series":"1.2.826.0.1.3680043.8.498.72983662659814230231263866943509588041","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13972325309264527611989585341636140156","series":"1.2.826.0.1.3680043.8.498.92595789272599769707955166092086885419","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14024129019390041106306062001538710978","series":"1.2.826.0.1.3680043.8.498.11383743949272989213711679118934560170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14066537811671722419109228137841995847","series":"1.2.826.0.1.3680043.8.498.25664256745516102490840715102963696604","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14137431039800236510749401303688858447","series":"1.2.826.0.1.3680043.8.498.72211981036528367939653311881383961372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14229598279617168024212382336641104878","series":"1.2.826.0.1.3680043.8.498.10817292517810590487790339797526502182","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14300107110014472682373589824643789093","series":"1.2.826.0.1.3680043.8.498.10935172567704357104145038779079359299","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14377917653321813299237274871837695586","series":"1.2.826.0.1.3680043.8.498.11721344741570665399553262162945717680","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14453847272947043746856632410254555895","series":"1.2.826.0.1.3680043.8.498.28011861625140900968117786968840272157","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14477978091360159774052709992034078166","series":"1.2.826.0.1.3680043.8.498.12512979049929551486649664757378615153","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14495028403937132710814780922642512791","series":"1.2.826.0.1.3680043.8.498.62877609021287648596959995232926073888","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14558486491688263090686005096911018438","series":"1.2.826.0.1.3680043.8.498.12663407850658002522049775289371245633","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14744714658078107007743063095375646143","series":"1.2.826.0.1.3680043.8.498.42124904643662366714124313222102794977","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14776241668061173271058268311868302014","series":"1.2.826.0.1.3680043.8.498.60873830037045327494189352948588014839","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14850534390650184292244380816212794996","series":"1.2.826.0.1.3680043.8.498.50466301514188270757926622338695634764","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.14959732862523218997149041503831001633","series":"1.2.826.0.1.3680043.8.498.46471467449732835125319499493522735051","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904","series":"1.2.826.0.1.3680043.8.498.21006847893738543560570001907414512147","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15065556607080211420503043544482923211","series":"1.2.826.0.1.3680043.8.498.71738527975707524459525867058092936730","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15133818248365103660456294775831248835","series":"1.2.826.0.1.3680043.8.498.12416874256364079701624007103364157157","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15219824433491621460467738389345697247","series":"1.2.826.0.1.3680043.8.498.62442469813000649442734664254118050701","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15337067581747365729183643142512558411","series":"1.2.826.0.1.3680043.8.498.12546750180485779506634171741176682250","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15376535778246978844757121776560020722","series":"1.2.826.0.1.3680043.8.498.48530032898890762142779584359393210126","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15428051518761311558410606976607852351","series":"1.2.826.0.1.3680043.8.498.11127204938666287203625571430310150518","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15486463042236628073590077209793828847","series":"1.2.826.0.1.3680043.8.498.13081210567550015669803608045184283627","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15539098084847502447238233198010208924","series":"1.2.826.0.1.3680043.8.498.10643193514369891755576991881839301690","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15599161312748548709240395406977042093","series":"1.2.826.0.1.3680043.8.498.40693559505917368411549833086991265613","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15709232000390534671340322807442769184","series":"1.2.826.0.1.3680043.8.498.12169600960700386311549102199459944392","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15863325224755490561125835019647303556","series":"1.2.826.0.1.3680043.8.498.12478491249330863400266865298315760259","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16046164723877343553376490423345098256","series":"1.2.826.0.1.3680043.8.498.39114355630313663203476036726817791727","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16101435445990180482665354366724730305","series":"1.2.826.0.1.3680043.8.498.76970929267106268308914765223848953016","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081","series":"1.2.826.0.1.3680043.8.498.45478483414904893207897805994444151293","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943","series":"1.2.826.0.1.3680043.8.498.78043287320720592978158748691677617840","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16318078762749420171833135145622117452","series":"1.2.826.0.1.3680043.8.498.35234113749316087053218419663685116117","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16370303687278282143772106415664171358","series":"1.2.826.0.1.3680043.8.498.82998045061814207795341502245074953641","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16469256144078887607437578587242258846","series":"1.2.826.0.1.3680043.8.498.19460431447325008038979709834301417546","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16534078612126477228267765508485297367","series":"1.2.826.0.1.3680043.8.498.58388371238280647527279841970673406078","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835","series":"1.2.826.0.1.3680043.8.498.10248087719476589017843109082789164432","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16657553974040122719239506008948241297","series":"1.2.826.0.1.3680043.8.498.10529657333589879078593570090289763817","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16674237773422474998796191258459643061","series":"1.2.826.0.1.3680043.8.498.59414769682066749223732363306105632353","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16709696133689803501344038926512501779","series":"1.2.826.0.1.3680043.8.498.13363390236927506021792822866837808222","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16774450022563283092653506225603012645","series":"1.2.826.0.1.3680043.8.498.52479789100608413093718629123990581590","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16843381581429596234606436682284624187","series":"1.2.826.0.1.3680043.8.498.57902422303173429217982567541619634023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16864649493021092944977766326505053153","series":"1.2.826.0.1.3680043.8.498.77587893353987852937710224810902634180","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17049315826418217889213226790962592213","series":"1.2.826.0.1.3680043.8.498.92991063145772262679524280827617643931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17074746747536999918446136673966046653","series":"1.2.826.0.1.3680043.8.498.22877297944776162015416544805070485593","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17093025979426350838724553020541643719","series":"1.2.826.0.1.3680043.8.498.68672773707868850161184821229583273706","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17123244662958741977525806700986256210","series":"1.2.826.0.1.3680043.8.498.54879101269827880780943651579300513124","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17241550277273207658303011467222594165","series":"1.2.826.0.1.3680043.8.498.56375615879889358609259967031325979401","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509","series":"1.2.826.0.1.3680043.8.498.20305377716969141454548673873550176504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17389120886685633448235711042721669823","series":"1.2.826.0.1.3680043.8.498.10327837334716075578292438576117163289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17427902052389033416295416299697276519","series":"1.2.826.0.1.3680043.8.498.76234020749678031049385438694578717513","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17596657287050550465463272509976106854","series":"1.2.826.0.1.3680043.8.498.11510447329951609725860197496498264585","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17662880041629379035049367252086125592","series":"1.2.826.0.1.3680043.8.498.86719482378523514047542359609870528803","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17741907690913585163443565102293192244","series":"1.2.826.0.1.3680043.8.498.45852534962379304084567914948764204561","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17769942698581399165275319502954100325","series":"1.2.826.0.1.3680043.8.498.84708640769448954628924463003333037496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711","series":"1.2.826.0.1.3680043.8.498.42927162914563423947167748937620376779","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17865613300681542407687000381594067366","series":"1.2.826.0.1.3680043.8.498.12643413171994340255424796706769211717","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17931021655186901394796168474942485728","series":"1.2.826.0.1.3680043.8.498.54783523305456002398679325819694820326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18080511167840608448833484000937072258","series":"1.2.826.0.1.3680043.8.498.27745898060703585699184067998517658860","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18092766733837308394607532608319746043","series":"1.2.826.0.1.3680043.8.498.12634367675822811374273849655485884755","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18175450083338316799857698898702333009","series":"1.2.826.0.1.3680043.8.498.27663991349576346857906529584056970961","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18199465156572252531439880765733639584","series":"1.2.826.0.1.3680043.8.498.56806124599210622308553163270409047857","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080","series":"1.2.826.0.1.3680043.8.498.12427687198117894644342826723421670255","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18350185433331570523543207086298834067","series":"1.2.826.0.1.3680043.8.498.54217611392100165970381736916901886853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18403627643718880914843209837672242908","series":"1.2.826.0.1.3680043.8.498.42298905803013548293596982981586411347","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18529656610752887218162968403101035390","series":"1.2.826.0.1.3680043.8.498.10992008002163844878405768578004676151","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18614704377104425868545800677445972261","series":"1.2.826.0.1.3680043.8.498.10390956308472917307180400175524267678","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18653838887148864162027930218548124748","series":"1.2.826.0.1.3680043.8.498.11372382960242173938833799008133506369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18695727379446620258431748063749176639","series":"1.2.826.0.1.3680043.8.498.87711795707485121760670841348797784990","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18758199166360428760124471581842191566","series":"1.2.826.0.1.3680043.8.498.20879478117965970989038957036168486753","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334","series":"1.2.826.0.1.3680043.8.498.93330971076275081719785851308495021450","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18879049235810379709186936621085221483","series":"1.2.826.0.1.3680043.8.498.60263611658299911274428032924867107235","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18968967345606818331567480514437075646","series":"1.2.826.0.1.3680043.8.498.66182567046913710720817055709133238392","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295","series":"1.2.826.0.1.3680043.8.498.11942774234894508724279171863947672415","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19124879088354547836118593585646705198","series":"1.2.826.0.1.3680043.8.498.12025081159438997817439153531460131301","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19194090255958016414082788444327981898","series":"1.2.826.0.1.3680043.8.498.71875959274894439316245182966355975378","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.19198152070740205434848422424805073748","series":"1.2.826.0.1.3680043.8.498.36846584531948926903360253109571691160","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19342628541833173544083802327412975225","series":"1.2.826.0.1.3680043.8.498.13261728522305402844964071111650502529","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19439950168933299231685815631786334293","series":"1.2.826.0.1.3680043.8.498.69780733583753360000196348257462939838","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19512754570664577075249836633718382506","series":"1.2.826.0.1.3680043.8.498.21740322436293871740715002903990674908","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19573195389434178432956976770560221462","series":"1.2.826.0.1.3680043.8.498.98102111057730033016064343767061849990","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19624674714110931897302481276137067638","series":"1.2.826.0.1.3680043.8.498.13276839091817892595141422130800579912","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19722687746450343001601428638894755031","series":"1.2.826.0.1.3680043.8.498.40977185836289154112940589928237589500","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19821498229702528191367700655079805824","series":"1.2.826.0.1.3680043.8.498.49439013289068714743016242437967613713","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19832308119617081676615236082965891208","series":"1.2.826.0.1.3680043.8.498.43900428940670302294366505113320708945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19887850741193535685529812263489892001","series":"1.2.826.0.1.3680043.8.498.56288722371435353867363112303841189347","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19925337578871911327225897926650188274","series":"1.2.826.0.1.3680043.8.498.48160277096113644370934238934486402829","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20023559426196341254845485444587607391","series":"1.2.826.0.1.3680043.8.498.57177290392493442737175320599956334090","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20123813962665544845986028832159432587","series":"1.2.826.0.1.3680043.8.498.12687860397380878968117022462647254996","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20184541870535223885310764061146422870","series":"1.2.826.0.1.3680043.8.498.10801901960200854666072300884511585263","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20204907530808218314127264658715054041","series":"1.2.826.0.1.3680043.8.498.10019090832164579635579350690996815081","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20266523169651039493799975212435349766","series":"1.2.826.0.1.3680043.8.498.80456535139889483603517687328453906405","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20375739629244657719385962474248679850","series":"1.2.826.0.1.3680043.8.498.12696950871477160937137670393072296598","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20472525526345683475357475395677168372","series":"1.2.826.0.1.3680043.8.498.10059648484821638292067302546527303351","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886","series":"1.2.826.0.1.3680043.8.498.93875345827262994385398561395266431044","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20522330861615570894210641293344468905","series":"1.2.826.0.1.3680043.8.498.12941800252868258448288615444854008946","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20600318722531772866761132262772105640","series":"1.2.826.0.1.3680043.8.498.17985907610433988782076131883221955471","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20690227395174139480327772367312879333","series":"1.2.826.0.1.3680043.8.498.26775183330095578301837582671361567926","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20801986011906346655458394198951999413","series":"1.2.826.0.1.3680043.8.498.97804407530687459313804518213257662216","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20852351697358589606507109300359572624","series":"1.2.826.0.1.3680043.8.498.16457575526252172518464981547230881582","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20947841686457891570351579406919873254","series":"1.2.826.0.1.3680043.8.498.43202764352700341319811140202790885653","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20969353615978236040448556356026349615","series":"1.2.826.0.1.3680043.8.498.47176825864135895710686224061910027541","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21060274255072711241653174133146945101","series":"1.2.826.0.1.3680043.8.498.10791145151809893065807069122194605907","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21135201439483157418374033420156525658","series":"1.2.826.0.1.3680043.8.498.10145114822375225288718996935566071440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21294732642571417618432023007693943849","series":"1.2.826.0.1.3680043.8.498.72271069959084886916207029783721765300","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21322638458116180116030396109651929596","series":"1.2.826.0.1.3680043.8.498.11789656762228591310153584082881387402","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21413285047211074156709106352750752275","series":"1.2.826.0.1.3680043.8.498.68305400410269757862628892935103716658","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871","series":"1.2.826.0.1.3680043.8.498.12159773988930868854146694845458972150","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21558570608416990974711670673431746511","series":"1.2.826.0.1.3680043.8.498.86521153825429427634939857350963424777","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21605506484276750075895228202564072044","series":"1.2.826.0.1.3680043.8.498.43138885607690036363029502611030494413","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21636337451527049130190928852193267826","series":"1.2.826.0.1.3680043.8.498.10966434502421244547605113349276759868","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21769352980698937182060933429354012104","series":"1.2.826.0.1.3680043.8.498.12373081538281234525464917373363268914","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21868734450232158209414868901539475548","series":"1.2.826.0.1.3680043.8.498.79834466185023983330164474177089269391","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21937000442536121262190458446148124910","series":"1.2.826.0.1.3680043.8.498.69000112300418561227334194392910430014","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21991252201684723900045301917670383228","series":"1.2.826.0.1.3680043.8.498.15410608987934641344077739465114552356","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22025729757015485203756049440674914537","series":"1.2.826.0.1.3680043.8.498.32021150219773952021041742066644123061","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22057596860528658818833903685827585805","series":"1.2.826.0.1.3680043.8.498.10092747845942312574782012294203278484","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404","series":"1.2.826.0.1.3680043.8.498.12149801966763404687544209324060987440","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22126824950877890715014783370071292997","series":"1.2.826.0.1.3680043.8.498.13370981123290044053927844686603836919","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22249763791584161324425922920269052844","series":"1.2.826.0.1.3680043.8.498.26568157629512677976650654959351694741","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22338400095583099753286935111647807038","series":"1.2.826.0.1.3680043.8.498.47556242857269833829110775910114942516","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22398041676780235097751294563297953841","series":"1.2.826.0.1.3680043.8.498.12252424651213973593312630373490693001","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22407175411826914385667795929161924645","series":"1.2.826.0.1.3680043.8.498.48182290657297836144331138513279555031","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22480218016816209183384996714060953859","series":"1.2.826.0.1.3680043.8.498.90030338670340302007346245726134429873","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22583985355672251619550173090948161714","series":"1.2.826.0.1.3680043.8.498.33833519385982759175172555120672575521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22596004572382994883130705870731193046","series":"1.2.826.0.1.3680043.8.498.10214927844145097152934346228439400282","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22632919018724241015271844108954804020","series":"1.2.826.0.1.3680043.8.498.24196107334076803498288469665124595772","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22684505799921376181913295621462268931","series":"1.2.826.0.1.3680043.8.498.93893970097754189365376722168899667707","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22867776500261605752195206933939172136","series":"1.2.826.0.1.3680043.8.498.39227238541207898441282647022853694775","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22985467719058223198642920273233975053","series":"1.2.826.0.1.3680043.8.498.48635335020683727895056146044605234993","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.23043120269252616884307662769766701449","series":"1.2.826.0.1.3680043.8.498.86012814429464121056174275143820312401","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23065868481290440008286663193810913414","series":"1.2.826.0.1.3680043.8.498.72867756698114455956316804867623491855","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23081294963277525274551516232134221416","series":"1.2.826.0.1.3680043.8.498.13136062992783739368184395353900269656","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23166300530486600338569917595744444657","series":"1.2.826.0.1.3680043.8.498.67370592962379132871928521902421550782","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23394601328148358087542766577727876635","series":"1.2.826.0.1.3680043.8.498.42189912109337008878817651050020505382","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23422918444346962998731376196027420200","series":"1.2.826.0.1.3680043.8.498.57286878989555251271728863883434818266","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23523176367321534630723586470190430254","series":"1.2.826.0.1.3680043.8.498.10384888573508554161005993734963741711","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23686732555936573712275826587678502899","series":"1.2.826.0.1.3680043.8.498.12753711859665227330448716520505256174","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23719982492392272262684855675016475265","series":"1.2.826.0.1.3680043.8.498.88214894297123788771912263709940547164","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23822313132245448173452915228147497501","series":"1.2.826.0.1.3680043.8.498.15380244927879894692543638461914823221","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23849672490426589132734390202638283865","series":"1.2.826.0.1.3680043.8.498.99848987582985764988986949684418713934","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23956169649865635269920565218680494366","series":"1.2.826.0.1.3680043.8.498.89464265112945110647654933898968586691","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.24012076387083114283116476459425015413","series":"1.2.826.0.1.3680043.8.498.70698958745501737055918043326456226859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24126422042626612740219325272793737656","series":"1.2.826.0.1.3680043.8.498.13115708793194672010984090462193265002","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24220732394554303343522053195582322788","series":"1.2.826.0.1.3680043.8.498.11882051271841449564250567409175454930","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.24283484945988825507450239904606754405","series":"1.2.826.0.1.3680043.8.498.92330388423988887522792188353535312934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24302889622350917085868032747474929504","series":"1.2.826.0.1.3680043.8.498.10155922706019224296169209173124101364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24432243084670627958434695475653885175","series":"1.2.826.0.1.3680043.8.498.29876621372733732833737754137781827351","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24529300482982370922479679056174585978","series":"1.2.826.0.1.3680043.8.498.10279241857036589738247656422922121811","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24563801799946310644914912532825629118","series":"1.2.826.0.1.3680043.8.498.12093509788867022895407332393883429219","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24661137747502154216858169258339731263","series":"1.2.826.0.1.3680043.8.498.82265794394407384087285396800868977082","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24680736086205072406257935876858402593","series":"1.2.826.0.1.3680043.8.498.93262057968373100540166431453530390210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24808246532191535554375304081795021306","series":"1.2.826.0.1.3680043.8.498.12345272716554220910940456062879625020","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24866943620580666216571030343221534720","series":"1.2.826.0.1.3680043.8.498.41778198489689697090185735136597299205","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983","series":"1.2.826.0.1.3680043.8.498.61480987283897623659434166661785670290","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25010661650105533302140829168731585935","series":"1.2.826.0.1.3680043.8.498.32916724983246481099074088426518794990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25107837907285885353845034555062540189","series":"1.2.826.0.1.3680043.8.498.11320146095100417690709768043651414471","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25156207169523051276979595262558652072","series":"1.2.826.0.1.3680043.8.498.85208634253484501451611946990309061448","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25196393144875332073285250936506555534","series":"1.2.826.0.1.3680043.8.498.10206706303591790916343722924639349073","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25259588756496584739814764896903786709","series":"1.2.826.0.1.3680043.8.498.12706372671039627066444348439769293058","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25292715244575997322139502072242615319","series":"1.2.826.0.1.3680043.8.498.24655911172679739406536869379765628357","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670","series":"1.2.826.0.1.3680043.8.498.11464340350754606852236416337102338616","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25480883220477368504575353972328124891","series":"1.2.826.0.1.3680043.8.498.41268965822871675631415849849761655987","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25512240159357553134500245590935010104","series":"1.2.826.0.1.3680043.8.498.40152678904691290443013272167975230856","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25550871519416839807700061317524004550","series":"1.2.826.0.1.3680043.8.498.27659768727892303944953632378525842968","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25616615589791001678067099907823838797","series":"1.2.826.0.1.3680043.8.498.31001125480979637711093992594457604318","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25643897330283020467388814112594686690","series":"1.2.826.0.1.3680043.8.498.22736720099797735751171045228181921759","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25663827732039068221184137540713755716","series":"1.2.826.0.1.3680043.8.498.90038188064802260756032079497574662683","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645","series":"1.2.826.0.1.3680043.8.498.36118508777211385472312430441958848682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25758766525934930593672922246867704103","series":"1.2.826.0.1.3680043.8.498.69982942818557643410872672636635808827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25766673238950703352855821505938941878","series":"1.2.826.0.1.3680043.8.498.10881662665068917651675360868198721819","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.25815857653380672532983223088817283976","series":"1.2.826.0.1.3680043.8.498.10322984394582187372712528823840129544","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25899681172115698926926115441929702502","series":"1.2.826.0.1.3680043.8.498.10207906635112121055611322600285729062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25928779639851562853500186715690156268","series":"1.2.826.0.1.3680043.8.498.30260392520009111737906453509757841806","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227","series":"1.2.826.0.1.3680043.8.498.10683901231951245862082511437641140449","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26006372873438224908989402529530833528","series":"1.2.826.0.1.3680043.8.498.10358388369995808560456659400558435097","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26066524302907849854339129271283426208","series":"1.2.826.0.1.3680043.8.498.20220336975639321569905280919097143901","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26201626427712146573573952548179938304","series":"1.2.826.0.1.3680043.8.498.11584296861904498935125641822673812024","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091","series":"1.2.826.0.1.3680043.8.498.16068072473119635555541496146089267121","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26351577824204863242390892891061594387","series":"1.2.826.0.1.3680043.8.498.63344312950302782990499150882930988800","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26408081008482310696738068933718440200","series":"1.2.826.0.1.3680043.8.498.59195579324867609266232704145933812409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26485916396240515006042297316506505324","series":"1.2.826.0.1.3680043.8.498.88821352489871248367822164503050462074","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26536992666456596813876340849399448897","series":"1.2.826.0.1.3680043.8.498.12723581492299541090934368710946177132","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26651323353208729423825552073457972796","series":"1.2.826.0.1.3680043.8.498.61596157175882382486917008578338768443","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26725172078893286313351418435377708028","series":"1.2.826.0.1.3680043.8.498.66243263472565801690392415235697276809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26842258746390750314059176647398369242","series":"1.2.826.0.1.3680043.8.498.39917519099406468277592063172316002154","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26875920364303755911342338961252130512","series":"1.2.826.0.1.3680043.8.498.30208695441195548450271679356624765695","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26946980494141305258556994153410856428","series":"1.2.826.0.1.3680043.8.498.70798213524657789953178224242011639472","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27020884753639915245324093232218617967","series":"1.2.826.0.1.3680043.8.498.17863609084745560316673436499585100723","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27025989808002468335294585261291982661","series":"1.2.826.0.1.3680043.8.498.63060356447163260689527522763070958275","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27157074709305277871045901905120405881","series":"1.2.826.0.1.3680043.8.498.44462336081111744064926954011330884201","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27184343589256623289925252380526385858","series":"1.2.826.0.1.3680043.8.498.70188212361538589785223436171200278071","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27192678730060203972082511503480566620","series":"1.2.826.0.1.3680043.8.498.13239156871442516367090172880688779977","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27224327142578332905227939015479767285","series":"1.2.826.0.1.3680043.8.498.40929553596323036867666332656445327531","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27306039618857661522863167157208122018","series":"1.2.826.0.1.3680043.8.498.47697555772137913999473045868695505987","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211","series":"1.2.826.0.1.3680043.8.498.10099231781303174010006408777590646905","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27398034409980350921804398626171135707","series":"1.2.826.0.1.3680043.8.498.22793415227595929901932163851781592986","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924","series":"1.2.826.0.1.3680043.8.498.47805453452773251051087617080685747579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27460982183399364904497832933739191826","series":"1.2.826.0.1.3680043.8.498.65283862457973884606747711262385892706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063","series":"1.2.826.0.1.3680043.8.498.11367835684275642321335302251160883505","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27653320597564308144389724321115208375","series":"1.2.826.0.1.3680043.8.498.55956317947207026587272182453333447170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27671779587445983390840223475173360482","series":"1.2.826.0.1.3680043.8.498.56676015270531392509759570251894657374","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27710718386318170070605046744941324421","series":"1.2.826.0.1.3680043.8.498.23608464296858835751346871661030361719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351","series":"1.2.826.0.1.3680043.8.498.32450946952581303653221632920516227528","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27796604203156439069742212039368371002","series":"1.2.826.0.1.3680043.8.498.13072910794981884240098910082225675130","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27890860883826681731748268459485259935","series":"1.2.826.0.1.3680043.8.498.29932729139588455906162822149664808452","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27934397251575987048967084987994202973","series":"1.2.826.0.1.3680043.8.498.10777737233770176253941194223438814216","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27990965288663885936880638302158503894","series":"1.2.826.0.1.3680043.8.498.74736122799382952287115888781287779859","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28023795363827851445172311810918021563","series":"1.2.826.0.1.3680043.8.498.69472479603055682081456120300811349755","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28045606887691550130031344985830975325","series":"1.2.826.0.1.3680043.8.498.48080982933567067752071146881843478310","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28153286315438977482271934270979508349","series":"1.2.826.0.1.3680043.8.498.53114923690564883385808265974093069555","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28220626968612492259133318770141901567","series":"1.2.826.0.1.3680043.8.498.11278115490204474936344055493671216797","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28269350392124251309834584420564806882","series":"1.2.826.0.1.3680043.8.498.11804609068875260190563982387113933829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028","series":"1.2.826.0.1.3680043.8.498.92377831856468380672280305178126860238","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28351884945918062351436698411322252525","series":"1.2.826.0.1.3680043.8.498.45543911911522129610691310200679679345","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28481442608487320644829189127206806471","series":"1.2.826.0.1.3680043.8.498.39245469748590485449044864490555750271","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28490861391717560759667589681288427737","series":"1.2.826.0.1.3680043.8.498.12223529869875319673872045522058139100","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28595729988867300278425411189173775382","series":"1.2.826.0.1.3680043.8.498.12366625597863829110283852762547734542","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28628472249424790112957421215774036156","series":"1.2.826.0.1.3680043.8.498.96568345107544290465944673938523009150","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28638490997780173004029312240334483157","series":"1.2.826.0.1.3680043.8.498.97264523884261393571466097336765023863","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28670999234453413675502551916979760344","series":"1.2.826.0.1.3680043.8.498.38590257738908909065861204217070520432","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28746954364434312909368235768936980607","series":"1.2.826.0.1.3680043.8.498.11297688891779873243059316466409392635","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28779021651975021236045277212517629154","series":"1.2.826.0.1.3680043.8.498.72301646335633555659554148534487195336","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28888941572409703700315590339486528256","series":"1.2.826.0.1.3680043.8.498.15924239671596741238453326034887614140","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28944156754203202831026290463520445489","series":"1.2.826.0.1.3680043.8.498.81646426411648656372827312968447110897","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28954361146470748868500417103935640742","series":"1.2.826.0.1.3680043.8.498.11879865673699131556860652970244268473","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29055892482211392922525222851162308510","series":"1.2.826.0.1.3680043.8.498.92339134310169992597613891203115704821","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29280448517432444306374036429824425668","series":"1.2.826.0.1.3680043.8.498.45441829078144942888318832991550642996","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29343735048224321966824641432873034291","series":"1.2.826.0.1.3680043.8.498.60806251058135735596045504812272622020","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001","series":"1.2.826.0.1.3680043.8.498.28853278261544404817675802353463547199","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29375052467745327893128894062568944496","series":"1.2.826.0.1.3680043.8.498.11357685110038459588832422394512913882","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29443692810539372078922146461173545451","series":"1.2.826.0.1.3680043.8.498.31739513653921908632541814804364839357","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29509443439138257275071354031180789464","series":"1.2.826.0.1.3680043.8.498.88927943785682965446231241679175901900","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29735852958813434703969490187047665950","series":"1.2.826.0.1.3680043.8.498.10558909789342372313081941930313522121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29758331841591711520514486913006094054","series":"1.2.826.0.1.3680043.8.498.43011631140094154635984244419508656891","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29826129803278661744835136993610646069","series":"1.2.826.0.1.3680043.8.498.83445980914152741326379083530802730929","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29943089198208468441240036404506927883","series":"1.2.826.0.1.3680043.8.498.67410693668420098421471491978402779353","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30058585660043363937345050478072453854","series":"1.2.826.0.1.3680043.8.498.90936477132864508040681376245130641227","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30131339052241668380936148362472245259","series":"1.2.826.0.1.3680043.8.498.32880823291098629492156974351356581820","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30167179890902431464942556467698238073","series":"1.2.826.0.1.3680043.8.498.11072061022253998660364626952027792714","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30206987686375669521451035370290737934","series":"1.2.826.0.1.3680043.8.498.69154249726148014191398208833520033252","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911","series":"1.2.826.0.1.3680043.8.498.12908352013592575797593984722778976470","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30252501257451878373370333329266264057","series":"1.2.826.0.1.3680043.8.498.76286000578381230503436264574858789211","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30282025087399390929963075109487390046","series":"1.2.826.0.1.3680043.8.498.52823494043529682189038107782018201848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30420245540489522189154634720778032533","series":"1.2.826.0.1.3680043.8.498.69207531165245546531453447547652471037","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30435446905709054197096833427672604260","series":"1.2.826.0.1.3680043.8.498.39892734147632637847828913297440377200","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451","series":"1.2.826.0.1.3680043.8.498.27953400932665796215121103283059662924","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30557810085867943803354292590079995622","series":"1.2.826.0.1.3680043.8.498.20818516745343010089236838934182058662","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30662407196303880392459287553277761223","series":"1.2.826.0.1.3680043.8.498.89729442768468456054114970702387954524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30690232747840004203140392776057147177","series":"1.2.826.0.1.3680043.8.498.11105763509947400971637663109604739818","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30756534269149851281518502475458067520","series":"1.2.826.0.1.3680043.8.498.28098799133144585231074342846707679467","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30800677497901063887487934675082066316","series":"1.2.826.0.1.3680043.8.498.76205631020323430594776876120165024832","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30838586189496669675465778955893367497","series":"1.2.826.0.1.3680043.8.498.95817642989531218308253485002790176173","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30888379005655662168586864728687777425","series":"1.2.826.0.1.3680043.8.498.13490601221503212083052892377283439192","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30978547345367837000803077743620171961","series":"1.2.826.0.1.3680043.8.498.61795421272670030055356095917431865930","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31079821574648958955911441626193848961","series":"1.2.826.0.1.3680043.8.498.82126102532677755823490900413636356752","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31099878224304681441631176766255029271","series":"1.2.826.0.1.3680043.8.498.29930922206304089694975251519344308269","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31145453746697192222724674918300432873","series":"1.2.826.0.1.3680043.8.498.36529212585052191105691171332876785077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31175617350083221245857813741060042777","series":"1.2.826.0.1.3680043.8.498.12329576747322496669651474482352418228","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31218039240665208717816408042644491715","series":"1.2.826.0.1.3680043.8.498.27270336689033522859786929065744072810","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31280487369922340170695453174997912050","series":"1.2.826.0.1.3680043.8.498.40820378067178523083396027436771613242","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31284572943302383501947308874558551301","series":"1.2.826.0.1.3680043.8.498.79793527566785174170836526259218219126","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31421989235441217674752676866539112529","series":"1.2.826.0.1.3680043.8.498.12929781642568972073186688508180281357","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31555333355990829772911515362502000209","series":"1.2.826.0.1.3680043.8.498.33580879498226576347621036933955295126","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31610133883321894510649078615659708431","series":"1.2.826.0.1.3680043.8.498.11393767635726412667394204085828122048","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31790457319528233269194462675995569260","series":"1.2.826.0.1.3680043.8.498.13225639732473153456279619807728056978","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31806829168516955528191705338336428721","series":"1.2.826.0.1.3680043.8.498.11478208645872139676030623479988819669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31851627014639252866084081561126592250","series":"1.2.826.0.1.3680043.8.498.12668078403636881020295228222194579320","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31899121807187535405063131351086765296","series":"1.2.826.0.1.3680043.8.498.12636594412550970977492568528316983669","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778","series":"1.2.826.0.1.3680043.8.498.81828628697254427213831470696864960260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32138869097959653224534642965199855627","series":"1.2.826.0.1.3680043.8.498.60268194781784971344067857198083650860","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32179217289301203414651193280580594771","series":"1.2.826.0.1.3680043.8.498.43108483916565871974555385349800649951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32265622493980668732317541941196894795","series":"1.2.826.0.1.3680043.8.498.62527780329511441173933929856226618507","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945","series":"1.2.826.0.1.3680043.8.498.11606608801399746595082769699005627633","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461","series":"1.2.826.0.1.3680043.8.498.35504680756058444442194983681596030730","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32479930837336533083798155873656743007","series":"1.2.826.0.1.3680043.8.498.36955156910276284935269583602862118987","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32504785212530110880087745488871181111","series":"1.2.826.0.1.3680043.8.498.78337360074422840860981847809885242266","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32521102053110469817073126425955562934","series":"1.2.826.0.1.3680043.8.498.59406741486856451004336781753331199933","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32611545463217490575457831874695918229","series":"1.2.826.0.1.3680043.8.498.59940776786964708381763351044513475741","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32680606387445562892045224227371304764","series":"1.2.826.0.1.3680043.8.498.17757557232288032072385938945011849860","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32723296226443325838892542529446214677","series":"1.2.826.0.1.3680043.8.498.14747369012635818548397816412407756025","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32835080036019877751472792740274297358","series":"1.2.826.0.1.3680043.8.498.54310232210658516414529017781227913783","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32910372391370203605482772577821971133","series":"1.2.826.0.1.3680043.8.498.57909346675359730537882668252691021892","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33045687849853291066101186962073873042","series":"1.2.826.0.1.3680043.8.498.79986238111145792401502322583148553448","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33114390259375873203134285352951765534","series":"1.2.826.0.1.3680043.8.498.12967844695170122774519477630445185982","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33247377165700300890880607485836825807","series":"1.2.826.0.1.3680043.8.498.11256073331516773559636196051293532439","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510","series":"1.2.826.0.1.3680043.8.498.91746505878781150856808389543505527532","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443","series":"1.2.826.0.1.3680043.8.498.13387836131520411057370415743491815035","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33419086668421177906530808458004386826","series":"1.2.826.0.1.3680043.8.498.43111595564896163257712005068013527884","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33432461096351771607862098991677662209","series":"1.2.826.0.1.3680043.8.498.68966406443738206941896981663636637522","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33614319611050071354692274770073371314","series":"1.2.826.0.1.3680043.8.498.85204574051917475995622848572218986215","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33657688895130083119886800524324572941","series":"1.2.826.0.1.3680043.8.498.12370069882523579113298044669959275135","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33734536545775442043827840341408174347","series":"1.2.826.0.1.3680043.8.498.82450802407338014381480290827636695148","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33880013318702285028756482543453984598","series":"1.2.826.0.1.3680043.8.498.11974727396952284141976605546637082392","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830","series":"1.2.826.0.1.3680043.8.498.47079586257669333957913235534568933698","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33990091057243328830745904178686745167","series":"1.2.826.0.1.3680043.8.498.46642169875077300943714556768457196812","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34002185870196144582348922158474418674","series":"1.2.826.0.1.3680043.8.498.88250399164668502720981945609427504467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34076627216941578548083498328399240999","series":"1.2.826.0.1.3680043.8.498.10651165482836223029261548579608413117","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34109091435857467600499949506940613037","series":"1.2.826.0.1.3680043.8.498.91471938683224611334540859548975264650","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34262242337778924864334788374037992215","series":"1.2.826.0.1.3680043.8.498.50088943160291410205354195784673952871","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34360298211002490108642003575962873984","series":"1.2.826.0.1.3680043.8.498.14758142786851238879923274770553899928","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34404554752241524210538769325381023947","series":"1.2.826.0.1.3680043.8.498.10432337670053391914454588315957073058","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465","series":"1.2.826.0.1.3680043.8.498.10300349669508922546329856632372275862","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34546106114963066554367298449299214677","series":"1.2.826.0.1.3680043.8.498.79301989843969756072303423326355493616","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542","series":"1.2.826.0.1.3680043.8.498.24646993722416206798391255467240811420","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34654275504962399799861007974465776328","series":"1.2.826.0.1.3680043.8.498.34343404304026846052002086753787945631","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34717974828302316666446325243570202245","series":"1.2.826.0.1.3680043.8.498.81385186709935449979534220810783497140","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34753112960188810416088070814158959510","series":"1.2.826.0.1.3680043.8.498.44904180000439007464840165061744393218","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34795373994453585401035013232069587413","series":"1.2.826.0.1.3680043.8.498.55721916322046811874905970492313398688","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451","series":"1.2.826.0.1.3680043.8.498.10196170436371404993212763481864372311","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597","series":"1.2.826.0.1.3680043.8.498.13079139454873403083239070637129019769","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521","series":"1.2.826.0.1.3680043.8.498.85423861496971511398342912631293374949","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35090276406828326770336478753776261776","series":"1.2.826.0.1.3680043.8.498.39282305822490785241650964873281295974","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35127678462764233938878060108411012978","series":"1.2.826.0.1.3680043.8.498.10530513922595988130307784229545135871","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35230417208602644083092414161817929841","series":"1.2.826.0.1.3680043.8.498.21666215260507227983794650282470461750","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35330802038595377723999341992570259909","series":"1.2.826.0.1.3680043.8.498.15055768751389226788107502076953858961","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35526385429946467546636621603749416429","series":"1.2.826.0.1.3680043.8.498.65544136928829907584675266674952511878","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35549628666222168741574331855838621990","series":"1.2.826.0.1.3680043.8.498.10183169116260843226207305334567719143","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35639250779062198905356481136455419313","series":"1.2.826.0.1.3680043.8.498.11699546500044649782463948315300939017","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35677515667357567818845364400903668190","series":"1.2.826.0.1.3680043.8.498.10842823413593766374647496354280322448","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35777075794856195065665011443485613681","series":"1.2.826.0.1.3680043.8.498.63802031066696705323342780735625090114","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720","series":"1.2.826.0.1.3680043.8.498.12453746892583420475527008103932256335","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35836928928058091851993989259747528067","series":"1.2.826.0.1.3680043.8.498.13374379133848657327693109735078130187","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35871201339095431578743898508613853278","series":"1.2.826.0.1.3680043.8.498.51223233875126990771734573332564941094","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923146679164443588381195543533271939","series":"1.2.826.0.1.3680043.8.498.68709865044936842278528545563792934036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35923753382244056317223397206343376734","series":"1.2.826.0.1.3680043.8.498.12861766882487493186021415178535239036","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384","series":"1.2.826.0.1.3680043.8.498.62544509211062204323258606879964746829","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35971292331621735865925455612813197566","series":"1.2.826.0.1.3680043.8.498.14471545386167857541165917670983716406","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36035264348264378094569428634302238394","series":"1.2.826.0.1.3680043.8.498.15793515783636581786934469136006879873","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36090989845682961399757203739039806565","series":"1.2.826.0.1.3680043.8.498.10528024145341623403106667815977434538","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36180248571044206939894546783755557323","series":"1.2.826.0.1.3680043.8.498.43362897501797555306312777435492597690","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36193046155628980489031133466840036976","series":"1.2.826.0.1.3680043.8.498.25849166793413470068760056737386573917","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36293583160636884934622362743062825202","series":"1.2.826.0.1.3680043.8.498.41533025134097413352624613643118473714","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36315811661139684771871696847516662339","series":"1.2.826.0.1.3680043.8.498.72483638850507697226215786094865952252","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36470828078108352859868482351276699300","series":"1.2.826.0.1.3680043.8.498.38862754578828000758631556628721058621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36588608184578739092409073014150374185","series":"1.2.826.0.1.3680043.8.498.58310164070463833317778661350927242714","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36645945703275956149107297016813148264","series":"1.2.826.0.1.3680043.8.498.83005938824551845168987184905646103722","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36679002788952511154610176614545192374","series":"1.2.826.0.1.3680043.8.498.68396714913531458989398483526112723113","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36757476214794170118868510630678488185","series":"1.2.826.0.1.3680043.8.498.55549672144185995632224345824066384192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36846391753332367835804699576399553980","series":"1.2.826.0.1.3680043.8.498.11019395143770767395494278927864678256","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36914870739904705183135763739850515617","series":"1.2.826.0.1.3680043.8.498.57465159760300182196987001421145304436","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36930256628357053046574486650916442047","series":"1.2.826.0.1.3680043.8.498.46080586137859461895748997557243185594","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37007302166102452703965686598524467188","series":"1.2.826.0.1.3680043.8.498.10189054271756860384777585658472832260","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742","series":"1.2.826.0.1.3680043.8.498.23174969622873816054521935098776352184","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37110727485959679337431199930441321268","series":"1.2.826.0.1.3680043.8.498.12063788573257342115416209441821187343","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37210322466522537523056852160944888029","series":"1.2.826.0.1.3680043.8.498.66123716117068582363759515885894353125","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37332768072749723117839591872457536315","series":"1.2.826.0.1.3680043.8.498.19799308482776995449397576150864419137","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37371365833775442014007234936409739360","series":"1.2.826.0.1.3680043.8.498.10924162523772897182454097032929906690","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37413349933694832641902637847629969234","series":"1.2.826.0.1.3680043.8.498.69260474414443020898814697130811042178","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37432328401993499374675847888297416449","series":"1.2.826.0.1.3680043.8.498.12984146347152876135301008344098868625","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37463970179664003092832463732043124908","series":"1.2.826.0.1.3680043.8.498.11236931765980751843455287339231164528","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37508036912473928281276902883575142611","series":"1.2.826.0.1.3680043.8.498.13630441824248006716631690481790520844","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37558426555271635608942179071280432530","series":"1.2.826.0.1.3680043.8.498.11024091282818642876401903524153081663","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37580209294522316445963449455348371815","series":"1.2.826.0.1.3680043.8.498.86322539719942589339929790001078344653","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37591590536870291237852764995372956522","series":"1.2.826.0.1.3680043.8.498.25183362666949904780488688074693159953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37695832484726177391252504639155717084","series":"1.2.826.0.1.3680043.8.498.10544564453084309446872602662699564465","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37729566534337521443213426693965453255","series":"1.2.826.0.1.3680043.8.498.44915460737536039247206775697091309711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37838990693631869129590283596686992710","series":"1.2.826.0.1.3680043.8.498.50234886000479729077780226740855351958","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37862788793597360072184753016994526122","series":"1.2.826.0.1.3680043.8.498.98541108023240066208951525331910974684","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37921127120678939078868474340027123335","series":"1.2.826.0.1.3680043.8.498.40989213970605997221312954531190601747","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38005916106577944091708997377832932938","series":"1.2.826.0.1.3680043.8.498.14955396749228330984884692790253661683","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38013372055010421273425628837282557767","series":"1.2.826.0.1.3680043.8.498.42129127094102708911169847494475708036","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38069035744880456154810472748123499221","series":"1.2.826.0.1.3680043.8.498.37257449146749863634073393992932768285","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38104228771638294693272534500734878990","series":"1.2.826.0.1.3680043.8.498.85210164202537856428615503619508875504","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38224750236138877853888930317337541394","series":"1.2.826.0.1.3680043.8.498.25701078858494127101548295773305483156","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38272345519163551901865251730721530169","series":"1.2.826.0.1.3680043.8.498.66652553057694861121456977622893182821","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38303307720827621886420004896097661262","series":"1.2.826.0.1.3680043.8.498.12078866720450613335707816258980606593","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38325494489236992123219618004392575979","series":"1.2.826.0.1.3680043.8.498.12789087629724310543219429675294822019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38365187857616428023213060377488046375","series":"1.2.826.0.1.3680043.8.498.64921426710814803463362782330929759543","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38420384692181444061893234153745102059","series":"1.2.826.0.1.3680043.8.498.11368348391177117650012165649988474928","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38457315909459574570206119213798163082","series":"1.2.826.0.1.3680043.8.498.35713883663865346888513050975565454610","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38528511947818361078957998546327084024","series":"1.2.826.0.1.3680043.8.498.15951253624585825085806724423208454107","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38567342184301320850258918666303107813","series":"1.2.826.0.1.3680043.8.498.13649732170782604472855582706690291741","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38593351637971248239815933638950449587","series":"1.2.826.0.1.3680043.8.498.44891614329724819494802973184842874959","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38628165743951175602803204505216754926","series":"1.2.826.0.1.3680043.8.498.97362041783586735271858417561391007333","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38702677049080452580748506476598606517","series":"1.2.826.0.1.3680043.8.498.34886235217688118458392277716800875034","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38750761816114659458563557493234655604","series":"1.2.826.0.1.3680043.8.498.93675543862357330699121890350351418184","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38796519786809225694393617946244012065","series":"1.2.826.0.1.3680043.8.498.14445322467036296624553571831268693808","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38862475907405766248096419059100131863","series":"1.2.826.0.1.3680043.8.498.18382917753119301311768535773159722918","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38898732985465285166803609252960353405","series":"1.2.826.0.1.3680043.8.498.13019574564676419783518800136058434014","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38925755872951351471027638563742132144","series":"1.2.826.0.1.3680043.8.498.11170655115598113880685789840203521360","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38935694477580522005505074220300062204","series":"1.2.826.0.1.3680043.8.498.40588830627651560764792487721569276161","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38981318408220047366318316620365040799","series":"1.2.826.0.1.3680043.8.498.77112196631835587983185505979931897675","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161","series":"1.2.826.0.1.3680043.8.498.13383725820713325121266739450871062380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39068632185013624181565849012182917752","series":"1.2.826.0.1.3680043.8.498.36184314085650237954295468347716294373","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39190090804401525573012570218955054016","series":"1.2.826.0.1.3680043.8.498.88729266017452199456530445105928228221","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39250844287482885227700969308333354206","series":"1.2.826.0.1.3680043.8.498.39805504417797605400341881232781639533","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39324076659429494696721784912308233388","series":"1.2.826.0.1.3680043.8.498.11557991886730907726770066559332080491","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39394842976048577715164357752399606924","series":"1.2.826.0.1.3680043.8.498.52344861603651895024494505043624167000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500","series":"1.2.826.0.1.3680043.8.498.12798253935401276011357924521785184170","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39547752108509751520306190293259313911","series":"1.2.826.0.1.3680043.8.498.32252716900196974338822425615540658304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39636073039676814228254715705296535819","series":"1.2.826.0.1.3680043.8.498.12923263220349119444498234666642237521","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39787654910948635229369478006027304578","series":"1.2.826.0.1.3680043.8.498.13060486927797934462186166704304772670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39845300426582562781355154625767485381","series":"1.2.826.0.1.3680043.8.498.42631282401009705639487954412419919981","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39932424305167611414527482641898907759","series":"1.2.826.0.1.3680043.8.498.10875869749392606889535032192570371511","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39959632509842689058626880477779808473","series":"1.2.826.0.1.3680043.8.498.95659366826017961867163037886942268709","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40036526071307024601256787605725348112","series":"1.2.826.0.1.3680043.8.498.12311329262116919401988858008972147460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40131899651518057045444739705301716085","series":"1.2.826.0.1.3680043.8.498.11345687577478619789862988241262534038","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754","series":"1.2.826.0.1.3680043.8.498.12345761890620862034074597493689457332","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40254123984175499564145728084464192446","series":"1.2.826.0.1.3680043.8.498.16630648036487765071129101314165034223","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40269807401373034066229810558300266830","series":"1.2.826.0.1.3680043.8.498.12110622146919549853465481021099392729","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40372643537313871452054321931253684619","series":"1.2.826.0.1.3680043.8.498.79491338125913089653020229358559326153","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40419759559020255481034237210540871312","series":"1.2.826.0.1.3680043.8.498.80645892546025846637900603572140073364","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40437305053062892226978469238653825745","series":"1.2.826.0.1.3680043.8.498.59826455713762123903855800722525447816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40533893516114346078820881178450524201","series":"1.2.826.0.1.3680043.8.498.96882135745335839411160138883711128565","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315","series":"1.2.826.0.1.3680043.8.498.32584863954291879717688439152891265392","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40618283771317160262473004328692613267","series":"1.2.826.0.1.3680043.8.498.29853353466224813741096425724102531663","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872","series":"1.2.826.0.1.3680043.8.498.10496756228573153047517555331918633447","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40718200801895968234499223301900799376","series":"1.2.826.0.1.3680043.8.498.14669298449915036568403145950052222419","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40823842819404259265489417470859248075","series":"1.2.826.0.1.3680043.8.498.93835639780797533313733147874100476512","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40837296112545776586904280088902000849","series":"1.2.826.0.1.3680043.8.498.53941661154071494000755299226727514439","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40910368031948275476880475208016058754","series":"1.2.826.0.1.3680043.8.498.48014958400238537894997795114650658516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40987687016832998820664054439678138878","series":"1.2.826.0.1.3680043.8.498.48640845592156427201873445328344501491","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41057428195711190931490429649014776496","series":"1.2.826.0.1.3680043.8.498.43756619358284730057041449652960483956","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41124244151692842048993594448992046260","series":"1.2.826.0.1.3680043.8.498.51285596590545296756576482970445262342","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41240503248178304281405515398211762506","series":"1.2.826.0.1.3680043.8.498.38985502660966246070003947550409252642","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41315359092354940082611381451934245995","series":"1.2.826.0.1.3680043.8.498.85956614208816995361051103172141544578","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41366889271483886167284378157366992074","series":"1.2.826.0.1.3680043.8.498.75456279387074563148446711705667192985","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41439889720274233955076963938741061358","series":"1.2.826.0.1.3680043.8.498.22634464298408837640098568819053290080","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41454762422897656673710145732036115764","series":"1.2.826.0.1.3680043.8.498.62869956929700621394915734143384384286","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125","series":"1.2.826.0.1.3680043.8.498.13126694251861399147854858979559961819","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41619164749467663852546208143469380415","series":"1.2.826.0.1.3680043.8.498.43116678140767728941764254626899710303","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414","series":"1.2.826.0.1.3680043.8.498.12891757250639577727532479704532460705","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545","series":"1.2.826.0.1.3680043.8.498.34571140402609869401976242046269907560","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778","series":"1.2.826.0.1.3680043.8.498.99497688854122153258840214387845412380","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41751049067043481681879071436091069601","series":"1.2.826.0.1.3680043.8.498.53853666157942525831542309619219610245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41810826648175757249770990269365542940","series":"1.2.826.0.1.3680043.8.498.16341780686373540984302522054027282417","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004","series":"1.2.826.0.1.3680043.8.498.69457237268630238679796427348335488524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41939824564209194493855981805593285528","series":"1.2.826.0.1.3680043.8.498.13363801066221107870596619823552106691","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42018683052544325209168649309206539721","series":"1.2.826.0.1.3680043.8.498.33895308215064951641250462250088739037","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42051385452979123325023608442860201782","series":"1.2.826.0.1.3680043.8.498.27119964123538975148626802340314901843","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42120645248710860870761681999665860884","series":"1.2.826.0.1.3680043.8.498.11887492301127134204516552143749622804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42221373360893091454504422964928099406","series":"1.2.826.0.1.3680043.8.498.58717293991579553715954610026916884789","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42287188181305579531212818123200473947","series":"1.2.826.0.1.3680043.8.498.11259283258445119560185790014322009573","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42396216793489016293032290339618185275","series":"1.2.826.0.1.3680043.8.498.87913976315251187607175617140718120170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42486421970612099441197418307958561877","series":"1.2.826.0.1.3680043.8.498.14122335040506476908316395857134724970","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42566296562794128096734153388840524012","series":"1.2.826.0.1.3680043.8.498.33253732268256999049227752414638087122","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42598518026540237824479232797503218019","series":"1.2.826.0.1.3680043.8.498.78983759837901218511302412441549228293","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42634709955067416383713194799683605006","series":"1.2.826.0.1.3680043.8.498.83820996157790773592812265427659253488","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42776652100045349404152232788825670056","series":"1.2.826.0.1.3680043.8.498.46977489848782875455716849966898272148","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42816339547796375466199938922141987775","series":"1.2.826.0.1.3680043.8.498.13270480137985993445656469333466426767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42847376403722341195083350288786052637","series":"1.2.826.0.1.3680043.8.498.12396651034358109464405978740812955266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42921593713351589642747633505679170597","series":"1.2.826.0.1.3680043.8.498.25988469330741034277979294801787598843","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42965031665568825650795822006244115609","series":"1.2.826.0.1.3680043.8.498.11909645000614859072661152742339904176","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43014288535685961874615794778401152033","series":"1.2.826.0.1.3680043.8.498.72759375453505005520988777727982009974","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43117007553940146763245418710268028741","series":"1.2.826.0.1.3680043.8.498.86029161362801904208833240654018478599","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43144544376687968624598661376472625623","series":"1.2.826.0.1.3680043.8.498.63446884159424179822170787777246741798","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43256905750043156063416845850825787417","series":"1.2.826.0.1.3680043.8.498.10014196852555980013194055310655549742","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43339007024446439709455570527734115557","series":"1.2.826.0.1.3680043.8.498.39759317541347386530826245758230629808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43412154740072739455825163326814980838","series":"1.2.826.0.1.3680043.8.498.12547006960325349740419580319262011223","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43439413368880313042951208701983487343","series":"1.2.826.0.1.3680043.8.498.11117415448122949035353236115266631641","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43483537125776347707929354413013709736","series":"1.2.826.0.1.3680043.8.498.37371751046368566310673922146554007084","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43692339773096477132070133629776394739","series":"1.2.826.0.1.3680043.8.498.22883504403383285649159257007593452981","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822","series":"1.2.826.0.1.3680043.8.498.28591651952993216156787162544138862044","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43870319732657730742794291443940225011","series":"1.2.826.0.1.3680043.8.498.91771999109348120650340699379717964455","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052","series":"1.2.826.0.1.3680043.8.498.11566153120770489610090939731397160923","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44239214982361627984688924889728201208","series":"1.2.826.0.1.3680043.8.498.78339432259935251969801430601275370391","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44321626715626508427287815810252844889","series":"1.2.826.0.1.3680043.8.498.16444449646414934863978637619766460887","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44350638600163778456163342454087669278","series":"1.2.826.0.1.3680043.8.498.49759258349293976880512445588486204041","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44405554266141576380633014711246118209","series":"1.2.826.0.1.3680043.8.498.55876967020268226197606201663593211808","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44441813293962885868294317830142850713","series":"1.2.826.0.1.3680043.8.498.16821336296903766638698209130707055313","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44525218699266472141419416162979441275","series":"1.2.826.0.1.3680043.8.498.10567472361690560324552080030173514293","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44654558140536370040701171884970036040","series":"1.2.826.0.1.3680043.8.498.12687753537484617546383698961915203385","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852","series":"1.2.826.0.1.3680043.8.498.87214256513856283270411098337556243644","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44792591585917525576461101442761037507","series":"1.2.826.0.1.3680043.8.498.26808274135574678180910653402264416815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44879085997210091159117906674172231025","series":"1.2.826.0.1.3680043.8.498.60304083470734946481118281135172512890","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44902036121220687805715010480088379768","series":"1.2.826.0.1.3680043.8.498.66493143998270035520807656014841951285","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44996883917122941421301655137158093858","series":"1.2.826.0.1.3680043.8.498.71361798960448110450089458803758572159","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45072896942121803896760563154385918032","series":"1.2.826.0.1.3680043.8.498.11448668855829331408478452781899054462","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45118217412917282413100913670807141634","series":"1.2.826.0.1.3680043.8.498.22671011449335800095105347200192288812","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45251306032215692953561852960585282987","series":"1.2.826.0.1.3680043.8.498.75458951990137493512441713863583889316","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45286493874121093392171044034263851961","series":"1.2.826.0.1.3680043.8.498.25470227410465704813020697363007658982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45326662718293919909188368237039419949","series":"1.2.826.0.1.3680043.8.498.13053949868397496659717600947187087580","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641","series":"1.2.826.0.1.3680043.8.498.76240128521599955869691057570483889890","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45392604471538776666611947492530314159","series":"1.2.826.0.1.3680043.8.498.11777984056531664581007137588127795329","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45454539422542821586939369440135728144","series":"1.2.826.0.1.3680043.8.498.80397191343031032405306579704242017269","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775","series":"1.2.826.0.1.3680043.8.498.11054658209652840680024404180667300679","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45625191292968830335971647388746041117","series":"1.2.826.0.1.3680043.8.498.12487647594567484295984506911697976808","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45644703214235636656820126625301033696","series":"1.2.826.0.1.3680043.8.498.51997968163310990164225131086082306642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121","series":"1.2.826.0.1.3680043.8.498.75481644998610072376207050394615031838","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45865541491386404818767748310365919685","series":"1.2.826.0.1.3680043.8.498.66696333064598532860331052802123824172","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45894475516096841241467193775629362381","series":"1.2.826.0.1.3680043.8.498.31926658975355966302633718100746091971","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46016776602328665980436110567854612595","series":"1.2.826.0.1.3680043.8.498.40870897592883590339994479522847313601","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46051119885630602872050329412008545893","series":"1.2.826.0.1.3680043.8.498.12857563385674397274902034273096732424","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662","series":"1.2.826.0.1.3680043.8.498.49294336234828292951927172974547797202","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46228189366201925238527911905072482095","series":"1.2.826.0.1.3680043.8.498.17019512572495584938092661035775622718","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46282234455466954264170065462242039568","series":"1.2.826.0.1.3680043.8.498.64576628878160369624920900983881982738","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46305694398420300356121078124030126469","series":"1.2.826.0.1.3680043.8.498.56906716449460752960738152206291730429","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46407639440476738746130651250276012493","series":"1.2.826.0.1.3680043.8.498.13475299336634695101462093373491619176","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46491051379327985199309365945879841064","series":"1.2.826.0.1.3680043.8.498.93352893543426366117854950488913113689","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46658713832425741743740258795376270265","series":"1.2.826.0.1.3680043.8.498.62101545680432739314428055095579746495","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46713978512945139851373191422321462143","series":"1.2.826.0.1.3680043.8.498.21281159070928517089572538261240352316","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46770589129181839292542004762882468541","series":"1.2.826.0.1.3680043.8.498.11468777967764533398144538975576210099","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46825768767498441994570613185981335949","series":"1.2.826.0.1.3680043.8.498.12466056251177415701749102373853749178","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46884541791777826238103426426267571657","series":"1.2.826.0.1.3680043.8.498.49147497449602192288530551486564650327","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46925959240037679795735705370016021232","series":"1.2.826.0.1.3680043.8.498.74867119003035204416686734997734683597","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47036446751578280498105392189811189787","series":"1.2.826.0.1.3680043.8.498.48873585419484354886382146659640313595","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47063694709201049930726855357000925037","series":"1.2.826.0.1.3680043.8.498.11793673481147639184985152506029555781","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47130872195336642107576758658144949813","series":"1.2.826.0.1.3680043.8.498.10210927511402851828777772068971553093","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47255564202539432049700324486197024855","series":"1.2.826.0.1.3680043.8.498.60597306355920511601937251439165557109","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47336007784831556446570294227732688335","series":"1.2.826.0.1.3680043.8.498.12582575303115632022303502518651243988","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47391919345230380216864709248400573953","series":"1.2.826.0.1.3680043.8.498.80643072670279557972356597043046284723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47495385306095610608327984787779011133","series":"1.2.826.0.1.3680043.8.498.80804841900594557074631315867975806031","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47651715105501347255071941051553742767","series":"1.2.826.0.1.3680043.8.498.36383668345816465454241659253053716337","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166","series":"1.2.826.0.1.3680043.8.498.91610124920206553491766205850003290804","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.47753549323738763346112850705924843959","series":"1.2.826.0.1.3680043.8.498.10504766178060787675949313613599000064","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47864423097056282241181943425451597067","series":"1.2.826.0.1.3680043.8.498.15342539369131425377495490339839565294","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187","series":"1.2.826.0.1.3680043.8.498.69542398721543718229644391277314854342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48033467661219494489168041469309669510","series":"1.2.826.0.1.3680043.8.498.23980442247045483844523038949090061763","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48141366506505611287110229972010695331","series":"1.2.826.0.1.3680043.8.498.64318199027921486043872820147850944080","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274","series":"1.2.826.0.1.3680043.8.498.34236568294478626291652935651911222474","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48368929760320103375578806071106317953","series":"1.2.826.0.1.3680043.8.498.74180541198861428297338973872296978112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48486018503198264262324670490116120612","series":"1.2.826.0.1.3680043.8.498.10876006278741359006568268191228879978","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48521453765449688946535921546652765399","series":"1.2.826.0.1.3680043.8.498.72893922763862489754853559371973877142","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208","series":"1.2.826.0.1.3680043.8.498.43384149213026621796699365889038368768","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48771187298294322173131630438467163244","series":"1.2.826.0.1.3680043.8.498.11638243968480643907837400728834633193","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48784244479029704181755878324450207126","series":"1.2.826.0.1.3680043.8.498.73026215482666720127431919027342479596","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619","series":"1.2.826.0.1.3680043.8.498.97986298396372725430997220002960678022","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48918090379630186340047006216221526186","series":"1.2.826.0.1.3680043.8.498.56151097499757691328268797167085457747","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.48977699132767849812826572970218325717","series":"1.2.826.0.1.3680043.8.498.12192724253775595393378105655109319739","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547","series":"1.2.826.0.1.3680043.8.498.11208384045024513525093104356191312605","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49054454369199540406654153762248931799","series":"1.2.826.0.1.3680043.8.498.10121681746284817915936245658060699324","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727","series":"1.2.826.0.1.3680043.8.498.44506026119065372298528780888431586856","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49287143478474218875661109488367501329","series":"1.2.826.0.1.3680043.8.498.33888729961559942528508657825524311325","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49379034023506626029648394840207360925","series":"1.2.826.0.1.3680043.8.498.20786914732519424890585661663953150188","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49441835811239898312243678447796777305","series":"1.2.826.0.1.3680043.8.498.62057904362113150644539753685442148765","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49492260217998945587168161555623476238","series":"1.2.826.0.1.3680043.8.498.12556299438339216662615568422946967771","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49575149159425506211571562732929634662","series":"1.2.826.0.1.3680043.8.498.92514682874519647774070063511043680574","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49603554452147117475589797389254733654","series":"1.2.826.0.1.3680043.8.498.11328762447625421037316745236438486973","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49679916686449433962263937489400526112","series":"1.2.826.0.1.3680043.8.498.94769119672780517555148832282148987691","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49847104080351361589960954776757484668","series":"1.2.826.0.1.3680043.8.498.50669622214824664359453514488679250564","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49855677166670499939366963036804180912","series":"1.2.826.0.1.3680043.8.498.20992354231736883729506715244818991682","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49880369313850883713947031303955494125","series":"1.2.826.0.1.3680043.8.498.41337772693062653546694724364012169726","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913","series":"1.2.826.0.1.3680043.8.498.43701102984744197796704672973595059652","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50051845967696843074391018845332480653","series":"1.2.826.0.1.3680043.8.498.85136814120332049257891727813788869055","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50073674465735509400964384004695041387","series":"1.2.826.0.1.3680043.8.498.18414489651685149937770999543442942222","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50107479256735739327080544431929221135","series":"1.2.826.0.1.3680043.8.498.90785153339963803410222667720184686223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50182378358892377789975570268512056553","series":"1.2.826.0.1.3680043.8.498.85456836035973558388805647118183140177","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50220991646207533313828550949660577504","series":"1.2.826.0.1.3680043.8.498.10149247587633761086872289548737115388","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50261788831539215224639813005933285052","series":"1.2.826.0.1.3680043.8.498.11996255847111159474837039386905596554","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50318760104506317341991314680640521601","series":"1.2.826.0.1.3680043.8.498.10153600191566462877220069361526308934","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50424421287869498746932271082980000521","series":"1.2.826.0.1.3680043.8.498.39231415913828531178297355666032413080","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50487158921933419035809338652990508020","series":"1.2.826.0.1.3680043.8.498.11239479942609248802960100120258158050","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50512021577184367495598050974910055550","series":"1.2.826.0.1.3680043.8.498.12764418044671713767448567891914668723","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50648934951234229265309905009982297515","series":"1.2.826.0.1.3680043.8.498.15751456483322938148110549157732294803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50680622091377404463269575858175673158","series":"1.2.826.0.1.3680043.8.498.81114129017396666299233599940618950221","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50689256431939553935971505794681061679","series":"1.2.826.0.1.3680043.8.498.76425094784669578240922399940554216035","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310","series":"1.2.826.0.1.3680043.8.498.10669338349477380661036806768007910268","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50810596378229719761556926464290468652","series":"1.2.826.0.1.3680043.8.498.79962947819004209493261046599721627415","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50819987120895374623148160166392914169","series":"1.2.826.0.1.3680043.8.498.12970089278217998456447986381932293901","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50848799900564667487859253004538870622","series":"1.2.826.0.1.3680043.8.498.10868999964718772883988611241170571239","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50935247372419775856440294835587827419","series":"1.2.826.0.1.3680043.8.498.11017255009895446607765521933499445160","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50980137249034338585131194965993785201","series":"1.2.826.0.1.3680043.8.498.95985974056355646377437301809889949833","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51059575642903646008860869691477305022","series":"1.2.826.0.1.3680043.8.498.13259711020426137741426539306021792561","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51143994849141407980459861759302509053","series":"1.2.826.0.1.3680043.8.498.13013687791149465050047998632211985598","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51297777726540529860433097009654453742","series":"1.2.826.0.1.3680043.8.498.21131788351388098700597839736026705820","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51379389400506715529323532959275254794","series":"1.2.826.0.1.3680043.8.498.11511541193012281529007739592322566100","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51429653138043597611341459799295143598","series":"1.2.826.0.1.3680043.8.498.56647131023094566966482932728930717131","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51536234256866063100891626028999490545","series":"1.2.826.0.1.3680043.8.498.73933210472417463120768563504073117152","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51676679394093588751371807453793655464","series":"1.2.826.0.1.3680043.8.498.17087572568976680796961277907076352062","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51768733026033198941231586462443503827","series":"1.2.826.0.1.3680043.8.498.12604887644959564791985880319373632821","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51782185934115000642557689300687723099","series":"1.2.826.0.1.3680043.8.498.71627959369576451531798196997950456792","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643","series":"1.2.826.0.1.3680043.8.498.10137691260535600008030582217198753711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51837740845904075589194479205218060915","series":"1.2.826.0.1.3680043.8.498.82913213258717642101114012124880470698","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51886877282987244004912794726043055730","series":"1.2.826.0.1.3680043.8.498.41912173808202959218835772267090213297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51970198810758997428998886534742243215","series":"1.2.826.0.1.3680043.8.498.87918729980231637897787163933708419654","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52010165788889734743850946282043771445","series":"1.2.826.0.1.3680043.8.498.95955420337768318287163944497567357618","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52046170319252016846878355722404423329","series":"1.2.826.0.1.3680043.8.498.17455273985382969508600692250847328146","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52067134682915759489080529708019359378","series":"1.2.826.0.1.3680043.8.498.91354054643588300413169287279035483178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52132258638267172087240021957310219977","series":"1.2.826.0.1.3680043.8.498.11926463436132609111911120697105526937","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52140911874747534807033188343849537207","series":"1.2.826.0.1.3680043.8.498.32244735476552362769384599297677868261","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52347547859131274836026480407333415929","series":"1.2.826.0.1.3680043.8.498.12507548354960910285029812507312661064","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52409113683924904563676007800283978062","series":"1.2.826.0.1.3680043.8.498.31638302932487353168085657092236091949","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152","series":"1.2.826.0.1.3680043.8.498.61707949601744374481961306099515601661","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52499379042723926379822784411787551701","series":"1.2.826.0.1.3680043.8.498.22330293806887421284741265939677010821","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52558822670236188507569589448789513993","series":"1.2.826.0.1.3680043.8.498.10250205817854620519563411448379373897","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52639459702574533721011212286428487650","series":"1.2.826.0.1.3680043.8.498.20591323409340733748514210207732328342","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52844966504845792334740560021996858939","series":"1.2.826.0.1.3680043.8.498.95937479325903253869377121071909728535","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52846236545327527967419125989598675101","series":"1.2.826.0.1.3680043.8.498.89051172122271832128897566569996831827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52985748674486376309978868177895755694","series":"1.2.826.0.1.3680043.8.498.74011694921843983939991829290638049687","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53076472372145933727990308381815269368","series":"1.2.826.0.1.3680043.8.498.55913245911721058156404982584816656439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53094450151502907325041400000717725711","series":"1.2.826.0.1.3680043.8.498.70013890042734917766650817267606472058","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832","series":"1.2.826.0.1.3680043.8.498.10737262430562018030908570082471902190","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53173579822475854424126184867040523127","series":"1.2.826.0.1.3680043.8.498.74768183102869824001103930349110813903","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53221818460567216467597044606040911702","series":"1.2.826.0.1.3680043.8.498.92485476378911074411201590712415921554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53331626867618182966740022261110760133","series":"1.2.826.0.1.3680043.8.498.98424760642661901160887003459105226000","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53343379040681745197301575417064236501","series":"1.2.826.0.1.3680043.8.498.59069276104539541189284932869046710851","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53370670740736715669115043403494610200","series":"1.2.826.0.1.3680043.8.498.12984265038781937699412353021617713508","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53408696789721413873026963095773050643","series":"1.2.826.0.1.3680043.8.498.11211470412542842776378466183692669102","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53579416035647672067965892020505016454","series":"1.2.826.0.1.3680043.8.498.20451914976273663468019137333595822122","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53628897373416863107623485630264501760","series":"1.2.826.0.1.3680043.8.498.56215773020478003398908092045065909924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53745429232602455103594227782517963027","series":"1.2.826.0.1.3680043.8.498.65422926114492082671173500168043959924","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53884715696186989798696544661396487883","series":"1.2.826.0.1.3680043.8.498.38535251036570864297928108288000238999","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53901077733518036755308692304268815965","series":"1.2.826.0.1.3680043.8.498.44511076693656880530689549350669488171","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54000782656400326230944526201670909133","series":"1.2.826.0.1.3680043.8.498.11090028061046698824064106515185250951","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54049224487950229496345484554581532711","series":"1.2.826.0.1.3680043.8.498.23675788620906555960395439801935368105","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54101830029742852876183926998159899424","series":"1.2.826.0.1.3680043.8.498.91530842613975828157925022749660370419","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54142170846176833568961426171739782214","series":"1.2.826.0.1.3680043.8.498.90787326830458240872837876964879600762","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54202117421897784469771192940597638508","series":"1.2.826.0.1.3680043.8.498.63071876510702574015971868132394712752","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383","series":"1.2.826.0.1.3680043.8.498.10358487205589856462972806996820662631","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54336962625483046366337626878609850219","series":"1.2.826.0.1.3680043.8.498.97561320293451482258956124600427371742","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54394066317295451515247748115888976860","series":"1.2.826.0.1.3680043.8.498.90460618934739910206089272981524548487","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54451515695681765924346479727204451807","series":"1.2.826.0.1.3680043.8.498.27910555316861992704141538708945211924","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54483132338261866032495309977418409299","series":"1.2.826.0.1.3680043.8.498.11062384518502533539225806211330045508","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54619696938940618755334224550745735815","series":"1.2.826.0.1.3680043.8.498.76024561727275651427960488090628494568","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54662745951809363774369202018332309887","series":"1.2.826.0.1.3680043.8.498.10479250996018507035661694922366154044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54693572065777563403567480644050627025","series":"1.2.826.0.1.3680043.8.498.20980790119730453601886401791128158588","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54711709751232783991855540573189433446","series":"1.2.826.0.1.3680043.8.498.12363892816962229043867070649283981364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54792838438079861454532183998999249213","series":"1.2.826.0.1.3680043.8.498.78465813830107557645814965844975417325","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54866613282412053616997257524944654864","series":"1.2.826.0.1.3680043.8.498.56348451488149421872579793160641955812","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55015061060346597564205577461227720715","series":"1.2.826.0.1.3680043.8.498.93703325713139288387018505394648459210","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55061039227197546099231336400032878562","series":"1.2.826.0.1.3680043.8.498.27880980106461599712562949123367235748","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441","series":"1.2.826.0.1.3680043.8.498.38243972019088989776300585908437822203","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55202459816190607876303419067810637245","series":"1.2.826.0.1.3680043.8.498.98076695332199576629949971998323339809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55344075316305815718253234447285669886","series":"1.2.826.0.1.3680043.8.498.57841412483270529555976758795690557263","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55346033326903331672579723063380636237","series":"1.2.826.0.1.3680043.8.498.81226349878641456467506609393529462840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55461938012492393262172182405932212246","series":"1.2.826.0.1.3680043.8.498.41854872679426100380968440342191688570","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870","series":"1.2.826.0.1.3680043.8.498.35869994756981268206144609366370621803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55700532609077428304194916132759352518","series":"1.2.826.0.1.3680043.8.498.10848447519247750600528154638259744348","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55729193057949125029301812320609764642","series":"1.2.826.0.1.3680043.8.498.85453555092979684552387794869746583395","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55808772152667135571939417929724526296","series":"1.2.826.0.1.3680043.8.498.71748165981628530235347128666590769115","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55856308312040330141050707531982127791","series":"1.2.826.0.1.3680043.8.498.11811768020346102710950700874760220468","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55927741980493661714408715886641034285","series":"1.2.826.0.1.3680043.8.498.80697993227069291188536673258695786137","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55971241672517044174019654998425819738","series":"1.2.826.0.1.3680043.8.498.75290074460603574734911120114881126529","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56016606949603829995818303105991224499","series":"1.2.826.0.1.3680043.8.498.46505549367176319372077161215120072971","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56054334296747411394367010227992359191","series":"1.2.826.0.1.3680043.8.498.90882956097461595909996473169055546253","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56146582420561901778012523946835033628","series":"1.2.826.0.1.3680043.8.498.57589092154330007976848736806365379217","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535","series":"1.2.826.0.1.3680043.8.498.45990745567996907975648917308308750739","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56216873861595017403772627709751840588","series":"1.2.826.0.1.3680043.8.498.13686533335945342736950568283496974266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56305445410093481072159375249067050124","series":"1.2.826.0.1.3680043.8.498.89109588047233916689468694504286160845","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56474364950531483257947155855279443851","series":"1.2.826.0.1.3680043.8.498.83277463415022984668436973455932762674","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363","series":"1.2.826.0.1.3680043.8.498.11783414490291628033336129717491807486","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56564801707939049324472369774251260607","series":"1.2.826.0.1.3680043.8.498.93288108290187200949144877836611405866","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56628956454784173189908113425348977923","series":"1.2.826.0.1.3680043.8.498.52101065048674183881218589539254003684","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56751453017492477106384704540803963879","series":"1.2.826.0.1.3680043.8.498.10703333239679204984910137095550721580","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56796622077427019119540733986070534126","series":"1.2.826.0.1.3680043.8.498.58193654139029865617047539177537479253","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56814987912934301686833132930648230999","series":"1.2.826.0.1.3680043.8.498.10875353392371829185235732686376857059","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56880070152758706555016296832842012883","series":"1.2.826.0.1.3680043.8.498.12893559518309477836913128591067991076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56894990686683651282564584582335346350","series":"1.2.826.0.1.3680043.8.498.81066742285803860864566298338313983096","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56939860329243258146102948466697018543","series":"1.2.826.0.1.3680043.8.498.65805378423110839462201874252076016931","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56991590802231436015843619633497607680","series":"1.2.826.0.1.3680043.8.498.19456299098007195365253310275018964586","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56995522547527264862064392266001620445","series":"1.2.826.0.1.3680043.8.498.10225645168234371900481848884381842564","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57084445429567494093960261980941917499","series":"1.2.826.0.1.3680043.8.498.69592057434038540235405567190221064586","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57160448024145680660335079050450776852","series":"1.2.826.0.1.3680043.8.498.10790783356264073999992185501529367279","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57342523828166309133982222435655662667","series":"1.2.826.0.1.3680043.8.498.92442415199776568679647824612718995865","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57406640473235989459885313797729459197","series":"1.2.826.0.1.3680043.8.498.21049909862837616959238608286194469247","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57615268093937994500995086275007694564","series":"1.2.826.0.1.3680043.8.498.10878885708611585548964097781881454297","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57768539239475064336346600805236336044","series":"1.2.826.0.1.3680043.8.498.36886683606033238879986894325223063107","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57789603189240866041812385320399435864","series":"1.2.826.0.1.3680043.8.498.21334161432591836671614912014209776355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57829173797358025677240411871600928097","series":"1.2.826.0.1.3680043.8.498.11770568837352199161100194162081145078","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57859543314667538612269642333387175763","series":"1.2.826.0.1.3680043.8.498.10983900763683533946781540750166553540","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826","series":"1.2.826.0.1.3680043.8.498.11604033470282092525291733783263294344","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348","series":"1.2.826.0.1.3680043.8.498.21469755994489671691413336530907798532","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58167256197067081436345136244032069163","series":"1.2.826.0.1.3680043.8.498.29678469145333586974206848650539517812","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58261214953729037661916565473101636690","series":"1.2.826.0.1.3680043.8.498.46690649377474529723346381461716660495","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58298767414933858487578597918936525600","series":"1.2.826.0.1.3680043.8.498.16671642082299097704886084212638551730","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58381033890285289577066078050126251787","series":"1.2.826.0.1.3680043.8.498.86970522719254428812587886264541584946","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58416798660375193103814365795347286292","series":"1.2.826.0.1.3680043.8.498.98252400516776849568057636407842913409","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58482152742263188503728990365891896621","series":"1.2.826.0.1.3680043.8.498.78753042808860765623659810259172766352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58550899000434548218916895907938175917","series":"1.2.826.0.1.3680043.8.498.12994914688844729211430244247434831192","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58604600600114899516493668210902251434","series":"1.2.826.0.1.3680043.8.498.12101423418710654720531605264481497635","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58654915518882480245974544469656013058","series":"1.2.826.0.1.3680043.8.498.31921639731451876498697805113235510009","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58710813820037448352310278234947865485","series":"1.2.826.0.1.3680043.8.498.37058949112761323714316869501787233473","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58749835013162544157291112101941849646","series":"1.2.826.0.1.3680043.8.498.74244995711274006927217259629373217399","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58776844157027040145262759105410792687","series":"1.2.826.0.1.3680043.8.498.34901604524923133706299761263502757058","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58835863392949696556026907211309159182","series":"1.2.826.0.1.3680043.8.498.10802953193391984612776898939511703023","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58888406752590566453001353089284177456","series":"1.2.826.0.1.3680043.8.498.11295552421536799552279063312942525032","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59021913092219789232890327780064145948","series":"1.2.826.0.1.3680043.8.498.12974935994367556291638738855911448721","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59048646289883992375610835845636263714","series":"1.2.826.0.1.3680043.8.498.10239265789075702016009995193138357704","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59098825208018364061073394433706021769","series":"1.2.826.0.1.3680043.8.498.11454826442101732768426337547218391640","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59254379036462832374262221063106167294","series":"1.2.826.0.1.3680043.8.498.56464413949260850983382510214354606381","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59394856717216352192037635065925607933","series":"1.2.826.0.1.3680043.8.498.11902593582528344260819366266468516922","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59426134288436219215525445839849588348","series":"1.2.826.0.1.3680043.8.498.78880502354917094164032466888229798151","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59476657510212258499047356518058209126","series":"1.2.826.0.1.3680043.8.498.25697574398087619828361705067541697512","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59511171909586924989766685687032462369","series":"1.2.826.0.1.3680043.8.498.59233087790833069907713169162550986749","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59646951619367295963072805933618947027","series":"1.2.826.0.1.3680043.8.498.10559878756389696969010841280314457586","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59708410902079108538978687294461753235","series":"1.2.826.0.1.3680043.8.498.10539454798113138310080245579632872378","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59828592894392286103154555572376381359","series":"1.2.826.0.1.3680043.8.498.30322065342121222086748552054154089882","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59864697179957009967756900038846073439","series":"1.2.826.0.1.3680043.8.498.11694048493199836318512873941399815316","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504","series":"1.2.826.0.1.3680043.8.498.64485586160736929498373406137918588666","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59948559729295284519115137429127740378","series":"1.2.826.0.1.3680043.8.498.12055486282515384986879282676079601835","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59973351395253648445741322652773810641","series":"1.2.826.0.1.3680043.8.498.83024134903276693672071995409814981176","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60088605286413557861235417321985538357","series":"1.2.826.0.1.3680043.8.498.58242982644016380335454541196960213906","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60189166269847256202786309096136242746","series":"1.2.826.0.1.3680043.8.498.39355742963723974124780533746411401638","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60240929196665639608259748424475337443","series":"1.2.826.0.1.3680043.8.498.82484366274087580439937798620947043311","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60347768295306794807599672836772410567","series":"1.2.826.0.1.3680043.8.498.24479281067607611891406869696153104304","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.60435128564448668334758790827765960420","series":"1.2.826.0.1.3680043.8.498.85480287103566344107737727868956461766","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60446085094710623303973373214794179184","series":"1.2.826.0.1.3680043.8.498.18579242503683294541152406438995893380","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60473167624534682552272469123806503959","series":"1.2.826.0.1.3680043.8.498.39343940764939597791653233042185906809","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60548477793215320533051194155539307724","series":"1.2.826.0.1.3680043.8.498.73388059751616113170966697403798878100","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60611146141493700985009985723275645474","series":"1.2.826.0.1.3680043.8.498.72096957722259958725647069953705701219","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60633008040453481523409852849000823589","series":"1.2.826.0.1.3680043.8.498.83902973031821689027383758344235084570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60736996513214411991331917490659995815","series":"1.2.826.0.1.3680043.8.498.52442411399132208442120338297499771494","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60924385750386872880015870524131415592","series":"1.2.826.0.1.3680043.8.498.11779996984418741531394915038014816826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61024356179284390979230045743542661223","series":"1.2.826.0.1.3680043.8.498.53818043905665003566334000409246081017","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61072566792285434404885416160728089641","series":"1.2.826.0.1.3680043.8.498.41469948900277769534785401993638379811","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61148578767041261085865406067025346106","series":"1.2.826.0.1.3680043.8.498.29941615005230393154937263850750503979","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61227644940421894951950605627016308160","series":"1.2.826.0.1.3680043.8.498.83648230259028003660273293853761641840","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770","series":"1.2.826.0.1.3680043.8.498.86953556057249889928982957932390501252","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61354742313056287432707914716598361679","series":"1.2.826.0.1.3680043.8.498.39252064154322949406249184757752168245","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61490475141024845836646569993549106628","series":"1.2.826.0.1.3680043.8.498.13031748511073858202046122755580222892","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.61529451559250327125552112075119161933","series":"1.2.826.0.1.3680043.8.498.75329116110014481061788147108887089325","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61570904948128887877542043572765432417","series":"1.2.826.0.1.3680043.8.498.87658211807336265813277702683064368474","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61715630583919056026446344483871058799","series":"1.2.826.0.1.3680043.8.498.11066918476209303864054303905975134761","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61812343111344551704536863711676179812","series":"1.2.826.0.1.3680043.8.498.62304142604122219581566446532384082486","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61934170443339188744573234190617483017","series":"1.2.826.0.1.3680043.8.498.38252986998676734908976144619794684082","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61981891095235024266625598999854298547","series":"1.2.826.0.1.3680043.8.498.12501969674232689316530859136871639026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950","series":"1.2.826.0.1.3680043.8.498.53104799425290013222351492620545692884","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62192347880785177732202565414591408086","series":"1.2.826.0.1.3680043.8.498.92975683413839197464011389026085693036","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62218684089117864632544493263539598403","series":"1.2.826.0.1.3680043.8.498.38158920385194241797101812614913501247","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62252176347004905787746821113523889523","series":"1.2.826.0.1.3680043.8.498.49701489505313564886493739152176818015","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62400371770962739002393364782172386752","series":"1.2.826.0.1.3680043.8.498.26547241253109233412826382160677190906","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62406588098851994968957074186291720431","series":"1.2.826.0.1.3680043.8.498.60213942949826616025874340178256499349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395","series":"1.2.826.0.1.3680043.8.498.10490923445995430496946166111818528271","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62662430565901539554882496013173225758","series":"1.2.826.0.1.3680043.8.498.80643657044524771040311457974362649351","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62760659205261492447481830774781950875","series":"1.2.826.0.1.3680043.8.498.99945943091704171874008854565822740032","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62782163204193388751869649423542931657","series":"1.2.826.0.1.3680043.8.498.46537388809053163848321198186663856191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103","series":"1.2.826.0.1.3680043.8.498.44713988847282651207104219569602400747","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62948920639798990757395955917890550512","series":"1.2.826.0.1.3680043.8.498.10290619310256642453385343296036943217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63005973438879239920248535292375481294","series":"1.2.826.0.1.3680043.8.498.39188709539468679627059997689682626438","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63069333313670617776227577192591993067","series":"1.2.826.0.1.3680043.8.498.10458808895697308503844112029725510210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63126604639589769235241216875934773992","series":"1.2.826.0.1.3680043.8.498.82597439503521624052566489646209541586","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63195088465290711580668024587233364997","series":"1.2.826.0.1.3680043.8.498.49571409934962580420248704176346652338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63216944317982327305447295456609309952","series":"1.2.826.0.1.3680043.8.498.48912841449966270213136507939507797961","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113","series":"1.2.826.0.1.3680043.8.498.11184429803036008477801610533885522697","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63280826430574305683941548157934410641","series":"1.2.826.0.1.3680043.8.498.38883176689099876812699448279310692748","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63360204391136938955220146755331421179","series":"1.2.826.0.1.3680043.8.498.75848784126406014921512871482316455876","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63426442500798189991842988966227275166","series":"1.2.826.0.1.3680043.8.498.60224196496701258110882101863675023242","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386","series":"1.2.826.0.1.3680043.8.498.50734813723127799327416713087907975177","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63521033783841675104013677430088400343","series":"1.2.826.0.1.3680043.8.498.10983865213128582255170896597469916258","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63550258873504696667112631869753870779","series":"1.2.826.0.1.3680043.8.498.51601502527769203307275259452647906358","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941","series":"1.2.826.0.1.3680043.8.498.78006226682510721660843103746960571753","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63584266247144369196126771996126173497","series":"1.2.826.0.1.3680043.8.498.50709740657818752906394548730705786927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63726512375511246975989322174096818969","series":"1.2.826.0.1.3680043.8.498.86670339428308635048705355882328110346","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63884838982826315257233262184190435906","series":"1.2.826.0.1.3680043.8.498.41553634803251561756686654761587954232","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63986652686300452077410299429386420400","series":"1.2.826.0.1.3680043.8.498.66667859664715657792629151649204522842","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64029768638431636860053484202080135316","series":"1.2.826.0.1.3680043.8.498.11264914717427199652533018356333331526","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64066465191710322298540005681534170036","series":"1.2.826.0.1.3680043.8.498.62096756886898344100266223186038520531","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453","series":"1.2.826.0.1.3680043.8.498.13384458520869252098551861492013033867","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64240637244650029205175793729578897734","series":"1.2.826.0.1.3680043.8.498.12077212779807759818276628243296388723","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64325734401816269429960464734504322970","series":"1.2.826.0.1.3680043.8.498.13311233227540384376514642332587569209","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64397747170052970974000650185716672477","series":"1.2.826.0.1.3680043.8.498.68919540392773289350667863799332365026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64421299492271548595446260681698412384","series":"1.2.826.0.1.3680043.8.498.11400778818526412163840145329651508637","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64487230511522940587342058274736571996","series":"1.2.826.0.1.3680043.8.498.96166048588616710058284477769637207310","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724","series":"1.2.826.0.1.3680043.8.498.54424266537008704287002554662014190629","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223","series":"1.2.826.0.1.3680043.8.498.10013280725198842098983071119588907872","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64673069370410118387363952155758553567","series":"1.2.826.0.1.3680043.8.498.39234703211177482911801448934003647470","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64865300853258390816475799272221318005","series":"1.2.826.0.1.3680043.8.498.18756009845317325151099022351020325007","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64917073507426524387269500871970032783","series":"1.2.826.0.1.3680043.8.498.71633769430706266279152741358388093384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65013994414976103275014874284064145461","series":"1.2.826.0.1.3680043.8.498.22804142400586679850115390099098627447","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65085954283273471512936799274064949557","series":"1.2.826.0.1.3680043.8.498.12617301568371449723865613657487176036","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65160241567146620406193593170843451582","series":"1.2.826.0.1.3680043.8.498.81412723454250240440510383785612701232","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65222333228514862688331581862087607325","series":"1.2.826.0.1.3680043.8.498.60675073661228438213809350973449985243","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65312506382127486486519184894274326380","series":"1.2.826.0.1.3680043.8.498.98322215530297679310059849179809212781","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65356758967883114612797250487949537460","series":"1.2.826.0.1.3680043.8.498.77699500741973987453398327037825123435","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65515828911930515002046151253057175178","series":"1.2.826.0.1.3680043.8.498.11577557357216070833528042560061666822","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65530757406333318449497070048719890442","series":"1.2.826.0.1.3680043.8.498.74581647698111747364870962351435366103","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65597375829884314152221851871550349856","series":"1.2.826.0.1.3680043.8.498.12189734336249358361508984312316618466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65667678378249093118816281743956646856","series":"1.2.826.0.1.3680043.8.498.57677348651595775508988578990248724929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65715827585629489888518528736848969678","series":"1.2.826.0.1.3680043.8.498.78261608213212978898999524772595376978","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65783500506568702195262661666345236666","series":"1.2.826.0.1.3680043.8.498.47419614188237070306665115519214534668","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65792625478083832901309715174678358900","series":"1.2.826.0.1.3680043.8.498.10547575010623704917023229233771447185","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65855247249608663104822603172047250057","series":"1.2.826.0.1.3680043.8.498.74753281754521206532927272688590818846","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65909865035783831607773206653044421345","series":"1.2.826.0.1.3680043.8.498.42321219830836287981244428345559999463","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65959182350599581687336889990076028593","series":"1.2.826.0.1.3680043.8.498.13228507435970096440464076629094416469","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66022464665791424345622894674375661089","series":"1.2.826.0.1.3680043.8.498.19123846409796304667371102384337987306","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66077420344931334696112616504179985591","series":"1.2.826.0.1.3680043.8.498.77186148012459415241028952145615513330","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66156015559325316834458423557757995415","series":"1.2.826.0.1.3680043.8.498.21773167454173796655181301314107244090","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66264392187277583747691643146920431469","series":"1.2.826.0.1.3680043.8.498.10465800055523273702368641832681812421","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755","series":"1.2.826.0.1.3680043.8.498.12897558405178751949074468831467705658","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66321903250295261063309947910189882952","series":"1.2.826.0.1.3680043.8.498.65363529609377442700466843299738882598","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66334686028272068369345540785320291695","series":"1.2.826.0.1.3680043.8.498.83296331816405063154958678858867197758","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66404725797392899950170138361835649917","series":"1.2.826.0.1.3680043.8.498.11450104052174533941354855947032132537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66510565421813721258189101326477318907","series":"1.2.826.0.1.3680043.8.498.75985414576459072681213562249698438603","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66690100241274246406145618641481787709","series":"1.2.826.0.1.3680043.8.498.91505656694005450369173971535941377588","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66737859593831436259089736126768464771","series":"1.2.826.0.1.3680043.8.498.22038110406001331771551616600544414113","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66865810098959732285731429066569782524","series":"1.2.826.0.1.3680043.8.498.11334441802293458369073157166522074007","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66973089869591928349547584554030589647","series":"1.2.826.0.1.3680043.8.498.23819599092070536531545666508388775243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269","series":"1.2.826.0.1.3680043.8.498.13328206335877928258613232152178638898","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67181088492292316990397177612918195713","series":"1.2.826.0.1.3680043.8.498.89766393711132974713619789333557349725","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67208608901061914528631080268819069203","series":"1.2.826.0.1.3680043.8.498.57299141469421866944785076583727979342","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67281816624401881515444587844116136597","series":"1.2.826.0.1.3680043.8.498.31168475846276210690002160150425968889","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67386780309737226410965627751074124678","series":"1.2.826.0.1.3680043.8.498.42828426573670756088395070488398151672","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67456402757136973484427155324664966595","series":"1.2.826.0.1.3680043.8.498.13371960883500796658381649620190187391","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67490059263503953166467514121525025332","series":"1.2.826.0.1.3680043.8.498.43530363951700649886529178875923422353","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67527486841399075490744039058207332725","series":"1.2.826.0.1.3680043.8.498.10772926725835587405920276049667707864","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67613388743126519966672593774754849330","series":"1.2.826.0.1.3680043.8.498.11566105822309613579015683338157176453","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67678576373934022409084801948795250361","series":"1.2.826.0.1.3680043.8.498.72570114200462886091232291283612901411","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67716063997837501476742746033172868748","series":"1.2.826.0.1.3680043.8.498.57799928866441489537450840157494230537","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67740000270419228186274186410889719154","series":"1.2.826.0.1.3680043.8.498.16101732660387120247609718249911798028","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67788047649994083591931245362169694306","series":"1.2.826.0.1.3680043.8.498.85132211185849876834601331848027105062","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67860057199678546706175336855949799228","series":"1.2.826.0.1.3680043.8.498.91523870132991027373659450681205922135","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67902658411525109462089381503197501728","series":"1.2.826.0.1.3680043.8.498.37338039697427066948963474568710655583","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67941678374744711267840655419606994796","series":"1.2.826.0.1.3680043.8.498.10575049217717912059321427485538978120","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67981157392064878495682078197161506255","series":"1.2.826.0.1.3680043.8.498.32727954268656833201637387239855314981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669","series":"1.2.826.0.1.3680043.8.498.95071697530905425576952881499834935707","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666","series":"1.2.826.0.1.3680043.8.498.52258335052499678116050605291067195012","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68131564130881160284492534203572971623","series":"1.2.826.0.1.3680043.8.498.12054792138620857474530849992289942198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68166103964518457268906879600985959436","series":"1.2.826.0.1.3680043.8.498.97127696300903741818912645701856335763","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317","series":"1.2.826.0.1.3680043.8.498.92422893862614215640391971216897804122","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68256141451764547710152696919981893952","series":"1.2.826.0.1.3680043.8.498.95926720814544888975740539543417124339","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68264559730756309944466784364591509088","series":"1.2.826.0.1.3680043.8.498.91915778461805350669957077648682665123","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68328948893840135511720724977892905019","series":"1.2.826.0.1.3680043.8.498.95995937078382886363155842914949835995","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68434323590918408241182252996136001891","series":"1.2.826.0.1.3680043.8.498.32329844717798544029526480486792857623","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68448811873610037553554213141537599150","series":"1.2.826.0.1.3680043.8.498.10888066608591233815586407556686265876","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756","series":"1.2.826.0.1.3680043.8.498.10386320144425632854985307489659574330","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68578659816221770558506780445608768600","series":"1.2.826.0.1.3680043.8.498.70111080994366357874067978848104331817","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68637419175382604890136839259344633105","series":"1.2.826.0.1.3680043.8.498.14878357833036719140561989112929624315","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68740466085414970518655922465908223436","series":"1.2.826.0.1.3680043.8.498.84211889204296844885158302103391350169","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68769474672202195228430674317444754835","series":"1.2.826.0.1.3680043.8.498.45359771479710709727301784738020420372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68806105227826942108149090508198910591","series":"1.2.826.0.1.3680043.8.498.52632081045664319053093879762563600513","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68919687595587004665658610968739852899","series":"1.2.826.0.1.3680043.8.498.12487049535759177511986975104046891158","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68971603226593154050514800137550171537","series":"1.2.826.0.1.3680043.8.498.18435662308844128826339546294004060374","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69033651423155214548081618535669641812","series":"1.2.826.0.1.3680043.8.498.13146435665213410430853628205422558754","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69141410772327587821159027820329185006","series":"1.2.826.0.1.3680043.8.498.16570682372565285821437851676906527254","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69212713525853754921729012449897924087","series":"1.2.826.0.1.3680043.8.498.11635625801501139768135073994413175721","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69267821440055811776827497076481587198","series":"1.2.826.0.1.3680043.8.498.10061845342080774594468251271422501411","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002","series":"1.2.826.0.1.3680043.8.498.79225807496183598139611165476901586908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69417319234704516520173921900954337376","series":"1.2.826.0.1.3680043.8.498.53016360277447804283427480008805327667","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69495014653333523580255444915082056825","series":"1.2.826.0.1.3680043.8.498.22451750528682884971882770143064875602","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69544085715595825083308172629360899339","series":"1.2.826.0.1.3680043.8.498.98616158961937147677194836650448187687","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69635429172929809539574727887787303282","series":"1.2.826.0.1.3680043.8.498.77200193966895000234836004261150417074","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69673283369917182409597968479300552579","series":"1.2.826.0.1.3680043.8.498.88492510928984242352596717808402621149","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69729903409900053570304330127687235813","series":"1.2.826.0.1.3680043.8.498.12529259750687501204315422143619660396","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69828573582708185393942443598783108800","series":"1.2.826.0.1.3680043.8.498.97339506127273573690981816388621933994","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69917156795941398965793962392683899162","series":"1.2.826.0.1.3680043.8.498.80702477113993294343358436578225501190","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063","series":"1.2.826.0.1.3680043.8.498.40553489248617812582177463076241702791","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70204818275951821519024007466988533363","series":"1.2.826.0.1.3680043.8.498.11551890234651767562262658910182020734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70291739279544255146458190540233023337","series":"1.2.826.0.1.3680043.8.498.18530557683685322039845178338576762776","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70310491987092072540988486213408328865","series":"1.2.826.0.1.3680043.8.498.10498386111379907876945694888689277694","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70409266114961050236894235887802126808","series":"1.2.826.0.1.3680043.8.498.11023591160051492621081843451896335839","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056","series":"1.2.826.0.1.3680043.8.498.52213171546384663954249527676445929499","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70478068758081824347762712452252763894","series":"1.2.826.0.1.3680043.8.498.65430565948415332501391510475984061540","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70532653867267627708152426358320452518","series":"1.2.826.0.1.3680043.8.498.75498057665933080259241367671378891496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70674155597869824035214737999044110827","series":"1.2.826.0.1.3680043.8.498.74090446420297950758034269918389001497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70705483212961922828901596159131697717","series":"1.2.826.0.1.3680043.8.498.11375955924383882332929317024166731976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70831735578941318362881379802693293952","series":"1.2.826.0.1.3680043.8.498.39873433873930676645145397879271356758","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70877157427595288430686609728186100169","series":"1.2.826.0.1.3680043.8.498.39230937878290404796890438882091736861","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70909154434106251910698870110315237266","series":"1.2.826.0.1.3680043.8.498.44326195279736066133723182551017431327","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70931245733460964524244528230239226104","series":"1.2.826.0.1.3680043.8.498.49871247270773184845232115619634014874","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70948162965271404678418299669321564765","series":"1.2.826.0.1.3680043.8.498.10880799885660040467959825052392583477","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70984418257970716669463829979819981861","series":"1.2.826.0.1.3680043.8.498.44850124404316188667815856741988747128","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71047916312801688293181464016099792199","series":"1.2.826.0.1.3680043.8.498.81795546758117752428518519187577358537","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71157888664602228768823145752952837565","series":"1.2.826.0.1.3680043.8.498.48670945610118098428419983477095366926","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71311031275189592818091380416268522846","series":"1.2.826.0.1.3680043.8.498.99384292224580070789810645435063036536","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71388532549899713881913113571627425347","series":"1.2.826.0.1.3680043.8.498.10903895544145404027819568310223008735","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71507059468810869004142324822642900237","series":"1.2.826.0.1.3680043.8.498.39503889383651867650308321344451550398","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71636001534145439773297498405674521629","series":"1.2.826.0.1.3680043.8.498.23646791071648175204657497355634542795","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71686717249431292334159336816742076478","series":"1.2.826.0.1.3680043.8.498.93864526721671218410765769061689394609","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71701484984624141610445774249629295610","series":"1.2.826.0.1.3680043.8.498.22678152248243433485178514267206050498","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71709477515434076105531625950744135656","series":"1.2.826.0.1.3680043.8.498.13038976962248509093750392551875288577","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71749507378205349451480318735471291366","series":"1.2.826.0.1.3680043.8.498.12877272061274706360871458731044621100","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190","series":"1.2.826.0.1.3680043.8.498.12801377882988124134009872504232278469","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71830371861099586573789550797712053914","series":"1.2.826.0.1.3680043.8.498.11044032401135442718762365489035576029","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525","series":"1.2.826.0.1.3680043.8.498.37737600018298891238839535128987966142","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71901039762584748047172479049347609017","series":"1.2.826.0.1.3680043.8.498.38259784879735096717287118189942056290","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72017096768214637927798648334301077145","series":"1.2.826.0.1.3680043.8.498.12059267689345502877866525086955649994","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72023797276837219152365428999317131110","series":"1.2.826.0.1.3680043.8.498.61444039411932138727543006887582774060","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597","series":"1.2.826.0.1.3680043.8.498.13675038893463358645676078380587490100","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72185504245484151161044972024469667666","series":"1.2.826.0.1.3680043.8.498.29898964938187128508839108021976744154","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72248728319398595146365363062799556637","series":"1.2.826.0.1.3680043.8.498.82579403349910528267758484488063949939","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72255181548595299889516400134053439401","series":"1.2.826.0.1.3680043.8.498.74529313647742425100980825549653181202","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72276631472244076773626607623783737687","series":"1.2.826.0.1.3680043.8.498.13334696318860076860244537719127952730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72368645606329000923061796699431437924","series":"1.2.826.0.1.3680043.8.498.74766685636382355231890631066917831641","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72395496925770440630088720587484704971","series":"1.2.826.0.1.3680043.8.498.10681551343981878484164208219122821636","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72495590491484814220060275823761767516","series":"1.2.826.0.1.3680043.8.498.11635221320778650059419177814252534499","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72547312490317214326791189007411860203","series":"1.2.826.0.1.3680043.8.498.12331574199648102538128587655428879484","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72587440202779699193620339987034971895","series":"1.2.826.0.1.3680043.8.498.97595623318288114956350539729031484666","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72697172162233919540021967137047646179","series":"1.2.826.0.1.3680043.8.498.75045571203974707688914190221915068885","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72771360268592017354044657882090174639","series":"1.2.826.0.1.3680043.8.498.11894791424977163777799058378935184415","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72786773761211751423503785260617678955","series":"1.2.826.0.1.3680043.8.498.12694973477861653878363933345783597848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921","series":"1.2.826.0.1.3680043.8.498.57215472262185507167106208233273188371","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72896348955280307080916504120667484813","series":"1.2.826.0.1.3680043.8.498.45743193556197379313356833114655813246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72937134140407212536042212109839895294","series":"1.2.826.0.1.3680043.8.498.31839083454007831826040988624683604591","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73021428171736137311489171988512484948","series":"1.2.826.0.1.3680043.8.498.66667529917830917691344686993748967826","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398","series":"1.2.826.0.1.3680043.8.498.29965821567949062977426906118996592372","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73077380963620788952030569132380549826","series":"1.2.826.0.1.3680043.8.498.71065469650918694377951914612227146270","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73170541731840260065335942048010387599","series":"1.2.826.0.1.3680043.8.498.58654403076202496994035053284885680408","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73198335328613992289471885012751338458","series":"1.2.826.0.1.3680043.8.498.11272814581741749445951008877236390411","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73231153161312542696053060380619432126","series":"1.2.826.0.1.3680043.8.498.48235601771857062237899150357765090790","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73242978867957397254276973409271434573","series":"1.2.826.0.1.3680043.8.498.13364247422680291432659520074475572985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73275019425494043461771574462523171618","series":"1.2.826.0.1.3680043.8.498.16302031700577355613579282898837225831","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73325959289998105909842627876607163467","series":"1.2.826.0.1.3680043.8.498.11276010171643053060455896959987935581","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73448420644251136304786298254801920284","series":"1.2.826.0.1.3680043.8.498.13068472100121293288675082114540048664","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73480422982315825699847838826691577320","series":"1.2.826.0.1.3680043.8.498.20937930321719622535123919197235898581","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73522298807198784022883320957252530465","series":"1.2.826.0.1.3680043.8.498.27044054405232745095097303417430125491","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73545582819102316606081606511277579853","series":"1.2.826.0.1.3680043.8.498.10171278225769588899094597495037880695","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73616237718955147476241731466078978343","series":"1.2.826.0.1.3680043.8.498.25415711095516739786550523901987100045","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73716752081143767997507440278231972668","series":"1.2.826.0.1.3680043.8.498.80395533797716021114166858775570728384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73825252186234755453650759159564715478","series":"1.2.826.0.1.3680043.8.498.50990784220316390809469217007715514676","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73891117356545433634964184054304516063","series":"1.2.826.0.1.3680043.8.498.95530990272860605123183374516562182233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73988798714023285209290281409329527637","series":"1.2.826.0.1.3680043.8.498.13058260456328151423845469919195704060","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74078340346189210015785395961373387782","series":"1.2.826.0.1.3680043.8.498.12125971196918230236440101661319625904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74140806279409854828032496578072554031","series":"1.2.826.0.1.3680043.8.498.63224987048684326380669910674168420647","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74177736542522869266952610634091628727","series":"1.2.826.0.1.3680043.8.498.13580184590055805860375263497363601380","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74215611637606164186239511504256137871","series":"1.2.826.0.1.3680043.8.498.11752085734690793765141821212068562800","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74222294235046321147381553078000820604","series":"1.2.826.0.1.3680043.8.498.10514910777181618647057702399296875999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74247325306307303036016759070403973520","series":"1.2.826.0.1.3680043.8.498.12674768684671530384696381269236983043","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74282280145383357786409764912398881272","series":"1.2.826.0.1.3680043.8.498.10852820295431225056280652462069348730","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74388661917632804671669952747214084502","series":"1.2.826.0.1.3680043.8.498.61438934681457445274680398721753195967","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74423888576992387013857540016155410962","series":"1.2.826.0.1.3680043.8.498.86295765030797479460253097726603043511","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74471724874857406578316075456414902954","series":"1.2.826.0.1.3680043.8.498.42921295817621007997164422583487668928","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74688733778107958348746767768064374883","series":"1.2.826.0.1.3680043.8.498.11283855974785248268799953333069965247","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74725806565437665548056410415178280121","series":"1.2.826.0.1.3680043.8.498.95564489759109597676366289447413919343","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74855168460956358728708865599401983005","series":"1.2.826.0.1.3680043.8.498.79696672815999858543815934385007091319","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74914646210033032631540802996162814268","series":"1.2.826.0.1.3680043.8.498.10457829742037527099959638527164747718","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74940598515597814600837126537274199312","series":"1.2.826.0.1.3680043.8.498.18204311483922757099337685198153728008","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75126165133796914746114117844697263697","series":"1.2.826.0.1.3680043.8.498.15392025493664051523365355293218857706","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75186274888329123179129240384960518861","series":"1.2.826.0.1.3680043.8.498.73107234960996327959399955356481788257","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75276563505367629340309025030148269450","series":"1.2.826.0.1.3680043.8.498.11121087469548697778646461093675433689","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75378323910171368568360074903297446681","series":"1.2.826.0.1.3680043.8.498.11146018721627254306615160504309298157","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75419034794059601820999512200601915669","series":"1.2.826.0.1.3680043.8.498.55654046800949009796603402824009211963","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75427432788337288702609576920198810880","series":"1.2.826.0.1.3680043.8.498.41114493372736190199623135917436829711","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75522002047190283617659083787085821064","series":"1.2.826.0.1.3680043.8.498.12014004182240137250266732872077343355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75548650589349966221875964542604002971","series":"1.2.826.0.1.3680043.8.498.30823566681814300352044706126656624784","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75627574440860942023321071382766359827","series":"1.2.826.0.1.3680043.8.498.36382958796527226637482792827341384168","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75670996863043899245632198549551296292","series":"1.2.826.0.1.3680043.8.498.51624357801085993660832964301254057610","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75739561647115166735277767560308263446","series":"1.2.826.0.1.3680043.8.498.99467155067285071507452654219576907736","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75751784078102748563815298069799859542","series":"1.2.826.0.1.3680043.8.498.66917222226582771970360255178342655073","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75773447415943537938555952303103557864","series":"1.2.826.0.1.3680043.8.498.10667764341527077403935530839077841299","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75924957745868212343758911856809118662","series":"1.2.826.0.1.3680043.8.498.84823077839172258143096444319683719668","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75980374267002793040832157122311396174","series":"1.2.826.0.1.3680043.8.498.97449466356228427249096647992674122969","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76027799502777251766328820405966979386","series":"1.2.826.0.1.3680043.8.498.10107178824288925541943025533677616645","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76107330932071966465869847071480517541","series":"1.2.826.0.1.3680043.8.498.13077945896636845241955537383865461622","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76125027090596913549224250979076533606","series":"1.2.826.0.1.3680043.8.498.37263885624923149473852103759892886580","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76222770668660274805993655800271653712","series":"1.2.826.0.1.3680043.8.498.70491093236966747743670966590896340552","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76281930374588438789344170829280438697","series":"1.2.826.0.1.3680043.8.498.24748913430760312505968825864829193546","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76514087677277144682713362609812138150","series":"1.2.826.0.1.3680043.8.498.10827862685246127010978347914041852783","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76538737802277241976971950690663079376","series":"1.2.826.0.1.3680043.8.498.33122731482865375697062569216631749893","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76620177268652814425535081607587091903","series":"1.2.826.0.1.3680043.8.498.11147027324491136486550754717734995884","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76662146598924492573212477881527658730","series":"1.2.826.0.1.3680043.8.498.48976885437625362690408431256064595955","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76757251008849492985513280752011229119","series":"1.2.826.0.1.3680043.8.498.78289813834186086401601232573480227073","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76841077743272729415666709641073096909","series":"1.2.826.0.1.3680043.8.498.12000005181649088921296354829929785574","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76909970028125318280995002129970165723","series":"1.2.826.0.1.3680043.8.498.23569669590097802688851500633496759659","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76950154099427419631807259601983648254","series":"1.2.826.0.1.3680043.8.498.56921713197548014129504712441499543414","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77018355356797290743788712046815540223","series":"1.2.826.0.1.3680043.8.498.10217951647180921103082467208704037849","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77060858802024387619459645469871885412","series":"1.2.826.0.1.3680043.8.498.37032056580334169144316274873674572505","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77150650601457686196626648868552167695","series":"1.2.826.0.1.3680043.8.498.11614978110537962319418151766959233974","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77164129615096222757740198026718214733","series":"1.2.826.0.1.3680043.8.498.22624750622791414062733220305686058640","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77244685031928454051409193776790348219","series":"1.2.826.0.1.3680043.8.498.64964673135980548816004887156243716030","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044","series":"1.2.826.0.1.3680043.8.498.37638626090958159787991221364146474424","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77283634197289382453511269950488893300","series":"1.2.826.0.1.3680043.8.498.93529039172626705624673530388827819455","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77306755058630855443644908572157134062","series":"1.2.826.0.1.3680043.8.498.83885703385324688271735129347728213309","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003","series":"1.2.826.0.1.3680043.8.498.49091966385194517265009889541603216700","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77378803863827312829317008636359414012","series":"1.2.826.0.1.3680043.8.498.50802927580189953800289886282906661552","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77421709644087006134426085284321515809","series":"1.2.826.0.1.3680043.8.498.59948595512110365553405069002907176832","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77442647074650043391818472068835112806","series":"1.2.826.0.1.3680043.8.498.89766158177150054476701576906284829763","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77508762938574470575737016745357126587","series":"1.2.826.0.1.3680043.8.498.12933816065411170793131384601807893204","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77541648519018799027301684184453151471","series":"1.2.826.0.1.3680043.8.498.67393799598104528706581832258168738533","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77577135626047243581253857225991426727","series":"1.2.826.0.1.3680043.8.498.65565123449647584104845343952853326627","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77611073647079414772859496272387571059","series":"1.2.826.0.1.3680043.8.498.91069508739393283741296806012154704626","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77628479664122209492622263818114972176","series":"1.2.826.0.1.3680043.8.498.90477640557049233745497205499082901854","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581","series":"1.2.826.0.1.3680043.8.498.11114438654149519912156377368989428275","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77760453966349959552328150395676117417","series":"1.2.826.0.1.3680043.8.498.35286280890468068170935418674732254325","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77801860555102174958520570751272642387","series":"1.2.826.0.1.3680043.8.498.12257355353186245819327290720531169994","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77843765997593346714211135369376676497","series":"1.2.826.0.1.3680043.8.498.12568350805844417953195312673726963285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77911973519830915742655727368286737079","series":"1.2.826.0.1.3680043.8.498.63155901883047142544655146358519988258","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78003548255041391467292087330671945247","series":"1.2.826.0.1.3680043.8.498.10226628268300684793235658278341485232","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78079900750417018394321006978343945239","series":"1.2.826.0.1.3680043.8.498.29049279322319893601972906448169083059","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78084594560404633421210837175859618437","series":"1.2.826.0.1.3680043.8.498.65614551207484406087842150903751996806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952","series":"1.2.826.0.1.3680043.8.498.95538163610479972610205395052839438071","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.78238394512811614973689011297363399053","series":"1.2.826.0.1.3680043.8.498.13147980998793948573613661955748639796","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78286395605671977872377352305890466464","series":"1.2.826.0.1.3680043.8.498.12610999205387048179705761496902257671","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78465765647012043478539489340002371960","series":"1.2.826.0.1.3680043.8.498.10597547738522893302080301306352972305","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78476873086705419136436170130940915568","series":"1.2.826.0.1.3680043.8.498.10030972446008853822472209216602063972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78574437345603052437892513165832297960","series":"1.2.826.0.1.3680043.8.498.34025149529848171674505640925675181110","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78657055734742806687703707756153505756","series":"1.2.826.0.1.3680043.8.498.99155640537243022567834334047768018126","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78685661415919658264222803849222055515","series":"1.2.826.0.1.3680043.8.498.43461403602070980065390304654321120808","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052","series":"1.2.826.0.1.3680043.8.498.57281409743147577596751646279843599110","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78812661021494224637586904573732040856","series":"1.2.826.0.1.3680043.8.498.81643668572912287584520330578463038132","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78968420394708302953391711477016139949","series":"1.2.826.0.1.3680043.8.498.63032214662108564869992168801402428741","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79038153803632889185429988483097161887","series":"1.2.826.0.1.3680043.8.498.39383828321048680252161611314634890892","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79087227780630716132147615846840128474","series":"1.2.826.0.1.3680043.8.498.32067630994203896036897476766955220365","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79122189963582241618519333445055548347","series":"1.2.826.0.1.3680043.8.498.10048366580779524426815767397156901144","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79224785668205420138951103045759609475","series":"1.2.826.0.1.3680043.8.498.34601146480564906959832648056452110568","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79245771997885598399203314109623928718","series":"1.2.826.0.1.3680043.8.498.12106531351508401939655788910742401973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79267465462008228906382361587050826784","series":"1.2.826.0.1.3680043.8.498.12009697709832977331012155418058226273","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942","series":"1.2.826.0.1.3680043.8.498.36898834529003026596255105381017001101","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79341955397030543255969676347554165142","series":"1.2.826.0.1.3680043.8.498.24774563452685699830267543598783887971","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79524312100988116256514335064057385347","series":"1.2.826.0.1.3680043.8.498.13432222038487980413959590848715446452","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79560170727391465844958390474997798803","series":"1.2.826.0.1.3680043.8.498.22156610816936127482082482017199624803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807","series":"1.2.826.0.1.3680043.8.498.22931869662151323094708929338186135578","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79672421957132993896448375311641998688","series":"1.2.826.0.1.3680043.8.498.13648949542788477152367172699105112474","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79741629769872115526056483024289600492","series":"1.2.826.0.1.3680043.8.498.92842136731488728237674332544473245077","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79977436705480715175480363175316375576","series":"1.2.826.0.1.3680043.8.498.97706442407077874333307034975638800026","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80058869639900582578987760750483296668","series":"1.2.826.0.1.3680043.8.498.45657043992357273226721677452918817348","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80146250663015309456767479960653638934","series":"1.2.826.0.1.3680043.8.498.48369365263762233080831318676990737146","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80192355832283305029770183988436644845","series":"1.2.826.0.1.3680043.8.498.54881751591210584730838727742032157255","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80247974926646352443043954025450139721","series":"1.2.826.0.1.3680043.8.498.79536481073443667600570254387086647357","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80290749690597272632725356393056746299","series":"1.2.826.0.1.3680043.8.498.60629750038795573872610764386026240573","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80312559977515578735238310472949287111","series":"1.2.826.0.1.3680043.8.498.13287053153809519233714827328728592270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80325205657974094483016625702390834774","series":"1.2.826.0.1.3680043.8.498.87781462578821126534569173304779270184","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80433141990021472112645841723991584182","series":"1.2.826.0.1.3680043.8.498.61740846590793311333271423749763403162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80514754572353458107286859773300865138","series":"1.2.826.0.1.3680043.8.498.59389333591936707634338662339374543098","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80557835452734362083552169469427388003","series":"1.2.826.0.1.3680043.8.498.10539330415833759007434776325285960593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80627605463642274852766102874822833365","series":"1.2.826.0.1.3680043.8.498.30487688133195626892371532847259187211","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80642348849062865125509773837278900750","series":"1.2.826.0.1.3680043.8.498.10109980739813198270437457621591796680","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80661392139668653552110941896475066204","series":"1.2.826.0.1.3680043.8.498.55959086132149567738687833507236606594","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80698510014594772882040509716764819716","series":"1.2.826.0.1.3680043.8.498.11231601372518210131821728268695055783","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80741358365199721720335324482078866565","series":"1.2.826.0.1.3680043.8.498.44543641063733307350708945367329788251","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80851365368114686105558347899990395309","series":"1.2.826.0.1.3680043.8.498.68350745597980085469636118400254059198","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80878057166752747059190481131529937715","series":"1.2.826.0.1.3680043.8.498.11797446353133336857026355280665012480","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80931292574057303692838531323879513417","series":"1.2.826.0.1.3680043.8.498.85290420872524751575936539984396200173","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81024897463656728358703040759074583549","series":"1.2.826.0.1.3680043.8.498.89097676127893131167177247610874308011","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81096597169632636454986988944050298701","series":"1.2.826.0.1.3680043.8.498.46655028409163498946698317825352557601","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81142440700884513521007992957945568924","series":"1.2.826.0.1.3680043.8.498.12354830307920158990659600931604321847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859","series":"1.2.826.0.1.3680043.8.498.65681107235010937222094903579969095842","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81342416992724702402763591633702802359","series":"1.2.826.0.1.3680043.8.498.64718967974843443331053403229591742131","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81407504276375197122586620633759174421","series":"1.2.826.0.1.3680043.8.498.60990958612606300268136019058437679716","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81464322239381998685473520085495957135","series":"1.2.826.0.1.3680043.8.498.17579901208344094936572921760086449761","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81495303185504163367736663647489556517","series":"1.2.826.0.1.3680043.8.498.11196209577197797328125829640262703457","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81567030584250750745410094870790052181","series":"1.2.826.0.1.3680043.8.498.35363323757373440637572305104979678956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81631925197682464058124413934958934086","series":"1.2.826.0.1.3680043.8.498.42144272198092168845866058991775258168","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81651455914141365028015556083080638757","series":"1.2.826.0.1.3680043.8.498.84439861732159561473561823983485326610","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81698411330214788362326470979160431674","series":"1.2.826.0.1.3680043.8.498.11959340270217248689602636988135162049","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81790990482142045640192729399175106255","series":"1.2.826.0.1.3680043.8.498.87923035954243389858574430218752942958","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81796385087779065457668769443782458844","series":"1.2.826.0.1.3680043.8.498.19537582685406210926629804010314746082","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81822709772232914298414700148454511661","series":"1.2.826.0.1.3680043.8.498.32125835372295425838019516225355410511","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81892104897856100357169774214718573167","series":"1.2.826.0.1.3680043.8.498.11719158633000815940504709553111190459","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81964000393640770822199916465038759069","series":"1.2.826.0.1.3680043.8.498.10144666208926810402842707622102403691","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81981151447274200345866874172324295172","series":"1.2.826.0.1.3680043.8.498.58448424236982510147690562259047954089","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82012532783011843175172575198326783678","series":"1.2.826.0.1.3680043.8.498.39398901511378513230761425336463077432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82143276307898132314496735515204481508","series":"1.2.826.0.1.3680043.8.498.21505325131058576318764741147787366613","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974","series":"1.2.826.0.1.3680043.8.498.86517895006392259407872634218546422483","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82169173362972793731001247413129649263","series":"1.2.826.0.1.3680043.8.498.83033617594458607930224960165436346736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746","series":"1.2.826.0.1.3680043.8.498.46727660372122381730380699314725563984","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82447100001417065190678617013741085481","series":"1.2.826.0.1.3680043.8.498.73070150515907729162246629879218887671","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82523073559792523830827952754346276051","series":"1.2.826.0.1.3680043.8.498.13104733816291166577831114533024876506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82579118531566493365068788437906656162","series":"1.2.826.0.1.3680043.8.498.18584792792478252219435548310068647490","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82591599611540849522609405219902904134","series":"1.2.826.0.1.3680043.8.498.86950108751795793725670310255544669767","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82731606763525898032678680051515732914","series":"1.2.826.0.1.3680043.8.498.44510117843291479798041896388308327149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82858981777405241601610241542173548956","series":"1.2.826.0.1.3680043.8.498.13793753431964742965987199149755377798","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.82925474533853765570570726554869886925","series":"1.2.826.0.1.3680043.8.498.10213522702855644549318723032727986581","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83000639623679673469640458882929412717","series":"1.2.826.0.1.3680043.8.498.42274928172779372316854852774961334177","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83054899303871276184985881638619543362","series":"1.2.826.0.1.3680043.8.498.32947394021202860594815885675135460494","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83061811791652434833797599835671351097","series":"1.2.826.0.1.3680043.8.498.84025684521731697747175654055615533009","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83152040259754415849193046299744699239","series":"1.2.826.0.1.3680043.8.498.27232971055528981376856412037366886923","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83235374118671754689663732057314083332","series":"1.2.826.0.1.3680043.8.498.35314679125018472852246138962851816206","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83253778356746892201320443660102495790","series":"1.2.826.0.1.3680043.8.498.47140493893550172185403938831812976398","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83384018761005700404957421316476952190","series":"1.2.826.0.1.3680043.8.498.46573941398927675458397919238703080717","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83495448827553002152296206085110883602","series":"1.2.826.0.1.3680043.8.498.11464124303928510535709305955057536222","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83569065014916050100714967945484998462","series":"1.2.826.0.1.3680043.8.498.60808536451958395730511306242290597063","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83740112248863750162633901687556705057","series":"1.2.826.0.1.3680043.8.498.17608878576752116838500773102693311387","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.83847285247209768917850605454140194088","series":"1.2.826.0.1.3680043.8.498.12942976515779894448618196251090530408","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84047134932018909998648460546980598926","series":"1.2.826.0.1.3680043.8.498.88766180836033156577774321816131327830","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84062602050757262792437893691591415560","series":"1.2.826.0.1.3680043.8.498.35222746759250843790528618039534639659","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84069767112794890061709127671908414399","series":"1.2.826.0.1.3680043.8.498.98978225204650147157083685535791742301","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84230432137191939390680841389370206819","series":"1.2.826.0.1.3680043.8.498.74480389940283871734031880500356295516","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84292569246911697141902796190329671261","series":"1.2.826.0.1.3680043.8.498.10321899433171825545062098856487444495","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84477198486144701928775090913243250655","series":"1.2.826.0.1.3680043.8.498.11569307143873079186186231270752540644","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84486787871353595019134028899317143475","series":"1.2.826.0.1.3680043.8.498.13461120940106505397309001424037179551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652","series":"1.2.826.0.1.3680043.8.498.12948020789996463062577357257441366023","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84720476187201404764508451536959273676","series":"1.2.826.0.1.3680043.8.498.53444957040955873379192669654916848569","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84827027113422423894439539536275453132","series":"1.2.826.0.1.3680043.8.498.11257179093583576891688900444642245646","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84857368518703570476541203284655412396","series":"1.2.826.0.1.3680043.8.498.52515450380720666931306091516022078055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547","series":"1.2.826.0.1.3680043.8.498.36035684781295937473611340920645594609","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85036616529391787087213173897728060798","series":"1.2.826.0.1.3680043.8.498.64401634656189212743693926425027215875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538","series":"1.2.826.0.1.3680043.8.498.96722287350274129921096603633786586778","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85212353344613461454490252928274988058","series":"1.2.826.0.1.3680043.8.498.71311955402264269277199517005392542552","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85326542895076582985200613540648336047","series":"1.2.826.0.1.3680043.8.498.11440851724646850776981708987278589690","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85347263026430684291021533170619978114","series":"1.2.826.0.1.3680043.8.498.11496954700090394337110190474758498246","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85397267100597758920280635204297906286","series":"1.2.826.0.1.3680043.8.498.68084125463875671695513009042787301415","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85454884397223412543050876117416146750","series":"1.2.826.0.1.3680043.8.498.95909897582162901176368372619138065078","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85475892602877115812059264719695919613","series":"1.2.826.0.1.3680043.8.498.11576923921101662873484320930482354831","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85566900782101872452970957532250214590","series":"1.2.826.0.1.3680043.8.498.45570768947094130853267564560536584113","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85624441072689653093363046226467141762","series":"1.2.826.0.1.3680043.8.498.46485974091860453017519079340625804011","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85663162480758065951465617999516770647","series":"1.2.826.0.1.3680043.8.498.16612971812172815718797666195620024331","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85715956158218184426400151797364834203","series":"1.2.826.0.1.3680043.8.498.86406938944037921457377784207910946538","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85728170022176096748897349447671687164","series":"1.2.826.0.1.3680043.8.498.13014503548576510228771044250205967856","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85760360279564672425828346507309104394","series":"1.2.826.0.1.3680043.8.498.38637575835398335825895012964530002881","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85870577082315172424837183881210532800","series":"1.2.826.0.1.3680043.8.498.43932483228843304528846462790335476125","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85883562680953255833567595509408888410","series":"1.2.826.0.1.3680043.8.498.66246341777706739961189384740417956931","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85932495800565018166769303205378781707","series":"1.2.826.0.1.3680043.8.498.57385785817020792811946704088395937443","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85976668143927055109184433855426232055","series":"1.2.826.0.1.3680043.8.498.10203913153895404444730787301598588741","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85992651772359515416050249463715355943","series":"1.2.826.0.1.3680043.8.498.10031610669956278421454348392501437134","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86029947807373604003071651721618330827","series":"1.2.826.0.1.3680043.8.498.82817641536887601175394055856334201667","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86082522186055358984181640226529351801","series":"1.2.826.0.1.3680043.8.498.12172958033272250287591857229746945938","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86216155858324761845740646421500394797","series":"1.2.826.0.1.3680043.8.498.45057566907213916756772964485395030611","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86263264208309369957750915520115390972","series":"1.2.826.0.1.3680043.8.498.45354928950908678517007215445044654288","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86280033511814602594654628677479528508","series":"1.2.826.0.1.3680043.8.498.36208346073547593728922583430657122504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889","series":"1.2.826.0.1.3680043.8.498.18107545475162181541580251058485379571","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86351141114604781108401133410934252655","series":"1.2.826.0.1.3680043.8.498.77338707475453874744207452790588218053","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86409878732405036646840908998274047391","series":"1.2.826.0.1.3680043.8.498.60786873671793440549542650658352037802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86435271359723045834079326733478538148","series":"1.2.826.0.1.3680043.8.498.22144202160426098663769707671028006189","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86516555757334047240474191101770683543","series":"1.2.826.0.1.3680043.8.498.25774891197524614289953868553917798613","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251","series":"1.2.826.0.1.3680043.8.498.75490224058819385453983161734150451147","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86715997235054599024919893656486735623","series":"1.2.826.0.1.3680043.8.498.82679440737344072160986435282533830134","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86876540394528304678398111135600894869","series":"1.2.826.0.1.3680043.8.498.50098392453798652245595308590870046915","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037","series":"1.2.826.0.1.3680043.8.498.73414308736751658627435120649630352251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87018440624359538250658207994111333051","series":"1.2.826.0.1.3680043.8.498.12404980259373228942134169832840064805","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87064372531225943854092598173945043861","series":"1.2.826.0.1.3680043.8.498.49000213622517773233919416858806591762","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87185817772180109135714642184874643715","series":"1.2.826.0.1.3680043.8.498.10860254955896804585873051987989248239","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87208714421237491643915928765740747633","series":"1.2.826.0.1.3680043.8.498.12962374538604904686763588468307061483","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87223930502803944549504594723314220105","series":"1.2.826.0.1.3680043.8.498.22697923714349436391122864541808541159","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751","series":"1.2.826.0.1.3680043.8.498.12655813364018273608146684382367463358","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87338030934656808032044071206920470285","series":"1.2.826.0.1.3680043.8.498.73758726225406457221745364983499907863","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87413924318793242918916384453755054267","series":"1.2.826.0.1.3680043.8.498.21213931178430617126624737498227066645","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87476514600989622078259281213313867927","series":"1.2.826.0.1.3680043.8.498.10622948534822121958627116350634176674","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868","series":"1.2.826.0.1.3680043.8.498.11093114814538216346090786618836040683","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87645071186049036230936726742129446188","series":"1.2.826.0.1.3680043.8.498.41168489298227330523731967416471413581","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87679606849449562627719676100773778015","series":"1.2.826.0.1.3680043.8.498.21280972925231511104128150752789483267","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87742737485142445536023876862256284839","series":"1.2.826.0.1.3680043.8.498.52338604701379144885107827805369131742","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87797202230328868466634480198668365648","series":"1.2.826.0.1.3680043.8.498.29090051001051139226569782443619294640","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87845297884497964690686366000533570840","series":"1.2.826.0.1.3680043.8.498.56099044038304164818628808659863979474","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87998832540756872371878613339628615880","series":"1.2.826.0.1.3680043.8.498.90388789325567528259983461196072043187","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88064086448060280325724227239585797761","series":"1.2.826.0.1.3680043.8.498.19842885814682397529995444073905657882","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435","series":"1.2.826.0.1.3680043.8.498.13289651041724897374069399159164397015","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88189503151202366936330411627291827689","series":"1.2.826.0.1.3680043.8.498.11134284959205221090895275957229381024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88368930449171804348215887291291274615","series":"1.2.826.0.1.3680043.8.498.16820912159723089487964324685078442429","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88457076333783710706782784856816112502","series":"1.2.826.0.1.3680043.8.498.70058899484059877368499397190540325499","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88505044370271842163477475980178015566","series":"1.2.826.0.1.3680043.8.498.37886403150435095879713528995051974366","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88565389747258713872180859306907392080","series":"1.2.826.0.1.3680043.8.498.43269272045798415271544213602590644558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88574487918084130981335189023288704909","series":"1.2.826.0.1.3680043.8.498.99534204990992176798779137802603354275","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88620225126852321191690360547272730103","series":"1.2.826.0.1.3680043.8.498.10670850676655143343840942446742195479","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88679328523895187500714090963655363787","series":"1.2.826.0.1.3680043.8.498.10511808450493143627578469483590008353","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88687994744258163246685527273559741656","series":"1.2.826.0.1.3680043.8.498.10178277458944454091733905645794121030","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88752039058074948027322975455607258941","series":"1.2.826.0.1.3680043.8.498.12692434127737425958545665733954845852","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88801151487634957068752641200383474212","series":"1.2.826.0.1.3680043.8.498.30792472047678219072403064273622604542","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88872155446935518174304990308380906112","series":"1.2.826.0.1.3680043.8.498.83224160632897589675570726563148806312","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88896246957673838955012057210098034177","series":"1.2.826.0.1.3680043.8.498.12031239741428324037895571470034957240","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88962401400598535739494671683916272476","series":"1.2.826.0.1.3680043.8.498.51082148881366720962083167558592106479","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89045629523485194086986785222960864883","series":"1.2.826.0.1.3680043.8.498.44935157991390438392823838670952332251","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294","series":"1.2.826.0.1.3680043.8.498.75440791631586852233561055673370410689","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89123035099342878362576960682354791116","series":"1.2.826.0.1.3680043.8.498.14509287995035583775842780632710274558","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89240184178186399632216878563608101260","series":"1.2.826.0.1.3680043.8.498.70454325573259006278427300686560690714","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89302191571937190529967168726521003428","series":"1.2.826.0.1.3680043.8.498.13241256639855050099189777376705462839","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89332108074590559691047812196991368746","series":"1.2.826.0.1.3680043.8.498.77965853475859597275277543591620804974","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89350648596674998203116057439419132072","series":"1.2.826.0.1.3680043.8.498.21884138407487544206395444248124641417","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497","series":"1.2.826.0.1.3680043.8.498.16482565402805434085500658327534198135","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89464179959474561696319051639748844244","series":"1.2.826.0.1.3680043.8.498.96854414616556625055030592708295740494","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89576499383950909889665633610791093334","series":"1.2.826.0.1.3680043.8.498.91795957425772206929744092341156663436","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89720428711655200442262503508409163392","series":"1.2.826.0.1.3680043.8.498.39618788582695815180555511047071022930","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89746004948304570186940283501211828475","series":"1.2.826.0.1.3680043.8.498.10271133517841070537097590188871860916","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89857801102910604766959864464865910585","series":"1.2.826.0.1.3680043.8.498.60306923104327597943764582105636885275","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89943815722422273305517145510644432987","series":"1.2.826.0.1.3680043.8.498.11932477057665933426764952386969358430","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89967772422936726289461201791871771011","series":"1.2.826.0.1.3680043.8.498.26880821424155747731239615119071500102","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90037507233074511524602897439257102032","series":"1.2.826.0.1.3680043.8.498.45259057644595612032423599706239241646","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90103210546658057701715546323552886913","series":"1.2.826.0.1.3680043.8.498.12799211814512492453352821040357224974","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90172333170788218623219231170360850947","series":"1.2.826.0.1.3680043.8.498.12166348384512337444583493025151585800","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90223492515368389331845681759355261030","series":"1.2.826.0.1.3680043.8.498.78564469233815946593224540293828967142","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90251533154603802045666102095577144628","series":"1.2.826.0.1.3680043.8.498.13383342113454615503404873982673437255","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634","series":"1.2.826.0.1.3680043.8.498.48546784051653888035284136919830207611","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90288598546566291188788375261370919222","series":"1.2.826.0.1.3680043.8.498.78249252106150931941211333779870910563","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90331082101665002495739626395247831768","series":"1.2.826.0.1.3680043.8.498.35138853696099684532043962146037012434","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90433488790413321102007716267340900056","series":"1.2.826.0.1.3680043.8.498.80920481011371826336408853331588938562","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90466423642678253933409966898120881010","series":"1.2.826.0.1.3680043.8.498.84166593982084294378070669700075462657","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90508461713737190527021531642732695353","series":"1.2.826.0.1.3680043.8.498.47695185748887614771385529849222424553","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90551187439998209641370107095424494085","series":"1.2.826.0.1.3680043.8.498.37109207644391483397183954435171017883","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90629760164319310375909187696899268080","series":"1.2.826.0.1.3680043.8.498.74586565408656055150825142064461977605","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90679199017361780574384378857945322240","series":"1.2.826.0.1.3680043.8.498.55900669788582734989735834541732416123","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90764713690393618577990540005126771712","series":"1.2.826.0.1.3680043.8.498.12044547356992974355865687018037961533","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90858397443821905121918628339486070979","series":"1.2.826.0.1.3680043.8.498.36869765339731306297504564560703874802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90985037792656682958281230441654772668","series":"1.2.826.0.1.3680043.8.498.65929959922583951063750367431305959210","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91106275065657355304239097295699332135","series":"1.2.826.0.1.3680043.8.498.11447879003585950577035580313434019627","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91124979914013108948829704350601987380","series":"1.2.826.0.1.3680043.8.498.10351344245367186705243607443808109543","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91139126799820974665417903552066103454","series":"1.2.826.0.1.3680043.8.498.12879771537175282761006813996424871776","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91220255464004258635083940797356822178","series":"1.2.826.0.1.3680043.8.498.72745479526379833121266623353527763465","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91237082716839283641165609666644466929","series":"1.2.826.0.1.3680043.8.498.85379909208662201549710633899974392990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91328985601265188961626433080766431089","series":"1.2.826.0.1.3680043.8.498.80935096842709069929273781466774653398","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365","series":"1.2.826.0.1.3680043.8.498.88514465425565341590167126348593825887","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91391446167553025113367459399831345295","series":"1.2.826.0.1.3680043.8.498.65879075633255849165716424628142864556","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637","series":"1.2.826.0.1.3680043.8.498.10302676288832408062385920434616284137","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91499489440453244654238202136675083579","series":"1.2.826.0.1.3680043.8.498.54465200225249126737330535826782296602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91611403539216247154562810076063878924","series":"1.2.826.0.1.3680043.8.498.47643916648162020260428587232275230255","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91727248305411866145131034915485728037","series":"1.2.826.0.1.3680043.8.498.65940652757551612459730200070763092517","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273","series":"1.2.826.0.1.3680043.8.498.49977638453342949575078854371039659392","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.91883829404782805058767342767479130421","series":"1.2.826.0.1.3680043.8.498.45596264959713117306665854340586903629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91913894091933783677568170210064807665","series":"1.2.826.0.1.3680043.8.498.16064523796924404112527251443463791855","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016","series":"1.2.826.0.1.3680043.8.498.13343960847951042335151765410452308495","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92063841381005962285592592672311044745","series":"1.2.826.0.1.3680043.8.498.80875393150965374939733163414778247588","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92126358259118823397932994812520542494","series":"1.2.826.0.1.3680043.8.498.90638015107373900168428245428289321145","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92249560662718438332916289759028859881","series":"1.2.826.0.1.3680043.8.498.18905180657566402865459799766379929836","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92301915122994899889226415477164044164","series":"1.2.826.0.1.3680043.8.498.12773426231277574465922750367362162705","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92407834172018110142589158281575319154","series":"1.2.826.0.1.3680043.8.498.12657834895825318008696227125399371059","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.92445779773398371452908052305534240654","series":"1.2.826.0.1.3680043.8.498.11667116792843697695232091973557951874","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.92581137608969847756039955144478716506","series":"1.2.826.0.1.3680043.8.498.78319515119625416311962030697637997179","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92602515666185999130675702469977883616","series":"1.2.826.0.1.3680043.8.498.98915373361308314698393870478367068769","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92650426727723557280117118587545413917","series":"1.2.826.0.1.3680043.8.498.81638785801367479511703905067939445661","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780","series":"1.2.826.0.1.3680043.8.498.96633209981873608197483492142226418637","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92758635328468215409701857842011454608","series":"1.2.826.0.1.3680043.8.498.17093965173575762625971416161897231724","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92854445271746966212168817610305595078","series":"1.2.826.0.1.3680043.8.498.35477216210067572642453181115440044332","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055","series":"1.2.826.0.1.3680043.8.498.42956330769567762540384918288427995560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93062749017302143105207135454940053142","series":"1.2.826.0.1.3680043.8.498.24895941280742799881956481667940052661","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93141871723058796160969424330424882014","series":"1.2.826.0.1.3680043.8.498.10177096047990954879010263792735401557","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93182602591865019295859776146674822677","series":"1.2.826.0.1.3680043.8.498.12350719286613120139321272394734015815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93230313557251171119475173996942942855","series":"1.2.826.0.1.3680043.8.498.55997983641202201020826684551355404659","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93318293884671598055223778443354732793","series":"1.2.826.0.1.3680043.8.498.14599561132353666893682663255341025917","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93359387652821116106937882388838959209","series":"1.2.826.0.1.3680043.8.498.19174327913447654473467030690670547756","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93555726419626687118723083838655757151","series":"1.2.826.0.1.3680043.8.498.14349541653410245927611807531042566456","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93653421810951484288813767943453706233","series":"1.2.826.0.1.3680043.8.498.78363984273169634807496609837143164818","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93719148934772792523193842156217224487","series":"1.2.826.0.1.3680043.8.498.11419810567998799174831630274393000772","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93747304211180517979657679136059719120","series":"1.2.826.0.1.3680043.8.498.11932822315126859765930292991700201954","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93848205752465320137433228658870667418","series":"1.2.826.0.1.3680043.8.498.61743573001292487676235759113800028712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308","series":"1.2.826.0.1.3680043.8.498.82640830115722867469816039125688281605","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94002633501154835018549300667479350401","series":"1.2.826.0.1.3680043.8.498.40641392621838956864387744252102582154","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612","series":"1.2.826.0.1.3680043.8.498.42540465894547225058377872560412813180","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94252884278314596961426629318911909075","series":"1.2.826.0.1.3680043.8.498.11203033684347285526623644920234494042","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94274898137310893945218731717226819541","series":"1.2.826.0.1.3680043.8.498.91044670928107072398652825343240595502","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94335780236370744781443631231792792083","series":"1.2.826.0.1.3680043.8.498.13240003924154720829986308542975166394","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94393348753936295687030819683853459606","series":"1.2.826.0.1.3680043.8.498.99451239771402863798163062983712186463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94448720962054029575571654250994769402","series":"1.2.826.0.1.3680043.8.498.13323642473137073052550648095989628281","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94590838906158914501563687877014778914","series":"1.2.826.0.1.3680043.8.498.11884459390999248285375843143460248711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94607524665407029611795868588964734480","series":"1.2.826.0.1.3680043.8.498.10336961431591119011091249060412966585","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94658317716039594639343437780115140531","series":"1.2.826.0.1.3680043.8.498.11982894252638366047104938094321290267","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94722363227977785887442194094888460046","series":"1.2.826.0.1.3680043.8.498.11514534390961570553944872815847190676","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94753144127920558221268610279343359449","series":"1.2.826.0.1.3680043.8.498.48375356689650141776273684634223182131","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94774472240035721003264951868030913589","series":"1.2.826.0.1.3680043.8.498.82770061376488417706642658375775774937","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94921929374845793599006362190119159537","series":"1.2.826.0.1.3680043.8.498.72480613082295934967208631633881983743","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94945670991495145462605178471390637325","series":"1.2.826.0.1.3680043.8.498.51712183412796719631341903159534733431","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94955017957653373798729896600270551224","series":"1.2.826.0.1.3680043.8.498.87159653406138401833165972154176467671","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95060380238431209646600378585422128745","series":"1.2.826.0.1.3680043.8.498.26169321183228070047240426969401614661","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95091624999382167943525022414631622532","series":"1.2.826.0.1.3680043.8.498.45930284804483229975648398757527291195","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95193282490279394602564266574551123732","series":"1.2.826.0.1.3680043.8.498.78909989026992366638482168922566749800","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95223549428749744711279985897241419512","series":"1.2.826.0.1.3680043.8.498.30959452866959270588803086209736944844","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95321750682026667237717880394536750835","series":"1.2.826.0.1.3680043.8.498.84663229600550165611744658963963954867","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95334149193711661424105715330320414755","series":"1.2.826.0.1.3680043.8.498.12018155166667611945810330389175577426","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95433650759419050864053270236035947981","series":"1.2.826.0.1.3680043.8.498.95035918258628107959033187090816411249","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95490888360439852685783476266753304657","series":"1.2.826.0.1.3680043.8.498.43922387477864921634576437308922915827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95555755747044090477918262640906805885","series":"1.2.826.0.1.3680043.8.498.96227304464415937959178099577545399032","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95570207430778332247200880037915693375","series":"1.2.826.0.1.3680043.8.498.13063663166685726649208874621379496352","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95588576819758266413765165887081039863","series":"1.2.826.0.1.3680043.8.498.10811699484927951870569669118853699888","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95638601079527306640829424921455011698","series":"1.2.826.0.1.3680043.8.498.93102720366408886456423068251274732081","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95705755527332814775935856896567782522","series":"1.2.826.0.1.3680043.8.498.83824963772263263459390714443451746919","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95727100148189624148200652225439854460","series":"1.2.826.0.1.3680043.8.498.55776019235345277626272715003345092903","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603","series":"1.2.826.0.1.3680043.8.498.10875575232887719045673354046843867959","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95864678405973312042815189771902289608","series":"1.2.826.0.1.3680043.8.498.27258580971970399083598582142310782734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95912798959818676102590029534562878721","series":"1.2.826.0.1.3680043.8.498.10813873122177068362271807875568066551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96051765028927322365509303222455708654","series":"1.2.826.0.1.3680043.8.498.11533653364851466614874713196510396009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716","series":"1.2.826.0.1.3680043.8.498.31892728496482952529018853022367510541","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96152894373838410779390571000523933753","series":"1.2.826.0.1.3680043.8.498.19637576524064907776655890495085036095","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96174494699876332875347380499122120925","series":"1.2.826.0.1.3680043.8.498.10162798772737166441562271795844360686","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96323485839061204012967074568679195142","series":"1.2.826.0.1.3680043.8.498.15050837528282866451669115923902063231","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96448229695053562621553132798380456420","series":"1.2.826.0.1.3680043.8.498.13117901156340869840236878827557766588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96559819562317669526803257835398470484","series":"1.2.826.0.1.3680043.8.498.13482310706971063649865095935132860724","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96562865861483086790615358719391771217","series":"1.2.826.0.1.3680043.8.498.23073787898464876589751244295034841781","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96614828054790815108190478795539254570","series":"1.2.826.0.1.3680043.8.498.67717054303812986489818905314325898726","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96657465815875587552665111166265086308","series":"1.2.826.0.1.3680043.8.498.11179917569053659979125092245109078946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96846796615932368166905347054245013501","series":"1.2.826.0.1.3680043.8.498.29597227475005600221687204448835879925","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96944926470854551064948200936284887742","series":"1.2.826.0.1.3680043.8.498.10263031585701567707485003764222030019","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96950011369238410949579095568317214261","series":"1.2.826.0.1.3680043.8.498.11251493941412973833926792631241691706","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97004483591331502283638578516018997228","series":"1.2.826.0.1.3680043.8.498.60738813477876428542942885921170724050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97055845147740306780908051236353219644","series":"1.2.826.0.1.3680043.8.498.14589498943447531667662012506133120047","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97107472223824971578324157203273471706","series":"1.2.826.0.1.3680043.8.498.38537516654266349046383951071259436224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97220604868276521004168045790587997687","series":"1.2.826.0.1.3680043.8.498.77840670314508853814881492858194346905","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902","series":"1.2.826.0.1.3680043.8.498.12171766757557808756644764551613488018","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97269744864161337523188705262974234442","series":"1.2.826.0.1.3680043.8.498.10761918739479390353575232782957613119","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97296323335289879687414244774227160450","series":"1.2.826.0.1.3680043.8.498.27402865503052774073335973637423149952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97353980946516912216109392482235435606","series":"1.2.826.0.1.3680043.8.498.12804169623577251413011037809424593516","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97485476951995325021464683747847126869","series":"1.2.826.0.1.3680043.8.498.84119889195463860372086787795256220391","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369","series":"1.2.826.0.1.3680043.8.498.25088566792964463766562356301938904722","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391","series":"1.2.826.0.1.3680043.8.498.10016548406124462214411986773755008009","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97579738268290461886850261405719501384","series":"1.2.826.0.1.3680043.8.498.39069468426534566611289663707662867367","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97658595748192892503053963872409769829","series":"1.2.826.0.1.3680043.8.498.12908947260240170297023395581369481157","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97696436660690920809826140686268769509","series":"1.2.826.0.1.3680043.8.498.80535050610587984490729084481115087693","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97731968872354496470096174559789586056","series":"1.2.826.0.1.3680043.8.498.10719818108060719105431094671275269391","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97830923192341890652952842494018952920","series":"1.2.826.0.1.3680043.8.498.10753219581878359312489445972819874428","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97909881548577311929907613359449243075","series":"1.2.826.0.1.3680043.8.498.10877345168923350933805011837803408508","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97964867333695657200856428936254426086","series":"1.2.826.0.1.3680043.8.498.26615630349827384167064411047173410153","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98028018520101798863795620289151810642","series":"1.2.826.0.1.3680043.8.498.17424628130711114814131643315897602627","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265","series":"1.2.826.0.1.3680043.8.498.31219608891093107502576383096697280998","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98102475310924339032647376491442498974","series":"1.2.826.0.1.3680043.8.498.83002160890752609128967350982293856428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98143909986671737130865120569673002676","series":"1.2.826.0.1.3680043.8.498.99043412277405562276525207561860054531","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98181040478767526925214101502401448074","series":"1.2.826.0.1.3680043.8.498.13570080483509418676326956104998423449","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98255454187594895604610283922744869720","series":"1.2.826.0.1.3680043.8.498.10398222319302725471808768357083307924","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526","series":"1.2.826.0.1.3680043.8.498.12618036230218914146178568578199718663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98400072082366198211931728015692551635","series":"1.2.826.0.1.3680043.8.498.29881797547516228725832650336752085887","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98541661035340329000785741171799559104","series":"1.2.826.0.1.3680043.8.498.11270994073464195105302534492593756303","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98599322992880180540967962062521615070","series":"1.2.826.0.1.3680043.8.498.22105365422429735919285731188087132336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98781117432577714521231232440503778261","series":"1.2.826.0.1.3680043.8.498.32259642468465962513345235402044969322","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98824066826946884510314135448574304233","series":"1.2.826.0.1.3680043.8.498.50501728487408192382747237851152657225","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98883718766501717651651876978607853803","series":"1.2.826.0.1.3680043.8.498.92017353241330041482426857659681391438","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98914528123216365374152816424181510781","series":"1.2.826.0.1.3680043.8.498.73061804595844561432771349596724754771","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99052905406751383674032587504235261874","series":"1.2.826.0.1.3680043.8.498.88885378327392424537627274974646887651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99108147913292332418681050058312852494","series":"1.2.826.0.1.3680043.8.498.98671141327695609087912787819132706164","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99237666579371266866432031468578336182","series":"1.2.826.0.1.3680043.8.498.36813848322689376501413384637257790158","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99257151613846084492342083136797222759","series":"1.2.826.0.1.3680043.8.498.53432515589294055221467270051868334061","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99413398303724030328059886749330413303","series":"1.2.826.0.1.3680043.8.498.92338835644496409733046230685063005385","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99438805806853675793560604625783460513","series":"1.2.826.0.1.3680043.8.498.78812016946704454549478538673873554728","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99448612525108835020450967944392129536","series":"1.2.826.0.1.3680043.8.498.22047539503023315299258362836065385357","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99458163491875066451936719024359566928","series":"1.2.826.0.1.3680043.8.498.11820254473623635640942895920851010315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206","series":"1.2.826.0.1.3680043.8.498.17280432617865183599445746558731534322","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99635784824711754116112241806442235135","series":"1.2.826.0.1.3680043.8.498.77861290660928953999189514271795363767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99710592666731203419714499168506345263","series":"1.2.826.0.1.3680043.8.498.86493810626081437538869826989244612627","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99768154649604643975127490783989114506","series":"1.2.826.0.1.3680043.8.498.13086769161509053937173684068604205500","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99826037216341310731701527407057049820","series":"1.2.826.0.1.3680043.8.498.11921238913487093827347049836930662205","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99864558747514747686151806909794288262","series":"1.2.826.0.1.3680043.8.498.11372243683617754156251199333064168238","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290","series":"1.2.826.0.1.3680043.8.498.99238949500125352248111605622199971709","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966","series":"1.2.826.0.1.3680043.8.498.75079892939579407106916363523996097406","plane":"Coronal","fluid":0,"slot":"COR_T1"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1778
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
